
# Task 2 — Валидация темпорального графа знаний (Google Colab)

Этот блокнот является **естественным продолжением Task 1**.

Эксперт:
1. загружает YAML, созданный в первом блокноте;
2. получает **reference graph** (точная реконструкция ручной reasoning-схемы);
3. получает **automatic temporal KG**, построенный по статьям из YAML через **одну CLI-команду**;
4. сравнивает оба графа по триплетам и через интерактивную визуализацию;
5. при необходимости заполняет `graph_review_prefill.json` и `temporal_corrections_template.json`.

## Что уже учтено
- `start_date` и `end_date` вместо единственного `interval`
- двойная темпоральность: `valid_from` / `valid_to`
- поддержка `-inf`, `+inf`, `unknown`
- слой `scout/suggested_links.json` для поиска дополнительных ссылок
- `time_source` и `time_source_note` для разрешения неоднозначности, откуда взято время
- обратная совместимость с legacy `time_interval`

## Что увидит эксперт
- текстовые таблицы триплетов
- reference graph
- automatic temporal KG
- force-directed HTML-визуализацию (PyVis)
- интерактивный граф через hvPlot / HoloViews

> Если g4f-мультимодальные модели не проходят probe, блокнот автоматически переключит VLM на `Qwen/Qwen2.5-VL-3B-Instruct` и напомнит переключить Colab на **GPU**.



# Пошаговый туториал

## Шаг 0. Что важно знать
- Код редактировать **не обязательно**.
- Основной сценарий: запускать ячейки сверху вниз и нажимать кнопки.
- По умолчанию блокнот **клонирует официальный репозиторий** `top-papers/top-papers-graph`, затем накладывает улучшения для Task 2.

## Шаг 1. Запустите ячейку «Установка и импорт»
Она установит зависимости для:
- репозитория
- визуализации графов
- g4f probe
- локального fallback на Qwen2.5-VL

## Шаг 2. Нажмите «Клонировать и развернуть репозиторий»
Блокнот:
- клонирует GitHub-репозиторий
- применит улучшения Task 2
- установит пакет в editable-режиме

## Шаг 3. Загрузите YAML из Task 1
Можно использовать YAML, который эксперт получил в первом блокноте.

## Шаг 4. Нажмите «Проверить g4f и выбрать VLM»
Блокнот попробует мультимодальные модели g4f. Если ни одна не сработает, будет включён fallback на `Qwen/Qwen2.5-VL-3B-Instruct`.

## Шаг 5. Нажмите «Собрать Task 2 bundle»
Будет вызвана одна CLI-команда:

```bash
top-papers-graph prepare-task2-validation --trajectory <ваш_yaml>
```

## Шаг 6. Просмотрите результаты
После сборки блокнот покажет:
- summary
- таблицы triplets
- reference graph
- automatic graph
- ссылки-кандидаты для ручного анализа


In [ ]:
# @title
# Установка и импорт (запустите эту ячейку)
import os, sys, json, shutil, subprocess, textwrap, base64, tempfile, re
from pathlib import Path

# Базовые зависимости для UI, визуализации и работы с репозиторием
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ipywidgets', 'pyyaml', 'pandas', 'networkx', 'pyvis',
                'hvplot', 'holoviews', 'bokeh', 'panel', 'requests', 'pillow'], check=True)

import yaml
import pandas as pd
import networkx as nx
import ipywidgets as W
import hvplot.networkx as hvnx
import holoviews as hv
import panel as pn

from IPython.display import display, Markdown, HTML, IFrame, clear_output

hv.extension('bokeh')
pn.extension()

WORK_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
SESSION = {
    'repo_dir': None,
    'trajectory_path': None,
    'bundle_dir': None,
    'vlm_mode': None,
    'vlm_model': None,
    'g4f_probe': [],
}

print('Готово. Рабочая директория:', WORK_ROOT)


In [ ]:
# @title
# Вспомогательные функции и patch Task 2 (не нужно редактировать)
PATCHES = json.loads(r"""{"src/scireason/config.py": "from __future__ import annotations\\n\\nfrom pydantic_settings import BaseSettings, SettingsConfigDict\\n\\n\\nclass Settings(BaseSettings):\\n    model_config = SettingsConfigDict(env_file=\\".env\\", env_file_encoding=\\"utf-8\\", extra=\\"ignore\\")\\n\\n    # ===== LLM =====\\n    # Default: `auto` for classroom robustness.\\n    # auto -> try local Ollama (if reachable) -> g4f (if installed) -> LiteLLM providers -> mock (offline)\\n    llm_provider: str = \\"auto\\"  # auto|mock|ollama|g4f|openai|anthropic|...\\n    llm_model: str = \\"auto\\"\\n    ollama_base_url: str = \\"http://localhost:11434\\"\\n\\n    # g4f fine-tuning (optional)\\n    g4f_providers: str | None = None\\n    g4f_api_key: str | None = None\\n\\n    # ===== Embeddings =====\\n    embed_provider: str = \\"hash\\"  # hash|sentence-transformers|openai|ollama|...\\n    embed_model: str = \\"sentence-transformers/all-MiniLM-L6-v2\\"\\n    hash_embed_dim: int = 384\\n\\n    # ===== Infra (optional services) =====\\n    neo4j_uri: str = \\"bolt://localhost:7687\\"\\n    neo4j_user: str = \\"neo4j\\"\\n    neo4j_password: str = \\"please_change_me\\"\\n\\n    qdrant_url: str = \\"http://localhost:6333\\"\\n    qdrant_api_key: str | None = None\\n\\n    grobid_url: str = \\"http://localhost:8070\\"\\n\\n    # ===== HTTP Client (cache + rate limiting) =====\\n    http_cache_dir: str = \\".cache/http\\"\\n    http_cache_ttl_seconds: int = 7 * 24 * 3600\\n    http_timeout_seconds: int = 30\\n\\n    # ===== External APIs =====\\n    contact_email: str | None = None\\n    user_agent: str | None = None\\n\\n    # Semantic Scholar\\n    s2_api_key: str | None = None\\n\\n    # NCBI / PubMed\\n    ncbi_api_key: str | None = None\\n    ncbi_tool: str = \\"top-papers-graph\\"\\n    ncbi_email: str | None = None\\n\\n    # Crossref / OpenAlex\\n    crossref_mailto: str | None = None\\n    openalex_mailto: str | None = None\\n    openalex_api_key: str | None = None\\n\\n    # ===== Demo store (retrieval few-shot) =====\\n    demo_enabled: bool = True\\n    demo_schema_version: str = \\"1.0\\"\\n    demo_quality: str = \\"gold\\"  # gold|silver\\n    demo_top_k_triplets: int = 3\\n    demo_top_k_hypothesis: int = 2\\n    demo_max_chars_total: int = 3500\\n    demo_collection_triplets: str = \\"demos_temporal_triplets\\"\\n    demo_collection_hypothesis: str = \\"demos_hypothesis_test\\"\\n\\n    # ===== Multimodal (VL + MM embeddings) =====\\n    vlm_backend: str = \\"none\\"  # none|qwen2_vl|llava|phi3_vision\\n    vlm_model_id: str = \\"Qwen/Qwen2.5-VL-3B-Instruct\\"\\n    mm_embed_backend: str = \\"none\\"  # none|open_clip\\n    open_clip_model: str = \\"ViT-B-32\\"\\n    open_clip_pretrained: str = \\"laion2b_s34b_b79k\\"\\n    pdf_render_dpi: int = 150\\n\\n    # ===== Temporal GraphRAG =====\\n    temporal_default_granularity: str = \\"year\\"  # year|month|day\\n\\n    # ===== Domain =====\\n    domain_id: str = \\"science\\"\\n    domain_config_path: str | None = None\\n\\n    # ===== Agentic hypothesis generation (code-writing agent) =====\\n    hyp_agent_enabled: bool = True\\n    # internal|smolagents\\n    # - internal: lightweight built-in code agent (works fully offline with --llm-provider mock)\\n    # - smolagents: Hugging Face smolagents CodeAgent (optional dependency)\\n    hyp_agent_backend: str = \\"internal\\"\\n    hyp_agent_max_steps: int = 4\\n    hyp_agent_timeout_seconds: int = 20\\n\\n    # ===== smolagents integration (optional) =====\\n    # Only used when HYP_AGENT_BACKEND=smolagents.\\n    #\\n    # smol_model_backend:\\n    # - scireason: wrap this project's LLM router (LLM_PROVIDER=auto|g4f|ollama|...) into a smolagents Model\\n    # - transformers: smolagents.TransformersModel for local HF models (requires smolagents[transformers])\\n    # - g4f: direct smolagents Model that calls g4f client (requires g4f)\\n    smol_model_backend: str = \\"scireason\\"\\n    smol_model_id: str = \\"HuggingFaceTB/SmolLM2-1.7B-Instruct\\"\\n    smol_max_new_tokens: int = 768\\n    smol_device_map: str | None = None\\n    smol_torch_dtype: str | None = None  # e.g. float16|bfloat16\\n    smol_g4f_model: str = \\"auto\\"\\n    smol_executor: str = \\"local\\"  # local|docker (docker requires Docker)\\n    smol_print_steps: bool = False\\n\\n    # ===== Temporal link prediction / TGNN =====\\n    # Enabled by default: the project should prefer event-stream temporal prediction over\\n    # static graph link prediction whenever the temporal KG is available.\\n    hyp_tgnn_enabled: bool = True\\n    hyp_tgnn_recent_window_years: int = 3\\n    hyp_tgnn_half_life_years: float = 2.0\\n    hyp_tgnn_min_candidate_score: float = 0.05\\n\\n    # ===== Static GNN baseline (PyTorch Geometric) =====\\n    # Kept as an optional baseline for ablations and course comparisons.\\n    hyp_gnn_enabled: bool = False\\n    hyp_gnn_epochs: int = 80\\n    hyp_gnn_hidden_dim: int = 64\\n    hyp_gnn_lr: float = 0.01\\n    # To keep training fast for classroom-sized runs, we restrict to an induced\\n    # subgraph of the most connected terms.\\n    hyp_gnn_node_cap: int = 300\\n    hyp_gnn_seed: int = 7\\n\\n    # ===== Neo4j vector indexing =====\\n    neo4j_vector_enabled: bool = True\\n    neo4j_vector_chunk_dimensions: int = 384\\n    neo4j_vector_assertion_dimensions: int = 384\\n\\n\\nsettings = Settings()\\n", "src/scireason/domain.py": "from __future__ import annotations\\n\\n\\"\\"\\"Domain configuration utilities.\\n\\nWhy this exists\\n---------------\\nSciReason is designed to support many scientific domains (battery, bio, ...). Most things that are\\ndomain-sensitive (seed queries, ontologies, experiment backends, validation rules for expert\\nartifacts) should be configured via YAML rather than hard-coded.\\n\\nThe loader below is intentionally lightweight: it tries to load a YAML config file from the\\nrepository (when running from source) but also works when the config is missing (e.g. when the\\npackage is installed without the `configs/` directory).\\n\\"\\"\\"\\n\\nfrom dataclasses import dataclass, field\\nfrom pathlib import Path\\nfrom typing import Any, Dict, List, Optional\\n\\nimport yaml  # type: ignore\\n\\nfrom .config import settings\\n\\n\\n@dataclass\\nclass DomainConfig:\\n    domain_id: str\\n    title: str = \\"Science\\"\\n    keywords: List[str] = field(default_factory=list)\\n    seed_queries: List[str] = field(default_factory=list)\\n\\n    # Optional sections (free-form dictionaries)\\n    kg: Dict[str, Any] = field(default_factory=dict)\\n    evaluation: Dict[str, Any] = field(default_factory=dict)\\n    artifact_validation: Dict[str, Any] = field(default_factory=dict)\\n\\n    # Term graph / temporal KG knobs (optional)\\n    term_graph: Dict[str, Any] = field(default_factory=dict)\\n\\n\\ndef _candidate_paths(domain_id: str) -> List[Path]:\\n    \\"\\"\\"Resolve config candidate paths.\\n\\n    Priority:\\n    1) explicit settings.domain_config_path\\n    2) configs/domains/<domain_id>.yaml relative to CWD\\n    3) configs/domains/<domain_id>.yaml relative to repository root (best effort)\\n    \\"\\"\\"\\n    paths: List[Path] = []\\n    if settings.domain_config_path:\\n        paths.append(Path(settings.domain_config_path))\\n\\n    paths.append(Path(\\"configs\\") / \\"domains\\" / f\\"{domain_id}.yaml\\")\\n\\n    # When running from source, this file lives in <repo>/src/scireason/domain.py\\n    try:\\n        repo_root = Path(__file__).resolve().parents[2]\\n        paths.append(repo_root / \\"configs\\" / \\"domains\\" / f\\"{domain_id}.yaml\\")\\n    except Exception:\\n        pass\\n\\n    # Unique, existing\\n    seen: set[str] = set()\\n    out: List[Path] = []\\n    for p in paths:\\n        key = str(p)\\n        if key in seen:\\n            continue\\n        seen.add(key)\\n        if p.exists():\\n            out.append(p)\\n    return out\\n\\n\\ndef load_domain_config(domain_id: Optional[str] = None) -> DomainConfig:\\n    \\"\\"\\"Load domain configuration.\\n\\n    Supports both canonical `domain_id` values (for example `science`) and Wikidata QIDs\\n    stored in expert trajectory artifacts (for example `Q336`).\\n\\n    If the config file is missing, returns a minimal DomainConfig so the system stays usable.\\n    \\"\\"\\"\\n    did = (domain_id or settings.domain_id or \\"science\\").strip()\\n\\n    def _from_data(data: dict[str, Any], fallback: str) -> DomainConfig:\\n        return DomainConfig(\\n            domain_id=str(data.get(\\"domain_id\\") or fallback),\\n            title=str(data.get(\\"title\\") or fallback),\\n            keywords=list(data.get(\\"keywords\\") or []),\\n            seed_queries=list(data.get(\\"seed_queries\\") or []),\\n            kg=dict(data.get(\\"kg\\") or {}),\\n            evaluation=dict(data.get(\\"evaluation\\") or {}),\\n            artifact_validation=dict(data.get(\\"artifact_validation\\") or {}),\\n            term_graph=dict(data.get(\\"term_graph\\") or {}),\\n        )\\n\\n    for path in _candidate_paths(did):\\n        data = yaml.safe_load(path.read_text(encoding=\\"utf-8\\")) or {}\\n        return _from_data(data, did)\\n\\n    # Expert trajectories store a Wikidata QID in `domain`. Support that form directly.\\n    qid = did.upper()\\n    if qid.startswith(\\"Q\\") and qid[1:].isdigit():\\n        candidate_dirs: list[Path] = []\\n        try:\\n            repo_root = Path(__file__).resolve().parents[2]\\n            candidate_dirs.append(repo_root / \\"configs\\" / \\"domains\\")\\n        except Exception:\\n            pass\\n        candidate_dirs.append(Path(\\"configs\\") / \\"domains\\")\\n\\n        seen: set[str] = set()\\n        for directory in candidate_dirs:\\n            key = str(directory)\\n            if key in seen or not directory.exists():\\n                continue\\n            seen.add(key)\\n            for cfg_path in sorted(directory.glob(\\"*.yaml\\")):\\n                try:\\n                    data = yaml.safe_load(cfg_path.read_text(encoding=\\"utf-8\\")) or {}\\n                except Exception:\\n                    continue\\n                if str(data.get(\\"wikidata_qid\\") or \\"\\").strip().upper() == qid:\\n                    return _from_data(data, did)\\n\\n    # Fallback\\n    return DomainConfig(domain_id=did, title=did)\\n", "src/scireason/graph/review_applier.py": "from __future__ import annotations\\n\\nimport json\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\nfrom typing import Any, Dict, Iterable, List, Tuple\\n\\n\\n@dataclass\\nclass ReviewStats:\\n    accepted: int = 0\\n    rejected: int = 0\\n    needs_fix: int = 0\\n    added: int = 0\\n\\n\\ndef _iter_review_files(graph_reviews_dir: Path) -> Iterable[Path]:\\n    yield from sorted(graph_reviews_dir.glob(\\"**/*.json\\"))\\n\\n\\ndef _weight_for(verdict: str) -> float:\\n    if verdict == \\"accepted\\":\\n        return 1.0\\n    if verdict == \\"rejected\\":\\n        return -1.0\\n    if verdict in {\\"needs_time_fix\\", \\"needs_evidence_fix\\"}:\\n        return -0.25\\n    if verdict == \\"added\\":\\n        return 0.75\\n    return 0.0\\n\\n\\ndef _time_token(v: Any, *, default: str = \\"unknown\\") -> str:\\n    if v is None:\\n        return default\\n    s = str(v).strip()\\n    return s or default\\n\\n\\ndef _canonical_time_interval(a: Dict[str, Any]) -> str:\\n    legacy = str(a.get(\\"time_interval\\") or \\"\\").strip()\\n    if legacy:\\n        return legacy\\n\\n    start = _time_token(a.get(\\"start_date\\"), default=\\"unknown\\")\\n    end = _time_token(a.get(\\"end_date\\"), default=\\"unknown\\")\\n    valid_from = _time_token(a.get(\\"valid_from\\"), default=start)\\n    valid_to = _time_token(a.get(\\"valid_to\\"), default=\\"+inf\\")\\n    return f\\"evidence:{start}..{end}|valid:{valid_from}..{valid_to}\\"\\n\\n\\ndef _normalize_assertions(doc: Dict[str, Any], source_path: Path) -> List[Dict[str, Any]]:\\n    \\"\\"\\"Support both new and legacy expert formats.\\n\\n    New format (preferred):\\n      { assertions: [ {subject,predicate,object,time_interval,evidence,verdict,...}, ... ] }\\n\\n    Legacy format (kept for backward compatibility in early course phases):\\n      { accepted_edges: [...], rejected_edges: [...], added_edges: [...] }\\n    \\"\\"\\"\\n    assertions = doc.get(\\"assertions\\")\\n    if isinstance(assertions, list) and assertions:\\n        return assertions\\n\\n    out: List[Dict[str, Any]] = []\\n\\n    for a in doc.get(\\"accepted_edges\\", []) or []:\\n        out.append({\\n            \\"assertion_id\\": a.get(\\"assertion_id\\") or \\"legacy\\",\\n            \\"subject\\": a.get(\\"subject\\"),\\n            \\"predicate\\": a.get(\\"predicate\\"),\\n            \\"object\\": a.get(\\"object\\"),\\n            \\"time_interval\\": a.get(\\"time_interval\\", \\"unknown\\"),\\n            \\"start_date\\": a.get(\\"start_date\\"),\\n            \\"end_date\\": a.get(\\"end_date\\"),\\n            \\"valid_from\\": a.get(\\"valid_from\\"),\\n            \\"valid_to\\": a.get(\\"valid_to\\"),\\n            \\"time_source\\": a.get(\\"time_source\\") or \\"legacy\\",\\n            \\"evidence\\": {\\n                \\"page\\": a.get(\\"evidence\\", {}).get(\\"page\\"),\\n                \\"figure_or_table\\": a.get(\\"evidence\\", {}).get(\\"figure_or_table\\"),\\n                \\"snippet_or_summary\\": a.get(\\"evidence\\", {}).get(\\"text_snippet\\") or a.get(\\"evidence\\", {}).get(\\"snippet_or_summary\\"),\\n            },\\n            \\"verdict\\": \\"accepted\\",\\n            \\"rationale\\": a.get(\\"rationale\\") or \\"legacy accepted_edges\\",\\n        })\\n\\n    for a in doc.get(\\"rejected_edges\\", []) or []:\\n        out.append({\\n            \\"assertion_id\\": a.get(\\"assertion_id\\") or \\"legacy\\",\\n            \\"subject\\": a.get(\\"subject\\"),\\n            \\"predicate\\": a.get(\\"predicate\\"),\\n            \\"object\\": a.get(\\"object\\"),\\n            \\"time_interval\\": a.get(\\"time_interval\\", \\"unknown\\"),\\n            \\"start_date\\": a.get(\\"start_date\\"),\\n            \\"end_date\\": a.get(\\"end_date\\"),\\n            \\"valid_from\\": a.get(\\"valid_from\\"),\\n            \\"valid_to\\": a.get(\\"valid_to\\"),\\n            \\"time_source\\": a.get(\\"time_source\\") or \\"legacy\\",\\n            \\"evidence\\": {\\n                \\"page\\": None,\\n                \\"figure_or_table\\": None,\\n                \\"snippet_or_summary\\": None,\\n            },\\n            \\"verdict\\": \\"rejected\\",\\n            \\"rationale\\": a.get(\\"reason\\") or \\"legacy rejected_edges\\",\\n        })\\n\\n    for a in doc.get(\\"added_edges\\", []) or []:\\n        out.append({\\n            \\"assertion_id\\": a.get(\\"assertion_id\\") or \\"legacy\\",\\n            \\"subject\\": a.get(\\"subject\\"),\\n            \\"predicate\\": a.get(\\"predicate\\"),\\n            \\"object\\": a.get(\\"object\\"),\\n            \\"time_interval\\": a.get(\\"time_interval\\", \\"unknown\\"),\\n            \\"start_date\\": a.get(\\"start_date\\"),\\n            \\"end_date\\": a.get(\\"end_date\\"),\\n            \\"valid_from\\": a.get(\\"valid_from\\"),\\n            \\"valid_to\\": a.get(\\"valid_to\\"),\\n            \\"time_source\\": a.get(\\"time_source\\") or \\"legacy\\",\\n            \\"evidence\\": {\\n                \\"page\\": None,\\n                \\"figure_or_table\\": None,\\n                \\"snippet_or_summary\\": a.get(\\"evidence_hint\\") or None,\\n            },\\n            \\"verdict\\": \\"added\\",\\n            \\"rationale\\": a.get(\\"reason\\") or \\"legacy added_edges\\",\\n        })\\n\\n    return out\\n\\n\\ndef compile_overrides(graph_reviews_dir: Path, out_path: Path) -> ReviewStats:\\n    \\"\\"\\"Compile expert graph reviews into a DB-agnostic overrides file (JSONL).\\n\\n    Each line:\\n      {assertion_id, verdict, weight, subject, predicate, object, time_interval, source_review}\\n\\n    The overrides file is used by:\\n    - retriever weighting (future)\\n    - rule-based reward now (immediate behavior changes)\\n    - optional Neo4j tagging via CLI (`apply-graph-reviews --to-neo4j`)\\n    \\"\\"\\"\\n    stats = ReviewStats()\\n    out_path.parent.mkdir(parents=True, exist_ok=True)\\n\\n    with out_path.open(\\"w\\", encoding=\\"utf-8\\") as out:\\n        for f in _iter_review_files(graph_reviews_dir):\\n            doc = json.loads(f.read_text(encoding=\\"utf-8\\"))\\n            assertions = _normalize_assertions(doc, f)\\n            for a in assertions:\\n                verdict = str(a.get(\\"verdict\\", \\"\\")).strip()\\n                rec = {\\n                    \\"assertion_id\\": a.get(\\"assertion_id\\") or \\"new\\",\\n                    \\"verdict\\": verdict,\\n                    \\"weight\\": _weight_for(verdict),\\n                    \\"subject\\": a.get(\\"subject\\"),\\n                    \\"predicate\\": a.get(\\"predicate\\"),\\n                    \\"object\\": a.get(\\"object\\"),\\n                    \\"start_date\\": _time_token(a.get(\\"start_date\\"), default=\\"unknown\\"),\\n                    \\"end_date\\": _time_token(a.get(\\"end_date\\"), default=\\"unknown\\"),\\n                    \\"valid_from\\": _time_token(a.get(\\"valid_from\\"), default=_time_token(a.get(\\"start_date\\"), default=\\"unknown\\")),\\n                    \\"valid_to\\": _time_token(a.get(\\"valid_to\\"), default=\\"+inf\\"),\\n                    \\"time_source\\": a.get(\\"time_source\\") or \\"unknown\\",\\n                    \\"time_interval\\": _canonical_time_interval(a),\\n                    \\"source_review\\": str(f),\\n                }\\n                out.write(json.dumps(rec, ensure_ascii=False) + \\"\\\\n\\")\\n\\n                if verdict == \\"accepted\\":\\n                    stats.accepted += 1\\n                elif verdict == \\"rejected\\":\\n                    stats.rejected += 1\\n                elif verdict in {\\"needs_time_fix\\", \\"needs_evidence_fix\\"}:\\n                    stats.needs_fix += 1\\n                elif verdict == \\"added\\":\\n                    stats.added += 1\\n\\n    return stats\\n", "src/scireason/graph/temporal_neo4j_store.py": "from __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom hashlib import sha1\\nfrom typing import Any, Dict, Iterable, Optional\\n\\ntry:  # pragma: no cover\\n    from neo4j import GraphDatabase\\nexcept Exception:  # pragma: no cover\\n    GraphDatabase = None  # type: ignore[assignment]\\n\\nfrom ..config import settings\\nfrom ..temporal.schemas import TemporalEvent, TimeInterval, TemporalTriplet\\n\\n\\ndef _assertion_id(paper_id: str, t: TemporalTriplet) -> str:\\n    key = (\\n        f\\"{paper_id}|{t.subject}|{t.predicate}|{t.object}|{t.polarity}|\\"\\n        f\\"{t.time.start if t.time else ''}|{t.time.end if t.time else ''}\\"\\n    )\\n    return sha1(key.encode(\\"utf-8\\")).hexdigest()[:16]\\n\\n\\n@dataclass\\nclass Neo4jTemporalStore:\\n    uri: str = settings.neo4j_uri\\n    user: str = settings.neo4j_user\\n    password: str = settings.neo4j_password\\n\\n    def __post_init__(self) -> None:\\n        if GraphDatabase is None:\\n            raise RuntimeError(\\n                \\"neo4j python driver is not installed. Install base dependencies with neo4j support enabled.\\"\\n            )\\n        self._driver = GraphDatabase.driver(self.uri, auth=(self.user, self.password))\\n\\n    def close(self) -> None:\\n        self._driver.close()\\n\\n    def ensure_schema(self) -> None:\\n        cypher = [\\n            \\"CREATE CONSTRAINT paper_id IF NOT EXISTS FOR (p:Paper) REQUIRE p.id IS UNIQUE\\",\\n            \\"CREATE CONSTRAINT entity_name IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE\\",\\n            \\"CREATE CONSTRAINT time_key IF NOT EXISTS FOR (t:Time) REQUIRE t.key IS UNIQUE\\",\\n            \\"CREATE CONSTRAINT assertion_id IF NOT EXISTS FOR (a:Assertion) REQUIRE a.id IS UNIQUE\\",\\n            \\"CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (c:Chunk) REQUIRE c.id IS UNIQUE\\",\\n            \\"CREATE CONSTRAINT event_id IF NOT EXISTS FOR (e:Event) REQUIRE e.id IS UNIQUE\\",\\n        ]\\n        with self._driver.session() as s:\\n            for q in cypher:\\n                s.run(q)\\n\\n    def ensure_vector_indexes(\\n        self,\\n        *,\\n        chunk_dimensions: Optional[int] = None,\\n        assertion_dimensions: Optional[int] = None,\\n        entity_dimensions: Optional[int] = None,\\n    ) -> None:\\n        \\"\\"\\"Best-effort vector indexes for Neo4j as graph + vector DB.\\n\\n        Compatible with current Neo4j vector index syntax. Older servers may reject these\\n        statements; callers should treat failures as non-fatal.\\n        \\"\\"\\"\\n\\n        queries = []\\n        if chunk_dimensions:\\n            queries.append(\\n                (\\n                    \\"chunk_embedding_idx\\",\\n                    f\\"CREATE VECTOR INDEX chunk_embedding_idx IF NOT EXISTS \\"\\n                    f\\"FOR (c:Chunk) ON c.embedding \\"\\n                    f\\"OPTIONS {{indexConfig: {{`vector.dimensions`: {int(chunk_dimensions)}, `vector.similarity_function`: 'cosine'}}}}\\",\\n                )\\n            )\\n        if assertion_dimensions:\\n            queries.append(\\n                (\\n                    \\"assertion_embedding_idx\\",\\n                    f\\"CREATE VECTOR INDEX assertion_embedding_idx IF NOT EXISTS \\"\\n                    f\\"FOR (a:Assertion) ON a.embedding \\"\\n                    f\\"OPTIONS {{indexConfig: {{`vector.dimensions`: {int(assertion_dimensions)}, `vector.similarity_function`: 'cosine'}}}}\\",\\n                )\\n            )\\n        if entity_dimensions:\\n            queries.append(\\n                (\\n                    \\"entity_embedding_idx\\",\\n                    f\\"CREATE VECTOR INDEX entity_embedding_idx IF NOT EXISTS \\"\\n                    f\\"FOR (e:Entity) ON e.embedding \\"\\n                    f\\"OPTIONS {{indexConfig: {{`vector.dimensions`: {int(entity_dimensions)}, `vector.similarity_function`: 'cosine'}}}}\\",\\n                )\\n            )\\n        with self._driver.session() as s:\\n            for _, q in queries:\\n                try:\\n                    s.run(q)\\n                except Exception:\\n                    # Best effort only: older Neo4j versions or restricted deployments may not support vector indexes.\\n                    continue\\n\\n    def upsert_paper(self, paper: Dict[str, Any]) -> None:\\n        q = \\"\\"\\"\\n        MERGE (p:Paper {id: $id})\\n        SET p.title = $title,\\n            p.year = $year,\\n            p.source = $source,\\n            p.url = $url\\n        \\"\\"\\"\\n        with self._driver.session() as s:\\n            s.run(q, **paper)\\n\\n    def upsert_chunk(\\n        self,\\n        paper_id: str,\\n        chunk_id: str,\\n        text: str,\\n        chunk_index: Optional[int] = None,\\n        embedding: Optional[list[float]] = None,\\n    ) -> None:\\n        \\"\\"\\"Upsert a text chunk node for provenance and optional vector retrieval.\\"\\"\\"\\n        t = (text or \\"\\").strip()\\n        if len(t) > 4000:\\n            t = t[:4000] + \\"…\\"\\n\\n        q = \\"\\"\\"\\n        MATCH (p:Paper {id:$paper_id})\\n        MERGE (c:Chunk {id:$chunk_id})\\n        SET c.paper_id=$paper_id,\\n            c.idx=$chunk_index,\\n            c.text=$text,\\n            c.embedding=$embedding\\n        MERGE (p)-[:HAS_CHUNK]->(c)\\n        \\"\\"\\"\\n        with self._driver.session() as s:\\n            s.run(\\n                q,\\n                paper_id=paper_id,\\n                chunk_id=chunk_id,\\n                chunk_index=chunk_index,\\n                text=t,\\n                embedding=embedding,\\n            )\\n\\n    def upsert_time(self, interval: TimeInterval) -> str:\\n        start = interval.start or \\"\\"\\n        end = interval.end or start\\n        key = f\\"{interval.granularity}:{start}:{end}\\"\\n\\n        q = \\"\\"\\"\\n        MERGE (t:Time {key: $key})\\n        SET t.granularity = $granularity,\\n            t.start = $start,\\n            t.end = $end\\n        RETURN t.key as key\\n        \\"\\"\\"\\n        with self._driver.session() as s:\\n            rec = s.run(q, key=key, granularity=interval.granularity, start=start, end=end).single()\\n            assert rec\\n            return rec[\\"key\\"]\\n\\n    def upsert_assertion(\\n        self,\\n        paper_id: str,\\n        t: TemporalTriplet,\\n        chunk_id: Optional[str] = None,\\n        evidence_quote: Optional[str] = None,\\n        embedding: Optional[list[float]] = None,\\n        extraction_method: str = \\"llm_triplet\\",\\n        review_status: str = \\"pending\\",\\n    ) -> str:\\n        aid = _assertion_id(paper_id, t)\\n        time_key = None\\n        if t.time:\\n            time_key = self.upsert_time(t.time)\\n\\n        q = \\"\\"\\"\\n        MERGE (s:Entity {name: $subj})\\n        MERGE (o:Entity {name: $obj})\\n        MATCH (p:Paper {id: $paper_id})\\n        MERGE (a:Assertion {id: $aid})\\n        SET a.predicate = $pred,\\n            a.confidence = $confidence,\\n            a.polarity = $polarity,\\n            a.paper_id = $paper_id,\\n            a.evidence_quote = $evidence_quote,\\n            a.embedding = $embedding,\\n            a.extraction_method = $extraction_method,\\n            a.review_status = $review_status,\\n            a.object = $obj,\\n            a.subject = $subj\\n        MERGE (a)-[:SUBJECT]->(s)\\n        MERGE (a)-[:OBJECT]->(o)\\n        MERGE (p)-[:HAS_ASSERTION]->(a)\\n        MERGE (a)-[:ASSERTED_IN]->(p)\\n        WITH a\\n        OPTIONAL MATCH (t:Time {key: $time_key})\\n        FOREACH (_ IN CASE WHEN t IS NULL THEN [] ELSE [1] END |\\n            MERGE (a)-[:AT_TIME]->(t)\\n        )\\n        RETURN a.id as aid\\n        \\"\\"\\"\\n        with self._driver.session() as s:\\n            rec = s.run(\\n                q,\\n                aid=aid,\\n                subj=t.subject,\\n                obj=t.object,\\n                pred=t.predicate,\\n                confidence=float(t.confidence),\\n                polarity=t.polarity,\\n                paper_id=paper_id,\\n                time_key=time_key,\\n                evidence_quote=(evidence_quote or t.evidence_quote or None),\\n                embedding=embedding,\\n                extraction_method=extraction_method,\\n                review_status=review_status,\\n            ).single()\\n            assert rec\\n            assertion_id = rec[\\"aid\\"]\\n\\n        if chunk_id:\\n            q_ev = \\"\\"\\"\\n            MATCH (a:Assertion {id:$aid})\\n            MERGE (c:Chunk {id:$chunk_id})\\n            MERGE (a)-[e:EVIDENCE]->(c)\\n            SET e.quote = $quote\\n            \\"\\"\\"\\n            quote = (evidence_quote or t.evidence_quote or \\"\\").strip()\\n            if len(quote) > 200:\\n                quote = quote[:200] + \\"…\\"\\n            with self._driver.session() as s:\\n                s.run(q_ev, aid=assertion_id, chunk_id=chunk_id, quote=quote)\\n\\n        return assertion_id\\n\\n    def upsert_event(self, event: TemporalEvent) -> str:\\n        event_id = event.stable_id()\\n        time_key = self.upsert_time(\\n            TimeInterval(start=event.ts_start, end=event.ts_end, granularity=event.granularity)\\n        )\\n        q = \\"\\"\\"\\n        MERGE (s:Entity {name:$subj})\\n        MERGE (o:Entity {name:$obj})\\n        MATCH (p:Paper {id:$paper_id})\\n        MATCH (t:Time {key:$time_key})\\n        MERGE (e:Event {id:$event_id})\\n        SET e.paper_id=$paper_id,\\n            e.chunk_id=$chunk_id,\\n            e.assertion_id=$assertion_id,\\n            e.predicate=$pred,\\n            e.confidence=$confidence,\\n            e.polarity=$polarity,\\n            e.ts_start=$ts_start,\\n            e.ts_end=$ts_end,\\n            e.granularity=$granularity,\\n            e.split=$split,\\n            e.event_type=$event_type,\\n            e.extraction_method=$extraction_method,\\n            e.weight=$weight,\\n            e.evidence_quote=$evidence_quote\\n        MERGE (e)-[:SOURCE_ENTITY]->(s)\\n        MERGE (e)-[:TARGET_ENTITY]->(o)\\n        MERGE (e)-[:FROM_PAPER]->(p)\\n        MERGE (e)-[:AT_TIME]->(t)\\n        WITH e\\n        OPTIONAL MATCH (a:Assertion {id:$assertion_id})\\n        FOREACH (_ IN CASE WHEN a IS NULL THEN [] ELSE [1] END |\\n            MERGE (e)-[:ASSERTS]->(a)\\n        )\\n        RETURN e.id as event_id\\n        \\"\\"\\"\\n        with self._driver.session() as s:\\n            rec = s.run(\\n                q,\\n                event_id=event_id,\\n                paper_id=event.paper_id,\\n                chunk_id=event.chunk_id,\\n                assertion_id=event.assertion_id,\\n                subj=event.subject,\\n                obj=event.object,\\n                pred=event.predicate,\\n                confidence=float(event.confidence),\\n                polarity=event.polarity,\\n                ts_start=event.ts_start,\\n                ts_end=event.ts_end,\\n                granularity=event.granularity,\\n                split=event.split,\\n                event_type=event.event_type,\\n                extraction_method=event.extraction_method,\\n                weight=float(event.weight),\\n                evidence_quote=event.evidence_quote,\\n                time_key=time_key,\\n            ).single()\\n            assert rec\\n            return str(rec[\\"event_id\\"])\\n\\n    def export_event_stream(self, limit: int = 1000) -> list[dict[str, Any]]:\\n        q = \\"\\"\\"\\n        MATCH (e:Event)-[:SOURCE_ENTITY]->(s:Entity)\\n        MATCH (e)-[:TARGET_ENTITY]->(o:Entity)\\n        OPTIONAL MATCH (e)-[:AT_TIME]->(t:Time)\\n        RETURN e.id as event_id,\\n               e.paper_id as paper_id,\\n               e.chunk_id as chunk_id,\\n               e.assertion_id as assertion_id,\\n               s.name as subject,\\n               e.predicate as predicate,\\n               o.name as object,\\n               e.confidence as confidence,\\n               e.polarity as polarity,\\n               coalesce(e.ts_start, t.start) as ts_start,\\n               coalesce(e.ts_end, t.end) as ts_end,\\n               coalesce(e.granularity, t.granularity) as granularity,\\n               e.split as split,\\n               e.event_type as event_type,\\n               e.extraction_method as extraction_method,\\n               e.weight as weight,\\n               e.evidence_quote as evidence_quote\\n        ORDER BY coalesce(e.ts_start, t.start) ASC, e.id ASC\\n        LIMIT $limit\\n        \\"\\"\\"\\n        with self._driver.session() as s:\\n            return [dict(r) for r in s.run(q, limit=int(limit))]\\n\\n    def search_chunks_by_vector(self, index_name: str, query_embedding: list[float], limit: int = 8) -> list[dict[str, Any]]:\\n        q = \\"\\"\\"\\n        CALL db.index.vector.queryNodes($index_name, $limit, $embedding)\\n        YIELD node, score\\n        RETURN node.id as id, node.paper_id as paper_id, node.text as text, score\\n        ORDER BY score DESC\\n        \\"\\"\\"\\n        with self._driver.session() as s:\\n            return [dict(r) for r in s.run(q, index_name=index_name, limit=int(limit), embedding=query_embedding)]\\n\\n    def query_assertions(self, entity: str, time: Optional[TimeInterval] = None, limit: int = 50):\\n        \\"\\"\\"Простой поиск утверждений вокруг сущности с (опциональным) фильтром по времени.\\"\\"\\"\\n        time_key = None\\n        if time:\\n            start = time.start or \\"\\"\\n            end = time.end or start\\n            time_key = f\\"{time.granularity}:{start}:{end}\\"\\n\\n        q = \\"\\"\\"\\n        MATCH (e:Entity {name:$entity})\\n        MATCH (a:Assertion)-[:SUBJECT|OBJECT]->(e)\\n        OPTIONAL MATCH (a)-[:AT_TIME]->(t:Time)\\n        WITH a, t\\n        WHERE coalesce(a.status, 'active') <> 'replaced'\\n          AND ($time_key IS NULL OR (t.key = $time_key))\\n        RETURN a.id as id, a.predicate as predicate, a.confidence as confidence, a.polarity as polarity,\\n               t.start as t_start, t.end as t_end, t.granularity as granularity\\n        ORDER BY a.confidence DESC\\n        LIMIT $limit\\n        \\"\\"\\"\\n        with self._driver.session() as s:\\n            return [dict(r) for r in s.run(q, entity=entity, time_key=time_key, limit=limit)]\\n\\n    def apply_expert_override(\\n        self,\\n        subj: str,\\n        pred: str,\\n        obj: str,\\n        verdict: str,\\n        weight: float,\\n        time_interval: str = \\"unknown\\",\\n        *,\\n        start_date: str = \\"unknown\\",\\n        end_date: str = \\"unknown\\",\\n        valid_from: str = \\"unknown\\",\\n        valid_to: str = \\"+inf\\",\\n        time_source: str = \\"unknown\\",\\n    ) -> None:\\n        q = \\"\\"\\"\\n        MATCH (a:Assertion)-[:SUBJECT]->(s:Entity {name: $subj})\\n        MATCH (a)-[:OBJECT]->(o:Entity {name: $obj})\\n        WHERE a.predicate = $pred\\n        SET a.expert_verdict = $verdict,\\n            a.expert_weight = $weight,\\n            a.expert_time_interval = $time_interval,\\n            a.expert_start_date = $start_date,\\n            a.expert_end_date = $end_date,\\n            a.expert_valid_from = $valid_from,\\n            a.expert_valid_to = $valid_to,\\n            a.expert_time_source = $time_source\\n        \\"\\"\\"\\n        with self._driver.session() as s:\\n            s.run(\\n                q,\\n                subj=subj,\\n                obj=obj,\\n                pred=pred,\\n                verdict=verdict,\\n                weight=float(weight),\\n                time_interval=time_interval,\\n                start_date=start_date,\\n                end_date=end_date,\\n                valid_from=valid_from,\\n                valid_to=valid_to,\\n                time_source=time_source,\\n            )\\n\\n    def get_assertion_details(self, assertion_id: str) -> Optional[dict]:\\n        q = \\"\\"\\"\\n        MATCH (p:Paper)-[:HAS_ASSERTION]->(a:Assertion {id:$aid})\\n        MATCH (a)-[:SUBJECT]->(s:Entity)\\n        MATCH (a)-[:OBJECT]->(o:Entity)\\n        OPTIONAL MATCH (a)-[:AT_TIME]->(t:Time)\\n        RETURN p.id as paper_id,\\n               s.name as subject,\\n               a.predicate as predicate,\\n               o.name as object,\\n               a.polarity as polarity,\\n               a.confidence as confidence,\\n               a.evidence_quote as evidence_quote,\\n               t.start as t_start,\\n               t.end as t_end,\\n               t.granularity as granularity\\n        \\"\\"\\"\\n        with self._driver.session() as s:\\n            rec = s.run(q, aid=assertion_id).single()\\n            return dict(rec) if rec else None\\n\\n    def link_replacement(self, old_id: str, new_id: str, *, rationale: str = \\"\\", reviewer_id: str = \\"\\") -> None:\\n        q = \\"\\"\\"\\n        MATCH (old:Assertion {id:$old_id})\\n        MATCH (new:Assertion {id:$new_id})\\n        MERGE (old)-[r:REPLACED_BY]->(new)\\n        SET old.status = 'replaced',\\n            r.rationale = $rationale,\\n            r.reviewer_id = $reviewer_id\\n        \\"\\"\\"\\n        with self._driver.session() as s:\\n            s.run(q, old_id=old_id, new_id=new_id, rationale=rationale, reviewer_id=reviewer_id)\\n", "src/scireason/cli.py": "from __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any, Dict, Optional\\n\\nimport typer\\nfrom rich.console import Console\\nfrom rich.table import Table\\n\\nfrom .config import settings\\nfrom .domain import load_domain_config\\n\\nfrom .connectors.arxiv import search as arxiv_search\\nfrom .connectors.openalex import search_works as openalex_search\\nfrom .connectors.semantic_scholar import search_papers as s2_search\\n\\nfrom .connectors.crossref import search_works as crossref_search\\nfrom .connectors.pubmed import search as pubmed_search\\nfrom .connectors.europe_pmc import search as europepmc_search\\nfrom .connectors.biorxiv import (\\n    details_by_doi as biorxiv_details_by_doi,\\n    details_by_interval as biorxiv_details_by_interval,\\n    normalize_record as biorxiv_normalize_record,\\n)\\n\\nfrom .ingest.pipeline import ingest_pdf\\nfrom .ingest.mm_pipeline import ingest_pdf_multimodal\\nfrom .ingest.one_click import one_click_ingest_arxiv, normalize_arxiv_id\\n\\nfrom .graph.build_kg import build_from_paper_dir\\nfrom .graph.build_tg_mmkg import build_temporal_and_multimodal\\nfrom .graph.graphrag_query import retrieve_context\\nfrom .graph.review_applier import compile_overrides\\nfrom .graph.temporal_neo4j_store import Neo4jTemporalStore\\n\\nfrom .temporal.schemas import TemporalTriplet, TimeInterval\\n\\nfrom .agents.debate_graph import run_debate\\nfrom .agents.hypothesis_tester import load_hypothesis_from_json, test_hypothesis\\n\\nfrom .pipeline.e2e import run_pipeline\\nfrom .pipeline.demo import run_demo_pipeline\\nfrom .pipeline.task2_validation import prepare_task2_validation_bundle\\n\\n\\napp = typer.Typer(help=\\"top-papers-graph CLI (ex SciReason)\\", add_completion=False)\\nconsole = Console()\\n\\ndef _user_agent() -> str:\\n    \\"\\"\\"Build a polite User-Agent for external APIs.\\n\\n    Many scholarly APIs (Crossref/OpenAlex/arXiv/NCBI) recommend identifying your client\\n    and providing a contact email.\\n    \\"\\"\\"\\n    if settings.user_agent:\\n        return settings.user_agent\\n    if settings.contact_email:\\n        return f\\"top-papers-graph (mailto:{settings.contact_email})\\"\\n    return \\"top-papers-graph\\"\\n\\n\\n\\ndef _normalize_meta(meta: Dict[str, Any], *, fallback_id: str, source: str = \\"\\") -> Dict[str, Any]:\\n    \\"\\"\\"Normalize metadata to the minimal fields used across the repo.\\"\\"\\"\\n    m = dict(meta)\\n\\n    # id\\n    if not m.get(\\"id\\"):\\n        # prefer DOI if present, else fallback\\n        if m.get(\\"doi\\"):\\n            m[\\"id\\"] = f\\"doi:{m['doi']}\\"\\n        else:\\n            m[\\"id\\"] = fallback_id\\n\\n    # title\\n    m.setdefault(\\"title\\", \\"\\")\\n\\n    # year\\n    if not m.get(\\"year\\"):\\n        published = str(m.get(\\"published\\") or \\"\\")\\n        if len(published) >= 4 and published[:4].isdigit():\\n            m[\\"year\\"] = int(published[:4])\\n\\n    # source / url\\n    if source:\\n        m.setdefault(\\"source\\", source)\\n    m.setdefault(\\"url\\", m.get(\\"id\\") if isinstance(m.get(\\"id\\"), str) and m[\\"id\\"].startswith(\\"http\\") else \\"\\")\\n\\n    return m\\n\\n\\n@app.command()\\ndef doctor() -> None:\\n    \\"\\"\\"Проверка окружения/настроек.\\"\\"\\"\\n    domain_cfg = load_domain_config()\\n\\n    t = Table(title=\\"top-papers-graph doctor\\")\\n    t.add_column(\\"Key\\")\\n    t.add_column(\\"Value\\")\\n\\n    t.add_row(\\"Domain\\", f\\"{domain_cfg.domain_id} — {domain_cfg.title}\\")\\n    t.add_row(\\"LLM provider\\", settings.llm_provider)\\n    t.add_row(\\"LLM model\\", settings.llm_model)\\n    t.add_row(\\"Embed provider\\", settings.embed_provider)\\n    t.add_row(\\"Embed model\\", settings.embed_model)\\n    t.add_row(\\"Neo4j\\", settings.neo4j_uri)\\n    t.add_row(\\"Qdrant\\", settings.qdrant_url)\\n    t.add_row(\\"GROBID\\", settings.grobid_url)\\n    t.add_row(\\"CONTACT_EMAIL\\", str(settings.contact_email or \\"\\"))\\n    t.add_row(\\"USER_AGENT\\", _user_agent())\\n    t.add_row(\\"NCBI_EMAIL\\", str((settings.ncbi_email or settings.contact_email) or \\"\\"))\\n    t.add_row(\\"VLM backend\\", settings.vlm_backend)\\n    t.add_row(\\"MM embed backend\\", settings.mm_embed_backend)\\n\\n    console.print(t)\\n    console.print(\\"[green]Если сервисы подняты через docker compose — вы готовы.[/green]\\")\\n\\n    # g4f sanity: list working models from g4f/models.py (no network call)\\n    try:\\n        import g4f  # type: ignore\\n        from g4f import models as gm  # type: ignore\\n\\n        models_list = []\\n        try:\\n            Model = getattr(gm, \\"Model\\", None)\\n            if Model is not None and hasattr(Model, \\"__all__\\"):\\n                cand = Model.__all__()  # type: ignore\\n                if isinstance(cand, (list, tuple)):\\n                    models_list = list(cand)\\n        except Exception:\\n            models_list = []\\n\\n        if not models_list:\\n            models_list = list(getattr(gm, \\"_all_models\\", []) or [])\\n\\n        console.print(f\\"g4f: {getattr(g4f, '__version__', 'unknown')} | models (working): {len(models_list)}\\")\\n        if models_list:\\n            console.print(\\"g4f sample models: \\" + \\", \\".join(models_list[:10]))\\n    except Exception as e:\\n        console.print(f\\"g4f: not available ({e})\\")\\n\\n\\n\\n@app.command()\\ndef fetch(\\n    query: str,\\n    source: str = typer.Option(\\n        \\"arxiv\\",\\n        help=\\"arxiv|openalex|s2|crossref|pubmed|europepmc|biorxiv|medrxiv\\",\\n    ),\\n    limit: int = typer.Option(10, help=\\"Сколько результатов вернуть (где поддерживается)\\"),\\n    out: Path = typer.Option(Path(\\"data/papers/search.json\\"), help=\\"Куда сохранить JSON\\"),\\n    with_abstract: bool = typer.Option(False, help=\\"PubMed: подтянуть абстракт (EFetch).\\"),\\n    cursor: int = typer.Option(0, help=\\"biorxiv/medrxiv: cursor для пагинации (по 100 записей).\\"),\\n    category: Optional[str] = typer.Option(None, help=\\"biorxiv/medrxiv: фильтр subject category.\\"),\\n    normalize: bool = typer.Option(False, help=\\"Нормализовать к единому PaperMetadata schema (Pydantic)\\"),\\n) -> None:\\n    \\"\\"\\"Поиск статей (метаданные) в одном из источников.\\n\\n    Источники:\\n    - arxiv: arXiv Atom API (пример query: \\"all:graph rag\\" или \\"cat:cs.AI AND all:retrieval\\")\\n    - openalex: OpenAlex works search\\n    - s2: Semantic Scholar Graph API\\n    - crossref: Crossref works search (часто полезно для кандидатов DOI)\\n    - pubmed: NCBI PubMed E-utilities (ESearch + ESummary; опц. EFetch для абстрактов)\\n    - europepmc: Europe PMC (агрегатор PubMed + PMC + preprints и др.)\\n    - biorxiv/medrxiv: bioRxiv details API (query = DOI \\"10.1101/...\\" или interval \\"YYYY-MM-DD/YYYY-MM-DD\\" / \\"Nd\\" / \\"N\\")\\n    \\"\\"\\"\\n    src = source.lower().strip()\\n    ua = _user_agent()\\n\\n    # Unified normalization layer: returns PaperMetadata[] for all sources.\\n    if normalize:\\n        from scireason.papers import PaperSource as _PS, search_papers as _search\\n\\n        srcs = None\\n        if src not in (\\"all\\", \\"*\\"):\\n            parts = [p.strip() for p in src.split(\\",\\") if p.strip()]\\n            srcs = []\\n            for p in parts:\\n                # map legacy shorthand\\n                if p in (\\"s2\\", \\"semanticscholar\\"):\\n                    p = \\"semantic_scholar\\"\\n                if p in (\\"europepmc\\", \\"europe_pmc\\"):\\n                    p = \\"europe_pmc\\"\\n                try:\\n                    srcs.append(_PS(p))\\n                except Exception:\\n                    continue\\n\\n        papers = _search(query, limit=limit, sources=srcs, with_abstracts=with_abstract)\\n        # PaperMetadata contains `published_date: date` → use Pydantic JSON mode\\n        # so standard `json.dumps(...)` does not fail.\\n        data = [p.model_dump(mode=\\"json\\") for p in papers]\\n\\n        out.parent.mkdir(parents=True, exist_ok=True)\\n        out.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n        console.print(f\\"[green]Saved (normalized):[/green] {out}\\")\\n        return\\n\\n    # Legacy raw mode (source-specific outputs)\\n    crossref_mailto = settings.crossref_mailto or settings.contact_email\\n    openalex_mailto = settings.openalex_mailto or settings.contact_email\\n    ncbi_email = settings.ncbi_email or settings.contact_email\\n\\n    if src == \\"arxiv\\":\\n        data = arxiv_search(query=query, max_results=limit, user_agent=ua)\\n    elif src == \\"openalex\\":\\n        data = openalex_search(query=query, per_page=limit, mailto=openalex_mailto, api_key=settings.openalex_api_key, user_agent=ua)\\n    elif src in (\\"s2\\", \\"semanticscholar\\", \\"semantic_scholar\\"):\\n        data = s2_search(query=query, limit=limit, api_key=settings.s2_api_key)\\n    elif src == \\"crossref\\":\\n        data = crossref_search(query=query, rows=limit, mailto=crossref_mailto, user_agent=ua)\\n    elif src == \\"pubmed\\":\\n        data = pubmed_search(\\n            query,\\n            retmax=limit,\\n            api_key=settings.ncbi_api_key,\\n            tool=settings.ncbi_tool,\\n            email=ncbi_email,\\n            with_abstract=with_abstract,\\n        )\\n    elif src in (\\"europepmc\\", \\"europe_pmc\\"):\\n        data = europe_pmc_search(query=query, page_size=limit, user_agent=ua)\\n    elif src in (\\"biorxiv\\", \\"medrxiv\\"):\\n        recs = biorxiv_search_details(\\n            query=query,\\n            server=src,\\n            cursor=cursor,\\n            category=category,\\n            user_agent=ua,\\n        )\\n        data = recs\\n    else:\\n        raise typer.BadParameter(\\n            \\"source must be one of: arxiv, openalex, s2, crossref, pubmed, europepmc, biorxiv, medrxiv\\"\\n        )\\n\\n    out.parent.mkdir(parents=True, exist_ok=True)\\n    out.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    console.print(f\\"[green]Saved:[/green] {out}\\")\\n@app.command()\\ndef parse(\\n\\n    pdf: Path = typer.Option(..., help=\\"Путь к PDF\\"),\\n    meta: Path = typer.Option(..., help=\\"Путь к meta.json\\"),\\n    out_dir: Path = typer.Option(Path(\\"data/processed/papers\\"), help=\\"Корневая папка для paper_dir\\"),\\n) -> None:\\n    \\"\\"\\"Парсинг PDF через GROBID и сохранение чанков.\\"\\"\\"\\n    meta_obj = json.loads(meta.read_text(encoding=\\"utf-8\\"))\\n    # best-effort normalize\\n    meta_obj = _normalize_meta(meta_obj, fallback_id=pdf.stem, source=str(meta_obj.get(\\"source\\") or \\"\\"))\\n    paper_dir = ingest_pdf(pdf_path=pdf, meta=meta_obj, out_dir=out_dir)\\n    console.print(f\\"[green]Paper stored:[/green] {paper_dir}\\")\\n\\n\\n@app.command(\\"parse-mm\\")\\ndef parse_mm(\\n    pdf: Path = typer.Option(..., help=\\"Путь к PDF\\"),\\n    meta: Path = typer.Option(..., help=\\"Путь к meta.json\\"),\\n    out_dir: Path = typer.Option(Path(\\"data/processed/papers\\"), help=\\"Корневая папка для paper_dir\\"),\\n    vlm: bool = typer.Option(True, help=\\"Включить VLM подписи/таблицы/формулы\\"),\\n) -> None:\\n    \\"\\"\\"Парсинг PDF + мультимодальность (страницы/картинки).\\"\\"\\"\\n    meta_obj = json.loads(meta.read_text(encoding=\\"utf-8\\"))\\n    meta_obj = _normalize_meta(meta_obj, fallback_id=pdf.stem, source=str(meta_obj.get(\\"source\\") or \\"\\"))\\n    paper_dir = ingest_pdf_multimodal(pdf_path=pdf, meta=meta_obj, out_dir=out_dir, run_vlm=vlm)\\n    console.print(f\\"[green]Paper stored (mm):[/green] {paper_dir}\\")\\n\\n\\n@app.command(\\"ingest-arxiv\\")\\ndef ingest_arxiv(\\n    arxiv_id: str = typer.Argument(..., help=\\"arXiv id или URL (например 2401.01234 или https://arxiv.org/abs/2401.01234)\\"),\\n    raw_dir: Path = typer.Option(Path(\\"data/raw/papers\\"), help=\\"Куда скачать PDF\\"),\\n    meta_dir: Path = typer.Option(Path(\\"data/raw/metadata\\"), help=\\"Куда сохранить metadata JSON\\"),\\n    processed_dir: Path = typer.Option(Path(\\"data/processed/papers\\"), help=\\"Куда сохранить обработанные paper_dir\\"),\\n    multimodal: bool = typer.Option(True, help=\\"Использовать мультимодальный ingest (страницы/таблицы/формулы).\\"),\\n    build_graph: bool = typer.Option(True, help=\\"После ingest собрать temporal+mm граф (TG-MMKG).\\"),\\n    collection_text: Optional[str] = typer.Option(None, help=\\"Qdrant коллекция (текст). По умолчанию из domain config.\\"),\\n    collection_mm: Optional[str] = typer.Option(None, help=\\"Qdrant коллекция (multimodal). По умолчанию <text>_mm.\\"),\\n) -> None:\\n    \\"\\"\\"Ingestion “в один клик”: скачать arXiv PDF + metadata resolver + ingest pipeline.\\"\\"\\"\\n    arxiv_norm = normalize_arxiv_id(arxiv_id)\\n    pdf_path, meta_path = one_click_ingest_arxiv(arxiv_id=arxiv_norm, raw_dir=raw_dir, meta_dir=meta_dir)\\n\\n    console.print(f\\"[green]Downloaded:[/green] {pdf_path}\\")\\n    console.print(f\\"[green]Metadata:[/green] {meta_path}\\")\\n\\n    meta_obj = json.loads(meta_path.read_text(encoding=\\"utf-8\\"))\\n    meta_obj = _normalize_meta(\\n        meta_obj,\\n        fallback_id=f\\"arxiv:{arxiv_norm}\\",\\n        source=\\"arxiv\\",\\n    )\\n    # make id stable and compact\\n    meta_obj[\\"id\\"] = meta_obj.get(\\"doi\\") and f\\"doi:{meta_obj['doi']}\\" or f\\"arxiv:{arxiv_norm}\\"\\n\\n    if multimodal:\\n        paper_dir = ingest_pdf_multimodal(pdf_path=pdf_path, meta=meta_obj, out_dir=processed_dir, run_vlm=True)\\n    else:\\n        paper_dir = ingest_pdf(pdf_path=pdf_path, meta=meta_obj, out_dir=processed_dir)\\n\\n    console.print(f\\"[green]Ingested:[/green] {paper_dir}\\")\\n\\n    if build_graph:\\n        domain_cfg = load_domain_config()\\n        ct = collection_text or (domain_cfg.kg.get(\\"collection\\") if domain_cfg.kg else None) or \\"demo\\"\\n        cm = collection_mm\\n        if cm is None:\\n            cm = f\\"{ct}_mm\\"\\n        build_temporal_and_multimodal(\\n            paper_dir=paper_dir,\\n            collection_text=ct,\\n            collection_mm=cm if multimodal else None,\\n            domain=domain_cfg.title,\\n        )\\n        console.print(\\"[green]TG-MMKG built.[/green]\\")\\n\\n\\n@app.command(\\"build-tg-mmkg\\")\\ndef build_tg_mmkg(\\n    paper_dir: Path = typer.Option(..., help=\\"Папка paper_dir (meta.json + chunks.jsonl + optional mm/)\\"),\\n    collection_text: str = typer.Option(\\"demo\\", help=\\"Qdrant коллекция (текст)\\"),\\n    collection_mm: Optional[str] = typer.Option(None, help=\\"Qdrant коллекция (multimodal). None => skip mm index.\\"),\\n    domain: str = typer.Option(\\"Science\\", help=\\"Доменные подсказки для LLM при извлечении утверждений\\"),\\n    max_chunks_for_triplets: int = typer.Option(16, help=\\"Сколько чанков использовать для извлечения темпоральных триплетов\\"),\\n) -> None:\\n    \\"\\"\\"Строит Temporal KG + (опционально) Multimodal индекс для одной статьи.\\"\\"\\"\\n    build_temporal_and_multimodal(\\n        paper_dir=paper_dir,\\n        collection_text=collection_text,\\n        collection_mm=collection_mm,\\n        domain=domain,\\n        max_chunks_for_triplets=max_chunks_for_triplets,\\n    )\\n    console.print(\\"[green]Done.[/green]\\")\\n\\n\\n@app.command(\\"build-corpus\\")\\ndef build_corpus(\\n    papers_dir: Path = typer.Option(Path(\\"data/processed/papers\\"), help=\\"Корневая папка с множеством paper_dir\\"),\\n    collection_text: Optional[str] = typer.Option(None, help=\\"Qdrant коллекция (текст). Default: domain config.\\"),\\n    collection_mm: Optional[str] = typer.Option(None, help=\\"Qdrant коллекция (multimodal). Default: <text>_mm.\\"),\\n    domain: Optional[str] = typer.Option(None, help=\\"Domain hint. Default: domain config title.\\"),\\n    max_papers: int = typer.Option(0, help=\\"Если >0 — ограничить число обработанных статей\\"),\\n    max_chunks_for_triplets: int = typer.Option(16, help=\\"Сколько чанков на статью использовать для извлечения триплетов\\"),\\n) -> None:\\n    \\"\\"\\"Build TG-MMKG for *all* processed papers in a directory.\\n\\n    This is the recommended entry point for batch ingestion once you have a folder of `paper_dir`.\\n    \\"\\"\\"\\n    domain_cfg = load_domain_config()\\n    ct = collection_text or (domain_cfg.kg.get(\\"collection\\") if domain_cfg.kg else None) or \\"demo\\"\\n    cm = collection_mm\\n    if cm is None:\\n        cm = f\\"{ct}_mm\\"\\n    dom = domain or domain_cfg.title\\n\\n    paper_dirs = sorted([p for p in papers_dir.iterdir() if p.is_dir() and (p / \\"meta.json\\").exists()])\\n    if max_papers and max_papers > 0:\\n        paper_dirs = paper_dirs[:max_papers]\\n\\n    if not paper_dirs:\\n        console.print(f\\"[yellow]No paper_dir found in {papers_dir}[/yellow]\\")\\n        raise typer.Exit(code=1)\\n\\n    console.print(f\\"[cyan]Building corpus:[/cyan] papers={len(paper_dirs)} text_collection={ct} mm_collection={cm}\\")\\n\\n    for i, pd in enumerate(paper_dirs, start=1):\\n        console.print(f\\"[dim]({i}/{len(paper_dirs)})[/dim] {pd}\\")\\n        try:\\n            pages_path = pd / \\"mm\\" / \\"pages.jsonl\\"\\n            has_mm = pages_path.exists()\\n            build_temporal_and_multimodal(\\n                paper_dir=pd,\\n                collection_text=ct,\\n                collection_mm=(cm if has_mm else None),\\n                domain=dom,\\n                max_chunks_for_triplets=max_chunks_for_triplets,\\n            )\\n        except Exception as e:\\n            console.print(f\\"[yellow]Skip {pd.name}: {e}[/yellow]\\")\\n\\n    console.print(\\"[green]Corpus build finished.[/green]\\")\\n\\n\\n@app.command(\\"build-kg\\")\\ndef build_kg(\\n    paper_dir: Path = typer.Option(..., help=\\"Папка paper_dir (meta.json + chunks.jsonl)\\"),\\n    collection: str = typer.Option(\\"demo\\", help=\\"Qdrant коллекция (текст)\\"),\\n    domain: str = typer.Option(\\"Science\\", help=\\"Доменные подсказки для LLM\\"),\\n) -> None:\\n    \\"\\"\\"Строит обычный KG в Neo4j и эмбеддинги в Qdrant из paper_dir.\\"\\"\\"\\n    build_from_paper_dir(paper_dir=paper_dir, collection=collection, domain=domain)\\n    console.print(\\"[green]Done.[/green]\\")\\n\\n\\n@app.command()\\ndef debate(\\n    query: str,\\n    collection: str = typer.Option(\\"demo\\", help=\\"Qdrant коллекция (текст)\\"),\\n    domain: str = typer.Option(\\"Science\\", help=\\"Domain hint for agents\\"),\\n    k: int = typer.Option(8, help=\\"Сколько документов достать в контекст\\"),\\n    max_rounds: int = typer.Option(3, help=\\"Сколько раундов дебатов\\"),\\n    allow_empty_context: bool = typer.Option(\\n        False,\\n        help=(\\n            \\"Разрешить запуск дебатов без найденного контекста (например, если коллекция ещё не создана). \\"\\n            \\"По умолчанию команда подскажет, как собрать коллекцию, и завершится с ошибкой.\\"\\n        ),\\n    ),\\n) -> None:\\n    \\"\\"\\"GraphRAG: достать контекст + дебаты агентов -> гипотеза.\\"\\"\\"\\n    try:\\n        ctx = retrieve_context(collection=collection, query=query, limit=k)\\n    except Exception as e:\\n        console.print(\\n            \\"[red]Failed to retrieve context from Qdrant.[/red] \\"\\n            \\"Make sure Qdrant is running and the collection is built (parse + build-kg).\\"\\n        )\\n        console.print(f\\"[dim]{e}[/dim]\\")\\n        if not allow_empty_context:\\n            raise typer.Exit(code=1)\\n        ctx = []\\n\\n    if not ctx and not allow_empty_context:\\n        console.print(\\n            \\"[yellow]No context chunks were found.[/yellow] \\"\\n            \\"Run `top-papers-graph parse ...` and `top-papers-graph build-kg ...` first, \\"\\n            \\"or pass --allow-empty-context to proceed without retrieval.\\"\\n        )\\n        raise typer.Exit(code=1)\\n    context_text = \\"\\\\n\\\\n\\".join(\\n        [f\\"[{c['payload'].get('paper_id')}] {c['payload'].get('text')}\\" for c in ctx]\\n    )\\n    res = run_debate(domain=domain, context=context_text, max_rounds=max_rounds)\\n    console.print(res.model_dump_json(indent=2, ensure_ascii=False))\\n\\n\\n@app.command(\\"test-hypothesis\\")\\ndef test_hypothesis_cmd(\\n    hypothesis_json: Path = typer.Option(..., help=\\"Путь к JSON (HypothesisDraft или DebateResult из команды debate)\\"),\\n    collection: str = typer.Option(\\"demo\\", help=\\"Qdrant коллекция (текст)\\"),\\n    domain: Optional[str] = typer.Option(None, help=\\"Domain hint. Default: domain config title.\\"),\\n    k: int = typer.Option(12, help=\\"Сколько чанков извлечь для проверки\\"),\\n) -> None:\\n    \\"\\"\\"Проверить (verification) гипотезу на основе литературы и темпорального KG.\\"\\"\\"\\n    domain_cfg = load_domain_config()\\n    dom = domain or domain_cfg.title\\n\\n    hyp = load_hypothesis_from_json(hypothesis_json)\\n    result = test_hypothesis(domain=dom, hypothesis=hyp, collection_text=collection, k=k)\\n    console.print(result.model_dump_json(indent=2, ensure_ascii=False))\\n\\n\\n@app.command(\\"refresh-feedback\\")\\ndef refresh_feedback(\\n    graph_reviews_dir: Path = typer.Option(Path(\\"data/experts/graph_reviews\\"), help=\\"Папка с graph_reviews (JSON).\\"),\\n    out_path: Path = typer.Option(Path(\\"data/derived/expert_overrides.jsonl\\"), help=\\"Куда сохранить overrides (JSONL).\\"),\\n) -> None:\\n    \\"\\"\\"Graph reviews → overrides (для мгновенного эффекта на reward/retriever).\\"\\"\\"\\n    stats = compile_overrides(graph_reviews_dir, out_path)\\n    console.print(f\\"[green]Compiled overrides:[/green] {out_path}\\")\\n    console.print(\\n        f\\"accepted={stats.accepted} rejected={stats.rejected} needs_fix={stats.needs_fix} added={stats.added}\\"\\n    )\\n\\n\\n@app.command(\\"apply-graph-reviews\\")\\ndef apply_graph_reviews(\\n    graph_reviews_dir: Path = typer.Option(Path(\\"data/experts/graph_reviews\\"), help=\\"Папка с graph_reviews (JSON).\\"),\\n    overrides_path: Path = typer.Option(Path(\\"data/derived/expert_overrides.jsonl\\"), help=\\"Куда сохранить overrides (JSONL).\\"),\\n    to_neo4j: bool = typer.Option(False, help=\\"Проставить вердикты/веса на Assertion-нодах в Neo4j.\\"),\\n) -> None:\\n    \\"\\"\\"Собрать overrides и (опционально) применить к Neo4j.\\"\\"\\"\\n    stats = compile_overrides(graph_reviews_dir, overrides_path)\\n    console.print(f\\"[green]Compiled overrides:[/green] {overrides_path}\\")\\n    console.print(\\n        f\\"accepted={stats.accepted} rejected={stats.rejected} needs_fix={stats.needs_fix} added={stats.added}\\"\\n    )\\n\\n    if not to_neo4j:\\n        return\\n\\n    try:\\n        tneo = Neo4jTemporalStore()\\n        tneo.ensure_schema()\\n\\n        count = 0\\n        for line in overrides_path.read_text(encoding=\\"utf-8\\").splitlines():\\n            if not line.strip():\\n                continue\\n            rec = json.loads(line)\\n            tneo.apply_expert_override(\\n                subj=str(rec.get(\\"subject\\")),\\n                pred=str(rec.get(\\"predicate\\")),\\n                obj=str(rec.get(\\"object\\")),\\n                verdict=str(rec.get(\\"verdict\\")),\\n                weight=float(rec.get(\\"weight\\", 0.0)),\\n                time_interval=str(rec.get(\\"time_interval\\", \\"unknown\\")),\\n                start_date=str(rec.get(\\"start_date\\", \\"unknown\\")),\\n                end_date=str(rec.get(\\"end_date\\", \\"unknown\\")),\\n                valid_from=str(rec.get(\\"valid_from\\", rec.get(\\"start_date\\", \\"unknown\\"))),\\n                valid_to=str(rec.get(\\"valid_to\\", \\"+inf\\")),\\n                time_source=str(rec.get(\\"time_source\\", \\"unknown\\")),\\n            )\\n            count += 1\\n        tneo.close()\\n        console.print(f\\"[green]Applied to Neo4j:[/green] {count} overrides\\")\\n    except Exception as e:\\n        console.print(f\\"[red]Neo4j apply failed:[/red] {e}\\")\\n\\n\\n@app.command(\\"apply-temporal-corrections\\")\\ndef apply_temporal_corrections(\\n    corrections_dir: Path = typer.Option(\\n        Path(\\"data/experts/temporal_corrections\\"), help=\\"Папка с temporal_corrections (JSON).\\"\\n    ),\\n    dry_run: bool = typer.Option(False, help=\\"Не писать в Neo4j, только показать план изменений.\\"),\\n) -> None:\\n    \\"\\"\\"Применить temporal_corrections к Neo4j Temporal KG.\\n\\n    Т.к. assertion_id включает время, исправление времени реализовано как:\\n    old_assertion -[:REPLACED_BY]-> new_assertion.\\n    \\"\\"\\"\\n\\n    paths = sorted(corrections_dir.glob(\\"**/*.json\\"))\\n    if not paths:\\n        console.print(f\\"[yellow]No JSON files found in {corrections_dir}[/yellow]\\")\\n        return\\n\\n    tneo = Neo4jTemporalStore()\\n    tneo.ensure_schema()\\n\\n    total = 0\\n    created = 0\\n    replaced = 0\\n\\n    for p in paths:\\n        doc = json.loads(p.read_text(encoding=\\"utf-8\\"))\\n        reviewer_id = str(doc.get(\\"reviewer_id\\", \\"\\"))\\n        for corr in doc.get(\\"corrections\\", []):\\n            total += 1\\n            old_id = str(corr.get(\\"assertion_id\\", \\"\\")).strip()\\n            if not old_id:\\n                continue\\n\\n            details = tneo.get_assertion_details(old_id)\\n            if not details:\\n                console.print(f\\"[yellow]Skip:[/yellow] cannot find assertion {old_id}\\")\\n                continue\\n\\n            try:\\n                corrected_time = TimeInterval.model_validate(corr.get(\\"corrected_time\\"))\\n            except Exception as e:\\n                console.print(f\\"[yellow]Skip:[/yellow] invalid corrected_time for {old_id} ({e})\\")\\n                continue\\n\\n            t = TemporalTriplet(\\n                subject=str(details.get(\\"subject\\")),\\n                predicate=str(details.get(\\"predicate\\")),\\n                object=str(details.get(\\"object\\")),\\n                confidence=float(details.get(\\"confidence\\") or 0.5),\\n                polarity=str(details.get(\\"polarity\\") or \\"unknown\\"),\\n                time=corrected_time,\\n                evidence_quote=str(corr.get(\\"evidence_quote\\") or details.get(\\"evidence_quote\\") or \\"\\").strip() or None,\\n            )\\n\\n            paper_id = str(details.get(\\"paper_id\\"))\\n            if dry_run:\\n                console.print(\\n                    f\\"[cyan]DRY RUN[/cyan] {old_id} -> time {corrected_time.start}-{corrected_time.end} ({corrected_time.granularity})\\"\\n                )\\n                continue\\n\\n            new_id = tneo.upsert_assertion(paper_id=paper_id, t=t, evidence_quote=t.evidence_quote)\\n            created += 1\\n            tneo.link_replacement(\\n                old_id,\\n                new_id,\\n                rationale=str(corr.get(\\"rationale\\", \\"\\")).strip(),\\n                reviewer_id=reviewer_id,\\n            )\\n            replaced += 1\\n\\n    tneo.close()\\n    console.print(\\n        f\\"[green]Temporal corrections processed[/green]: total={total}, new_assertions={created}, replaced_links={replaced}\\"\\n    )\\n\\n\\n@app.command(\\"prepare-task2-validation\\")\\ndef prepare_task2_validation(\\n    trajectory: Path = typer.Option(..., help=\\"Путь к YAML артефакту Task 1 (trajectory).\\"),\\n    out_dir: Path = typer.Option(Path(\\"runs/task2_validation\\"), help=\\"Куда сохранить bundle для эксперта.\\"),\\n    multimodal: bool = typer.Option(True, help=\\"Пробовать мультимодальный ingest PDF (страницы + VLM при наличии).\\"),\\n    vlm: bool = typer.Option(True, help=\\"Запускать VLM на страницах PDF, если VLM backend настроен.\\"),\\n    edge_mode: str = typer.Option(\\"auto\\", help=\\"auto|llm_triplets|cooccurrence\\"),\\n    suggest_links: bool = typer.Option(True, help=\\"Добавить scout/suggested_links.json для поиска дополнительных ссылок.\\"),\\n    max_papers: int = typer.Option(0, help=\\"Если >0 — ограничить число статей из trajectory YAML.\\"),\\n    max_link_queries: int = typer.Option(4, help=\\"Сколько topic/next_question запросов использовать для scout.\\"),\\n) -> None:\\n    \\"\\"\\"Task 2 orchestrator: trajectory YAML -> reference graph + automatic temporal KG + review templates.\\n\\n    Command is designed for Google Colab / notebook usage and does not require Neo4j/Qdrant.\\n    \\"\\"\\"\\n    bundle_dir = prepare_task2_validation_bundle(\\n        trajectory,\\n        out_dir=out_dir,\\n        include_multimodal=multimodal,\\n        run_vlm=vlm,\\n        edge_mode=edge_mode,\\n        suggest_links=suggest_links,\\n        max_papers=max_papers,\\n        max_link_queries=max_link_queries,\\n    )\\n    console.print(f\\"[green]Task 2 bundle prepared:[/green] {bundle_dir}\\")\\n\\n\\n@app.command(\\"pybamm-fastcharge\\")\\ndef pybamm_fastcharge(\\n    profile: str = typer.Option(\\"baseline_cc\\", help=\\"baseline_cc|proposed_two_stage|...\\"),\\n    profiles_dir: Path | None = typer.Option(\\n        None,\\n        \\"--profiles-dir\\",\\n        envvar=\\"CHARGING_PROFILES_DIR\\",\\n        help=\\"Папка с YAML профилями зарядки (можно задать через CHARGING_PROFILES_DIR)\\",\\n    ),\\n    out_dir: Path = typer.Option(Path(\\"results/pybamm/run\\"), help=\\"Куда сохранить результаты\\"),\\n) -> None:\\n    \\"\\"\\"Запуск симуляции PyBaMM для профиля зарядки (пример: battery_fastcharge).\\"\\"\\"\\n    from .experiments.pybamm_fastcharge import run_simulation\\n\\n    out = run_simulation(profile_name=profile, out_dir=out_dir, config_dir=profiles_dir)\\n    console.print(f\\"[green]Saved metrics:[/green] {out}\\")\\n\\n\\n@app.command(\\"import-top-papers\\")\\ndef import_top_papers(\\n    inp: Path = typer.Option(..., help=\\"JSON файл, который выдаёт top-papers-bot\\"),\\n    out_dir: Path = typer.Option(Path(\\"configs/top_papers_meta\\"), help=\\"Куда сохранить meta-файлы\\"),\\n) -> None:\\n    \\"\\"\\"Импорт JSON из top-papers-bot в meta-файлы SciReason.\\"\\"\\"\\n    from .integrations.top_papers_import import export_meta_files\\n\\n    files = export_meta_files(inp, out_dir)\\n    console.print(f\\"[green]Generated meta files:[/green] {len(files)} → {out_dir}\\")\\n\\n\\n@app.command(\\"run\\")\\ndef run_cmd(\\n    query: str = typer.Option(..., help=\\"Пользовательский запрос (topic/query).\\"),\\n    domain_id: str = typer.Option(\\n        None,  # type: ignore[arg-type]\\n        help=\\"ID домена (configs/domains/<id>.yaml). По умолчанию берётся из .env (DOMAIN_ID) или science.\\",\\n    ),\\n    sources: str = typer.Option(\\n        \\"all\\",\\n        help=\\"Источники через запятую: all|openalex,semantic_scholar,crossref,arxiv,pubmed,europe_pmc,biorxiv.\\",\\n    ),\\n    search_limit: int = typer.Option(50, help=\\"Сколько результатов запросить у источников.\\"),\\n    top_papers: int = typer.Option(20, help=\\"Сколько лучших статей взять в пайплайн.\\"),\\n    out_dir: Path = typer.Option(Path(\\"runs\\"), help=\\"Куда сохранить артефакты запуска.\\"),\\n    multimodal: bool = typer.Option(False, help=\\"Извлекать страницы/картинки (MM) при наличии зависимостей.\\"),\\n    no_llm_hypotheses: bool = typer.Option(False, help=\\"Не использовать LLM для переформулировки гипотез.\\"),\\n\\n    # --- LLM overrides (CLI > env/config defaults) ---\\n    llm: Optional[str] = typer.Option(\\n        None,\\n        \\"--llm\\",\\n        help=(\\n            \\"Переопределить LLM одним флагом. Форматы: 'g4f:deepseek-r1', 'g4f:gpt-4o-mini', \\"\\n            \\"'local:llama3.2' (Ollama), 'ollama:llama3.2', или 'openai/gpt-4o-mini' (LiteLLM).\\"\\n        ),\\n    ),\\n    g4f_model: Optional[str] = typer.Option(\\n        None,\\n        \\"--g4f-model\\",\\n        help=\\"Явно использовать g4f с указанной моделью (например deepseek-r1).\\",\\n    ),\\n    local_model: Optional[str] = typer.Option(\\n        None,\\n        \\"--local-model\\",\\n        help=\\"Явно использовать локальную Ollama модель (например llama3.2).\\",\\n    ),\\n    llm_provider: Optional[str] = typer.Option(\\n        None,\\n        \\"--llm-provider\\",\\n        help=\\"Явно задать провайдера (g4f|ollama|openai|anthropic|...).\\",\\n    ),\\n    llm_model: Optional[str] = typer.Option(\\n        None,\\n        \\"--llm-model\\",\\n        help=\\"Явно задать имя модели провайдера.\\",\\n    ),\\n    smol_model_backend: Optional[str] = typer.Option(\\n        None,\\n        \\"--smol-model-backend\\",\\n        help=\\"smolagents model backend (scireason|transformers|g4f). Overrides SMOL_MODEL_BACKEND.\\",\\n    ),\\n    smol_model_id: Optional[str] = typer.Option(\\n        None,\\n        \\"--smol-model-id\\",\\n        help=\\"HF model id/path for smolagents TransformersModel. Overrides SMOL_MODEL_ID.\\",\\n    ),\\n) -> None:\\n    \\"\\"\\"Полностью автоматический пайплайн: query → papers → temporal KG → hypotheses.\\"\\"\\"\\n    # ---- Apply LLM overrides ----\\n    def _apply_llm_overrides() -> None:\\n        # 1) single-flag format\\n        if llm:\\n            raw = llm.strip()\\n\\n            # Accept provider/model as \\"provider:model\\" or \\"provider/model\\"\\n            if \\":\\" in raw:\\n                prov, model = raw.split(\\":\\", 1)\\n            elif \\"/\\" in raw:\\n                prov, model = raw.split(\\"/\\", 1)\\n            else:\\n                # No separator -> assume g4f model\\n                prov, model = \\"g4f\\", raw\\n\\n            prov = prov.strip().lower()\\n            model = model.strip()\\n\\n            if prov in {\\"local\\", \\"ollama\\"}:\\n                settings.llm_provider = \\"ollama\\"\\n                settings.llm_model = model\\n            elif prov == \\"g4f\\":\\n                settings.llm_provider = \\"g4f\\"\\n                settings.llm_model = model\\n            else:\\n                # LiteLLM-style provider/model\\n                settings.llm_provider = prov\\n                settings.llm_model = model\\n            return\\n\\n        # 2) convenience flags\\n        if local_model:\\n            settings.llm_provider = \\"ollama\\"\\n            settings.llm_model = local_model.strip()\\n            return\\n\\n        if g4f_model:\\n            settings.llm_provider = \\"g4f\\"\\n            settings.llm_model = g4f_model.strip()\\n            return\\n\\n        # 3) explicit provider/model flags\\n        if llm_provider:\\n            settings.llm_provider = llm_provider.strip()\\n        if llm_model:\\n            settings.llm_model = llm_model.strip()\\n\\n    # Apply overrides (CLI > env/config defaults)\\n    _apply_llm_overrides()\\n\\n    # smolagents model overrides (CLI > env)\\n    if smol_model_backend:\\n        settings.smol_model_backend = smol_model_backend.strip()\\n    if smol_model_id:\\n        settings.smol_model_id = smol_model_id.strip()\\n\\n    console.print(\\n        f\\"[bold]LLM:[/bold] {settings.llm_provider}/{settings.llm_model}  |  \\"\\n        f\\"[bold]Embeddings:[/bold] {getattr(settings, 'embed_provider', 'hash')}\\"\\n    )\\n\\n    did = domain_id or settings.domain_id or \\"science\\"\\n    src_list = None\\n    if sources.strip().lower() != \\"all\\":\\n        src_list = [s.strip() for s in sources.split(\\",\\") if s.strip()]\\n\\n    run_path = run_pipeline(\\n        query=query,\\n        domain_id=did,\\n        sources=src_list,\\n        search_limit=search_limit,\\n        top_papers=top_papers,\\n        run_dir=out_dir,\\n        include_multimodal=multimodal,\\n        use_llm_for_hypotheses=not no_llm_hypotheses,\\n    )\\n    console.print(f\\"[bold green]Artifacts saved:[/bold green] {run_path}\\")\\n\\n\\n@app.command(\\"export-temporal-events\\")\\ndef export_temporal_events(\\n    out: Path = typer.Option(Path(\\"runs/temporal_events.json\\"), help=\\"Where to save the exported event stream JSON.\\"),\\n    limit: int = typer.Option(5000, help=\\"Maximum number of Neo4j events to export.\\"),\\n) -> None:\\n    \\"\\"\\"Export the Event layer from Neo4j for temporal model training/evaluation.\\"\\"\\"\\n    store = Neo4jTemporalStore()\\n    try:\\n        rows = store.export_event_stream(limit=limit)\\n    finally:\\n        store.close()\\n    out.parent.mkdir(parents=True, exist_ok=True)\\n    out.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    console.print(f\\"[green]Exported events:[/green] {out} (n={len(rows)})\\")\\n\\n\\n@app.command(\\"train-tgn\\")\\ndef train_tgn(\\n    temporal_kg_json: Path = typer.Option(..., help=\\"Path to temporal_kg.json produced by the pipeline.\\"),\\n    out: Path = typer.Option(Path(\\"runs/tgn_predictions.json\\"), help=\\"Where to save top temporal link predictions.\\"),\\n    top_k: int = typer.Option(20, help=\\"Number of temporal link predictions to keep.\\"),\\n) -> None:\\n    \\"\\"\\"Train/evaluate the lightweight TGNN-style predictor on a temporal KG artifact.\\"\\"\\"\\n    from .temporal.temporal_kg_builder import TemporalKnowledgeGraph, EdgeStats, NodeStats\\n    from .tgnn.event_dataset import build_event_stream, chronological_split, event_stats\\n    from .tgnn.tgn_link_prediction import TGNLinkPredConfig, tgn_link_prediction\\n\\n    raw = json.loads(temporal_kg_json.read_text(encoding=\\"utf-8\\"))\\n    kg = TemporalKnowledgeGraph(meta=dict(raw.get(\\"meta\\") or {}))\\n    for n in raw.get(\\"nodes\\", []):\\n        term = str(n.get(\\"term\\") or \\"\\")\\n        if not term:\\n            continue\\n        kg.nodes[term] = NodeStats(term=term, doc_freq=int(n.get(\\"doc_freq\\") or 0), yearly_doc_freq=dict(n.get(\\"yearly_doc_freq\\") or {}))\\n    for e in raw.get(\\"edges\\", []):\\n        edge = EdgeStats(\\n            source=str(e.get(\\"source\\") or \\"\\"),\\n            target=str(e.get(\\"target\\") or \\"\\"),\\n            predicate=str(e.get(\\"predicate\\") or \\"may_relate_to\\"),\\n            directed=bool(e.get(\\"directed\\", True)),\\n            total_count=int(e.get(\\"total_count\\") or 0),\\n            yearly_count={int(k): int(v) for k, v in dict(e.get(\\"yearly_count\\") or {}).items()},\\n            confidence_sum=float(e.get(\\"mean_confidence\\") or 0.0) * max(1, int(e.get(\\"total_count\\") or 1)),\\n            confidence_n=max(1, int(e.get(\\"total_count\\") or 1)),\\n            polarity_counts=dict(e.get(\\"polarity_counts\\") or {\\"supports\\": 0, \\"contradicts\\": 0, \\"unknown\\": 0}),\\n            papers=set(e.get(\\"papers\\") or []),\\n            evidence_quotes=list(e.get(\\"evidence_quotes\\") or []),\\n            features=dict(e.get(\\"features\\") or {}),\\n            score=float(e.get(\\"score\\") or 0.0),\\n        )\\n        kg.edges.append(edge)\\n\\n    events = build_event_stream(kg)\\n    train_events, valid_events, test_events = chronological_split(events)\\n    preds = tgn_link_prediction(\\n        train_events + valid_events,\\n        top_k=top_k,\\n        config=TGNLinkPredConfig(\\n            recent_window_years=int(getattr(settings, \\"hyp_tgnn_recent_window_years\\", 3) or 3),\\n            recency_half_life_years=float(getattr(settings, \\"hyp_tgnn_half_life_years\\", 2.0) or 2.0),\\n            min_candidate_score=float(getattr(settings, \\"hyp_tgnn_min_candidate_score\\", 0.05) or 0.05),\\n        ),\\n    )\\n    payload = {\\n        \\"stats\\": {\\n            **event_stats(events),\\n            \\"train_events\\": len(train_events),\\n            \\"valid_events\\": len(valid_events),\\n            \\"test_events\\": len(test_events),\\n        },\\n        \\"predictions\\": [\\n            {\\"source\\": u, \\"target\\": v, \\"score\\": score}\\n            for u, v, score in preds\\n        ],\\n    }\\n    out.parent.mkdir(parents=True, exist_ok=True)\\n    out.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    console.print(f\\"[green]Saved TGNN predictions:[/green] {out}\\")\\n\\n\\n@app.command(\\"demo-run\\")\\ndef demo_run_cmd(\\n    query: str = typer.Option(\\"temporal knowledge graph hypothesis\\", help=\\"Demo query (offline).\\"),\\n    edge_mode: str = typer.Option(\\"cooccurrence\\", help=\\"cooccurrence|llm_triplets\\"),\\n    out_dir: Path = typer.Option(Path(\\"runs\\"), help=\\"Where to write demo artifacts.\\"),\\n    domain_id: str = typer.Option(None, help=\\"Domain config id (defaults to env DOMAIN_ID or science).\\"),\\n    no_llm_hypotheses: bool = typer.Option(False, help=\\"Disable LLM rewriting for hypotheses.\\"),\\n    tgnn: bool = typer.Option(True, help=\\"Enable TGNN/TGN-style temporal link prediction (default on).\\"),\\n    gnn: bool = typer.Option(False, help=\\"Enable optional static GNN baseline (requires '.[gnn]').\\"),\\n    agent_backend: Optional[str] = typer.Option(\\n        None,\\n        help=\\"Override HYP_AGENT_BACKEND for this run (internal|smolagents).\\",\\n    ),\\n    llm_provider: Optional[str] = typer.Option(None, help=\\"Override LLM_PROVIDER for this run (e.g. mock).\\"),\\n    llm_model: Optional[str] = typer.Option(None, help=\\"Override LLM_MODEL for this run.\\"),\\n    smol_model_backend: Optional[str] = typer.Option(\\n        None,\\n        \\"--smol-model-backend\\",\\n        help=\\"smolagents model backend (scireason|transformers|g4f). Overrides SMOL_MODEL_BACKEND.\\",\\n    ),\\n    smol_model_id: Optional[str] = typer.Option(\\n        None,\\n        \\"--smol-model-id\\",\\n        help=\\"HF model id/path for smolagents TransformersModel. Overrides SMOL_MODEL_ID.\\",\\n    ),\\n) -> None:\\n    \\"\\"\\"Offline demo pipeline: build temporal KG + hypotheses from a tiny built-in corpus.\\n\\n    This command is used for smoke tests and for the first classroom run without network/services.\\n    \\"\\"\\"\\n\\n    if llm_provider:\\n        settings.llm_provider = llm_provider.strip()\\n    if llm_model:\\n        settings.llm_model = llm_model.strip()\\n    if smol_model_backend:\\n        settings.smol_model_backend = smol_model_backend.strip()\\n    if smol_model_id:\\n        settings.smol_model_id = smol_model_id.strip()\\n\\n    settings.hyp_tgnn_enabled = bool(tgnn)\\n    if gnn:\\n        settings.hyp_gnn_enabled = True\\n\\n    if agent_backend:\\n        settings.hyp_agent_backend = agent_backend.strip()\\n\\n    did = domain_id or settings.domain_id or \\"science\\"\\n    run_path = run_demo_pipeline(\\n        query=query,\\n        domain_id=did,\\n        edge_mode=edge_mode,\\n        out_dir=out_dir,\\n        use_llm_for_hypotheses=not no_llm_hypotheses,\\n    )\\n    console.print(f\\"[bold green]Demo artifacts saved:[/bold green] {run_path}\\")\\n\\n\\n@app.command(\\"smoke-all\\")\\ndef smoke_all(\\n    out_dir: Path = typer.Option(Path(\\"runs\\"), help=\\"Where to write artifacts.\\"),\\n    include_g4f: bool = typer.Option(\\n        False,\\n        help=\\"Also run smoke with LLM_PROVIDER=g4f (requires '.[g4f]' and internet; can be unstable).\\",\\n    ),\\n    smol_model_backend: Optional[str] = typer.Option(\\n        None,\\n        \\"--smol-model-backend\\",\\n        help=\\"smolagents model backend for smolagents runs (scireason|transformers|g4f).\\",\\n    ),\\n    smol_model_id: Optional[str] = typer.Option(\\n        None,\\n        \\"--smol-model-id\\",\\n        help=\\"HF model id/path for smolagents TransformersModel.\\",\\n    ),\\n) -> None:\\n    \\"\\"\\"Run an offline smoke matrix for key pipeline branches.\\"\\"\\"\\n\\n    # Prefer deterministic offline mode by default.\\n    llm_providers = [\\"mock\\"]\\n    if include_g4f:\\n        llm_providers.append(\\"g4f\\")\\n\\n    # Try both agent backends if smolagents is available.\\n    import importlib.util\\n\\n    agent_backends = [\\"internal\\"]\\n    if importlib.util.find_spec(\\"smolagents\\") is not None:\\n        agent_backends.append(\\"smolagents\\")\\n\\n    # smolagents model overrides (CLI > env)\\n    if smol_model_backend:\\n        settings.smol_model_backend = smol_model_backend.strip()\\n    if smol_model_id:\\n        settings.smol_model_id = smol_model_id.strip()\\n\\n    combos = [\\n        (\\"cooccurrence\\", True, False),\\n        (\\"cooccurrence\\", False, False),\\n        (\\"llm_triplets\\", True, False),\\n        (\\"llm_triplets\\", False, False),\\n        # Optional GNN branch (best-effort; will fall back if PyG isn't installed)\\n        (\\"cooccurrence\\", True, True),\\n        (\\"cooccurrence\\", False, True),\\n    ]\\n    for llm_provider in llm_providers:\\n        settings.llm_provider = llm_provider\\n        settings.llm_model = \\"mock\\" if llm_provider == \\"mock\\" else (settings.llm_model or \\"auto\\")\\n\\n        for agent_backend in agent_backends:\\n            settings.hyp_agent_backend = agent_backend\\n            for edge_mode, no_llm, gnn in combos:\\n                settings.hyp_gnn_enabled = bool(gnn)\\n                console.print(\\n                    f\\"[cyan]Smoke[/cyan] llm_provider={llm_provider} agent_backend={agent_backend} edge_mode={edge_mode} no_llm_hypotheses={no_llm} gnn={gnn}\\"\\n                )\\n                rp = run_demo_pipeline(\\n                    query=\\"demo smoke\\",\\n                    domain_id=settings.domain_id or \\"science\\",\\n                    edge_mode=edge_mode,\\n                    out_dir=out_dir,\\n                    use_llm_for_hypotheses=not no_llm,\\n                )\\n                # Ensure key artifacts exist\\n                for f in [\\"paper_records.json\\", \\"temporal_kg.json\\", \\"hypotheses.json\\"]:\\n                    p = rp / f\\n                    if not p.exists():\\n                        raise RuntimeError(f\\"Smoke failed: missing {p}\\")\\n    console.print(\\"[bold green]Smoke-all: OK[/bold green]\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    app()\\n", "src/scireason/mm/vlm.py": "from __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom functools import lru_cache\\nfrom pathlib import Path\\nfrom typing import Optional, Literal\\n\\nfrom ..config import settings\\n\\n\\nBackend = Literal[\\"none\\", \\"qwen2_vl\\", \\"llava\\", \\"phi3_vision\\", \\"g4f\\"]\\n\\n\\n@dataclass\\nclass VLMResult:\\n    \\"\\"\\"Результат работы VL-модели по изображению страницы/фигуры.\\"\\"\\"\\n    caption: str\\n    extracted_tables_md: Optional[str] = None\\n    extracted_equations_md: Optional[str] = None\\n\\n\\ndef _require(pkg: str) -> None:\\n    raise RuntimeError(\\n        f\\"Для VLM backend нужна зависимость '{pkg}'.\\n\\"\\n        \\"Установите extras: pip install -e '.[mm]'\\n\\"\\n        \\"Или отключите VLM: export VLM_BACKEND=none\\"\\n    )\\n\\n\\n@lru_cache(maxsize=1)\\ndef _load_transformers_vlm(model_id: str):\\n    \\"\\"\\"Load and cache a HF vision-language model.\\n\\n    Qwen2.5-VL имеет собственный класс в Transformers, поэтому для него\\n    используем официальный путь через `Qwen2_5_VLForConditionalGeneration`.\\n    Для остальных backends оставляем AutoModelForVision2Seq fallback.\\n    \\"\\"\\"\\n    try:\\n        import torch  # type: ignore\\n        from transformers import AutoProcessor  # type: ignore\\n    except Exception:\\n        _require(\\"transformers/torch\\")\\n\\n    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)\\n\\n    if \\"Qwen/Qwen2.5-VL\\" in model_id or \\"Qwen2.5-VL\\" in model_id:\\n        try:\\n            from transformers import Qwen2_5_VLForConditionalGeneration  # type: ignore\\n        except Exception:\\n            _require(\\"transformers>=Qwen2.5-VL support\\")\\n        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(\\n            model_id,\\n            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\\n            device_map=\\"auto\\" if torch.cuda.is_available() else None,\\n            trust_remote_code=True,\\n        )\\n        return processor, model, \\"qwen2_5_vl\\"\\n\\n    try:\\n        from transformers import AutoModelForVision2Seq  # type: ignore\\n    except Exception:\\n        _require(\\"transformers/torch\\")\\n\\n    model = AutoModelForVision2Seq.from_pretrained(\\n        model_id,\\n        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\\n        device_map=\\"auto\\" if torch.cuda.is_available() else None,\\n        trust_remote_code=True,\\n    )\\n    return processor, model, \\"generic\\"\\n\\n\\ndef _describe_image_g4f(image_path: Path, prompt: str, model_id: str) -> VLMResult:\\n    try:\\n        import base64\\n        from g4f.client import Client as G4FClient  # type: ignore\\n    except Exception as e:  # pragma: no cover\\n        raise RuntimeError(\\"Для VLM backend='g4f' нужен пакет g4f (pip install -e '.[g4f]').\\") from e\\n\\n    client = G4FClient(api_key=getattr(settings, \\"g4f_api_key\\", None) or None)\\n    providers = None\\n    raw_providers = getattr(settings, \\"g4f_providers\\", None)\\n    if raw_providers:\\n        providers = [p.strip() for p in str(raw_providers).split(\\",\\") if p.strip()]\\n        if len(providers) == 1:\\n            providers = providers[0]\\n\\n    mime = \\"image/png\\"\\n    suffix = image_path.suffix.lower()\\n    if suffix in {\\".jpg\\", \\".jpeg\\"}:\\n        mime = \\"image/jpeg\\"\\n    elif suffix == \\".webp\\":\\n        mime = \\"image/webp\\"\\n\\n    data_url = f\\"data:{mime};base64,\\" + base64.b64encode(image_path.read_bytes()).decode(\\"ascii\\")\\n\\n    text = None\\n    errors = []\\n\\n    try:\\n        resp = client.chat.completions.create(\\n            model=model_id,\\n            provider=providers,\\n            messages=[\\n                {\\n                    \\"role\\": \\"user\\",\\n                    \\"content\\": [\\n                        {\\"type\\": \\"text\\", \\"text\\": prompt},\\n                        {\\"type\\": \\"image_url\\", \\"image_url\\": {\\"url\\": data_url}},\\n                    ],\\n                }\\n            ],\\n        )\\n        text = resp.choices[0].message.content if getattr(resp, \\"choices\\", None) else None\\n    except Exception as e:\\n        errors.append(f\\"openai_style={type(e).__name__}: {e}\\")\\n\\n    if not text:\\n        try:\\n            resp = client.chat.completions.create(\\n                model=model_id,\\n                provider=providers,\\n                messages=[{\\"role\\": \\"user\\", \\"content\\": prompt}],\\n                images=[[data_url, image_path.name]],\\n            )\\n            text = resp.choices[0].message.content if getattr(resp, \\"choices\\", None) else None\\n        except Exception as e:\\n            errors.append(f\\"images_arg={type(e).__name__}: {e}\\")\\n\\n    if not text:\\n        raise RuntimeError(\\"; \\".join(errors) or \\"g4f multimodal call returned empty response\\")\\n\\n    return VLMResult(caption=str(text).strip())\\n\\n\\ndef _describe_image_qwen(image_path: Path, prompt: str, model_id: str, max_new_tokens: int) -> VLMResult:\\n    try:\\n        import torch  # type: ignore\\n        from PIL import Image  # type: ignore\\n    except Exception:\\n        _require(\\"torch/pillow\\")\\n\\n    processor, model, family = _load_transformers_vlm(model_id)\\n    img = Image.open(image_path).convert(\\"RGB\\")\\n    full_prompt = (\\n        \\"Ты — научный ассистент. \\"\\n        \\"1) Дай краткую подпись к изображению (1-3 предложения). \\"\\n        \\"2) Если на изображении есть таблицы — извлеки их в Markdown. \\"\\n        \\"3) Если есть формулы — перепиши их в LaTeX. \\"\\n        f\\"\\\\n\\\\nЗадача/контекст: {prompt}\\"\\n    )\\n\\n    if family == \\"qwen2_5_vl\\":\\n        messages = [\\n            {\\n                \\"role\\": \\"user\\",\\n                \\"content\\": [\\n                    {\\"type\\": \\"image\\", \\"image\\": img},\\n                    {\\"type\\": \\"text\\", \\"text\\": full_prompt},\\n                ],\\n            }\\n        ]\\n        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\\n        inputs = processor(text=[text], images=[img], padding=True, return_tensors=\\"pt\\")\\n        if torch.cuda.is_available():\\n            inputs = {k: v.to(model.device) for k, v in inputs.items()}\\n        with torch.no_grad():\\n            generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)\\n        prompt_len = inputs[\\"input_ids\\"].shape[1]\\n        trimmed = [out_ids[prompt_len:] for out_ids in generated_ids]\\n        raw_text = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()\\n    else:\\n        inputs = processor(images=img, text=full_prompt, return_tensors=\\"pt\\")\\n        if torch.cuda.is_available():\\n            inputs = {k: v.to(model.device) for k, v in inputs.items()}\\n        with torch.no_grad():\\n            out = model.generate(**inputs, max_new_tokens=max_new_tokens)\\n        raw_text = processor.batch_decode(out, skip_special_tokens=True)[0].strip()\\n\\n    caption = raw_text\\n    tables_md = None\\n    equations_md = None\\n    m = caption.split(\\"TABLES:\\", 1)\\n    if len(m) == 2:\\n        caption, rest = m[0].strip(), m[1].strip()\\n        m2 = rest.split(\\"EQUATIONS:\\", 1)\\n        if len(m2) == 2:\\n            tables_md, equations_md = m2[0].strip(), m2[1].strip()\\n        else:\\n            tables_md = rest\\n\\n    return VLMResult(caption=caption, extracted_tables_md=tables_md, extracted_equations_md=equations_md)\\n\\n\\ndef describe_image(\\n    image_path: Path,\\n    prompt: str,\\n    backend: Optional[Backend] = None,\\n    model_id: Optional[str] = None,\\n    max_new_tokens: int = 512,\\n) -> VLMResult:\\n    \\"\\"\\"Описывает изображение (страница PDF / figure / table) через VL-модель.\\"\\"\\"\\n    backend = backend or settings.vlm_backend  # type: ignore[attr-defined]\\n    model_id = model_id or settings.vlm_model_id  # type: ignore[attr-defined]\\n\\n    if backend == \\"none\\":\\n        return VLMResult(caption=\\"\\")\\n\\n    if backend == \\"g4f\\":\\n        return _describe_image_g4f(image_path=image_path, prompt=prompt, model_id=model_id)\\n\\n    if backend == \\"qwen2_vl\\":\\n        return _describe_image_qwen(image_path=image_path, prompt=prompt, model_id=model_id, max_new_tokens=max_new_tokens)\\n\\n    return _describe_image_qwen(image_path=image_path, prompt=prompt, model_id=model_id, max_new_tokens=max_new_tokens)\\n", "src/scireason/pipeline/task2_validation.py": "from __future__ import annotations\\n\\nimport json\\nimport re\\nimport shutil\\nfrom dataclasses import asdict\\nfrom datetime import datetime\\nfrom difflib import SequenceMatcher\\nfrom pathlib import Path\\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple\\nfrom urllib.parse import urlparse, unquote\\n\\nimport yaml  # type: ignore\\nfrom rich.console import Console\\n\\nfrom ..domain import DomainConfig, load_domain_config\\nfrom ..ingest.acquire import AcquireResult, acquire_pdfs\\nfrom ..ingest.mm_pipeline import ingest_pdf_multimodal_auto\\nfrom ..ingest.pipeline import ingest_pdf_auto\\nfrom ..papers.schema import ExternalIds, PaperMetadata, PaperSource\\nfrom ..papers.service import get_paper_by_doi, search_papers\\nfrom ..temporal.temporal_kg_builder import PaperRecord, TemporalKnowledgeGraph, build_temporal_kg, load_papers_from_processed\\n\\n\\nconsole = Console()\\n\\n\\nDOI_RE = re.compile(r\\"10\\\\.\\\\d{4,9}/[-._;()/:A-Z0-9]+\\", re.IGNORECASE)\\nARXIV_RE = re.compile(r\\"(?:arxiv\\\\.org/(?:abs|pdf)/|arxiv:)?([a-z\\\\-]+/\\\\d{7}|\\\\d{4}\\\\.\\\\d{4,5})(?:v\\\\d+)?\\", re.IGNORECASE)\\nOPENALEX_RE = re.compile(r\\"(?:openalex\\\\.org/)?(W\\\\d+)\\", re.IGNORECASE)\\nPMID_RE = re.compile(r\\"(?:pubmed(?:\\\\.ncbi\\\\.nlm\\\\.nih\\\\.gov)?/)?(\\\\d{5,10})\\", re.IGNORECASE)\\nPMCID_RE = re.compile(r\\"(PMC\\\\d+)\\", re.IGNORECASE)\\nDATE_TOKEN_RE = re.compile(r\\"^(?:\\\\d{4}|\\\\d{4}-\\\\d{2}|\\\\d{4}-\\\\d{2}-\\\\d{2}|unknown|\\\\+inf|-inf)$\\")\\n\\n\\nDEFAULT_SEARCH_SOURCES = [\\n    PaperSource.semantic_scholar,\\n    PaperSource.openalex,\\n    PaperSource.crossref,\\n    PaperSource.pubmed,\\n    PaperSource.europe_pmc,\\n    PaperSource.arxiv,\\n    PaperSource.biorxiv,\\n]\\n\\n\\ndef _slugify(text: str) -> str:\\n    s = re.sub(r\\"[^a-z0-9\\\\-\\\\s_]+\\", \\"\\", (text or \\"\\").strip().lower())\\n    s = re.sub(r\\"\\\\s+\\", \\"-\\", s).strip(\\"-\\")\\n    return s[:80] or \\"task2\\"\\n\\n\\ndef _utc_now() -> str:\\n    return datetime.utcnow().strftime(\\"%Y-%m-%dT%H:%M:%SZ\\")\\n\\n\\ndef _norm_ws(text: str) -> str:\\n    return \\" \\".join((text or \\"\\").split())\\n\\n\\ndef _norm_title(text: str) -> str:\\n    s = _norm_ws(text).lower()\\n    s = re.sub(r\\"\\\\s+\\", \\" \\", s)\\n    s = re.sub(r\\"[^\\\\w\\\\s]+\\", \\"\\", s, flags=re.UNICODE)\\n    return s.strip()\\n\\n\\ndef _looks_like_url(text: str) -> bool:\\n    try:\\n        u = urlparse((text or \\"\\").strip())\\n    except Exception:\\n        return False\\n    return bool(u.scheme and u.netloc)\\n\\n\\ndef _extract_doi(text: str) -> Optional[str]:\\n    s = unquote((text or \\"\\").strip())\\n    m = DOI_RE.search(s)\\n    if not m:\\n        return None\\n    doi = m.group(0).rstrip(\\").,;]\\")\\n    return doi\\n\\n\\ndef _extract_arxiv(text: str) -> Optional[str]:\\n    s = unquote((text or \\"\\").strip())\\n    m = ARXIV_RE.search(s)\\n    if not m:\\n        return None\\n    return m.group(1)\\n\\n\\ndef _extract_openalex(text: str) -> Optional[str]:\\n    s = unquote((text or \\"\\").strip())\\n    m = OPENALEX_RE.search(s)\\n    if not m:\\n        return None\\n    return m.group(1).upper()\\n\\n\\ndef _extract_pmid(text: str) -> Optional[str]:\\n    s = unquote((text or \\"\\").strip())\\n    m = PMID_RE.search(s)\\n    if not m:\\n        return None\\n    return m.group(1)\\n\\n\\ndef _extract_pmcid(text: str) -> Optional[str]:\\n    s = unquote((text or \\"\\").strip())\\n    m = PMCID_RE.search(s)\\n    if not m:\\n        return None\\n    return m.group(1).upper()\\n\\n\\ndef _paper_year(meta: PaperMetadata) -> Optional[int]:\\n    if meta.year is not None:\\n        try:\\n            return int(meta.year)\\n        except Exception:\\n            return None\\n    if meta.published_date is not None:\\n        try:\\n            return int(meta.published_date.year)\\n        except Exception:\\n            return None\\n    return None\\n\\n\\ndef _score_title_match(a: str, b: str) -> float:\\n    na = _norm_title(a)\\n    nb = _norm_title(b)\\n    if not na or not nb:\\n        return 0.0\\n    if na == nb:\\n        return 1.0\\n    return SequenceMatcher(None, na, nb).ratio()\\n\\n\\ndef _pick_best_candidate(entry: Dict[str, Any], candidates: Sequence[PaperMetadata]) -> Optional[PaperMetadata]:\\n    title = str(entry.get(\\"title\\") or \\"\\")\\n    year_raw = entry.get(\\"year\\")\\n    try:\\n        entry_year = int(year_raw) if year_raw not in (None, \\"\\") else None\\n    except Exception:\\n        entry_year = None\\n    raw_id = str(entry.get(\\"id\\") or \\"\\")\\n    doi_hint = _extract_doi(raw_id)\\n    arxiv_hint = _extract_arxiv(raw_id)\\n    openalex_hint = _extract_openalex(raw_id)\\n\\n    scored: List[Tuple[float, PaperMetadata]] = []\\n    for cand in candidates:\\n        score = 0.0\\n        if title:\\n            score += 4.0 * _score_title_match(title, cand.title)\\n        cy = _paper_year(cand)\\n        if entry_year is not None and cy is not None:\\n            score += max(0.0, 1.0 - 0.25 * abs(entry_year - cy))\\n        if doi_hint and cand.ids and cand.ids.doi and cand.ids.doi.lower() == doi_hint.lower():\\n            score += 5.0\\n        if arxiv_hint and cand.ids and cand.ids.arxiv and cand.ids.arxiv.lower() == arxiv_hint.lower():\\n            score += 4.0\\n        if openalex_hint and cand.ids and cand.ids.openalex and cand.ids.openalex.upper() == openalex_hint.upper():\\n            score += 4.0\\n        if raw_id and cand.id.lower() == raw_id.lower():\\n            score += 5.0\\n        if cand.pdf_url:\\n            score += 0.25\\n        if cand.abstract:\\n            score += 0.1\\n        scored.append((score, cand))\\n\\n    if not scored:\\n        return None\\n    scored.sort(key=lambda x: x[0], reverse=True)\\n    best_score, best = scored[0]\\n    if best_score < 1.75:\\n        return None\\n    return best\\n\\n\\ndef _fallback_paper(entry: Dict[str, Any]) -> PaperMetadata:\\n    raw_id = str(entry.get(\\"id\\") or \\"\\").strip()\\n    title = str(entry.get(\\"title\\") or raw_id or \\"Untitled paper\\").strip()\\n    try:\\n        year = int(entry.get(\\"year\\")) if entry.get(\\"year\\") not in (None, \\"\\") else None\\n    except Exception:\\n        year = None\\n\\n    canonical_id = raw_id\\n    if not canonical_id:\\n        canonical_id = f\\"manual:{_slugify(title)}\\"\\n    elif _extract_doi(canonical_id):\\n        canonical_id = f\\"doi:{_extract_doi(canonical_id)}\\"\\n    elif _extract_arxiv(canonical_id):\\n        canonical_id = f\\"arxiv:{_extract_arxiv(canonical_id)}\\"\\n    elif _extract_openalex(canonical_id):\\n        canonical_id = f\\"openalex:{_extract_openalex(canonical_id)}\\"\\n    elif _extract_pmcid(canonical_id):\\n        canonical_id = f\\"pmc:{_extract_pmcid(canonical_id)}\\"\\n    elif _extract_pmid(canonical_id) and not _looks_like_url(canonical_id):\\n        canonical_id = f\\"pmid:{_extract_pmid(canonical_id)}\\"\\n    elif _looks_like_url(canonical_id):\\n        canonical_id = canonical_id\\n    else:\\n        canonical_id = f\\"manual:{_slugify(canonical_id)}\\"\\n\\n    ids = ExternalIds(\\n        doi=_extract_doi(raw_id),\\n        arxiv=_extract_arxiv(raw_id),\\n        openalex=_extract_openalex(raw_id),\\n        pmid=_extract_pmid(raw_id) if not _looks_like_url(raw_id) else None,\\n        pmcid=_extract_pmcid(raw_id),\\n    )\\n\\n    return PaperMetadata(\\n        id=canonical_id,\\n        source=PaperSource.unknown,\\n        title=title,\\n        abstract=None,\\n        year=year,\\n        url=raw_id if _looks_like_url(raw_id) else None,\\n        pdf_url=None,\\n        ids=ids,\\n    )\\n\\n\\ndef _resolve_entry_by_exact_identifier(entry: Dict[str, Any]) -> Optional[PaperMetadata]:\\n    raw_id = str(entry.get(\\"id\\") or \\"\\").strip()\\n    if not raw_id:\\n        return None\\n\\n    doi = _extract_doi(raw_id)\\n    if doi:\\n        try:\\n            meta = get_paper_by_doi(doi)\\n            if meta is not None:\\n                return meta\\n        except Exception:\\n            pass\\n\\n    title = str(entry.get(\\"title\\") or \\"\\").strip()\\n    try:\\n        candidates = search_papers(title or raw_id, limit=8, sources=DEFAULT_SEARCH_SOURCES)\\n    except Exception:\\n        candidates = []\\n\\n    best = _pick_best_candidate(entry, candidates)\\n    if best is not None:\\n        return best\\n\\n    return None\\n\\n\\ndef resolve_papers_from_trajectory(doc: Dict[str, Any], *, search_limit: int = 8) -> List[PaperMetadata]:\\n    resolved: List[PaperMetadata] = []\\n    seen_ids: set[str] = set()\\n\\n    for entry in doc.get(\\"papers\\", []) or []:\\n        if not isinstance(entry, dict):\\n            continue\\n        meta = _resolve_entry_by_exact_identifier(entry)\\n        if meta is None:\\n            title = str(entry.get(\\"title\\") or \\"\\").strip()\\n            if title:\\n                try:\\n                    candidates = search_papers(title, limit=search_limit, sources=DEFAULT_SEARCH_SOURCES)\\n                except Exception:\\n                    candidates = []\\n                meta = _pick_best_candidate(entry, candidates)\\n        if meta is None:\\n            meta = _fallback_paper(entry)\\n\\n        # Fill missing fields from YAML entry when resolver found only partial metadata.\\n        if not meta.title and entry.get(\\"title\\"):\\n            meta.title = str(entry.get(\\"title\\") or \\"\\")\\n        if meta.year is None and entry.get(\\"year\\") not in (None, \\"\\"):\\n            try:\\n                meta.year = int(entry.get(\\"year\\"))\\n            except Exception:\\n                pass\\n        if not meta.url and _looks_like_url(str(entry.get(\\"id\\") or \\"\\")):\\n            meta.url = str(entry.get(\\"id\\") or \\"\\")\\n\\n        if meta.id in seen_ids:\\n            continue\\n        seen_ids.add(meta.id)\\n        resolved.append(meta)\\n\\n    return resolved\\n\\n\\ndef _source_label(src: Dict[str, Any], idx: int) -> str:\\n    s = str(src.get(\\"source\\") or f\\"source_{idx}\\")\\n    loc = str(src.get(\\"locator\\") or src.get(\\"page\\") or \\"\\").strip()\\n    return f\\"{s} {loc}\\".strip()\\n\\n\\ndef _paper_lookup(doc: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:\\n    out: Dict[str, Dict[str, Any]] = {}\\n    for p in doc.get(\\"papers\\", []) or []:\\n        if not isinstance(p, dict):\\n            continue\\n        pid = str(p.get(\\"id\\") or \\"\\").strip()\\n        if pid:\\n            out[pid] = p\\n            doi = _extract_doi(pid)\\n            if doi:\\n                out[doi] = p\\n                out[f\\"doi:{doi}\\"] = p\\n        title = str(p.get(\\"title\\") or \\"\\").strip()\\n        if title:\\n            out[_norm_title(title)] = p\\n    return out\\n\\n\\ndef _year_from_source(source_ref: str, papers_by_key: Dict[str, Dict[str, Any]]) -> Optional[int]:\\n    ref = (source_ref or \\"\\").strip()\\n    candidates = [ref]\\n    doi = _extract_doi(ref)\\n    if doi:\\n        candidates.extend([doi, f\\"doi:{doi}\\"])\\n    arxiv = _extract_arxiv(ref)\\n    if arxiv:\\n        candidates.extend([arxiv, f\\"arxiv:{arxiv}\\"])\\n    oa = _extract_openalex(ref)\\n    if oa:\\n        candidates.extend([oa, f\\"openalex:{oa}\\"])\\n    if _norm_title(ref):\\n        candidates.append(_norm_title(ref))\\n\\n    for key in candidates:\\n        paper = papers_by_key.get(key)\\n        if not isinstance(paper, dict):\\n            continue\\n        try:\\n            y = int(paper.get(\\"year\\")) if paper.get(\\"year\\") not in (None, \\"\\") else None\\n        except Exception:\\n            y = None\\n        if y is not None:\\n            return y\\n    return None\\n\\n\\ndef _step_time_window(step: Dict[str, Any], papers_by_key: Dict[str, Dict[str, Any]]) -> Tuple[str, str]:\\n    years: List[int] = []\\n    for src in step.get(\\"sources\\", []) or []:\\n        if not isinstance(src, dict):\\n            continue\\n        y = _year_from_source(str(src.get(\\"source\\") or \\"\\"), papers_by_key)\\n        if y is not None:\\n            years.append(y)\\n    if not years:\\n        return (\\"unknown\\", \\"unknown\\")\\n    return (str(min(years)), str(max(years)))\\n\\n\\ndef _coerce_time_token(value: Any, *, default: str = \\"unknown\\") -> str:\\n    if value is None:\\n        return default\\n    s = str(value).strip()\\n    if not s:\\n        return default\\n    if DATE_TOKEN_RE.match(s):\\n        return s\\n    return default\\n\\n\\ndef _canonical_time_interval(rec: Dict[str, Any]) -> str:\\n    start = _coerce_time_token(rec.get(\\"start_date\\"), default=\\"unknown\\")\\n    end = _coerce_time_token(rec.get(\\"end_date\\"), default=\\"unknown\\")\\n    valid_from = _coerce_time_token(rec.get(\\"valid_from\\"), default=start)\\n    valid_to = _coerce_time_token(rec.get(\\"valid_to\\"), default=\\"+inf\\")\\n    legacy = str(rec.get(\\"time_interval\\") or \\"\\").strip()\\n    if legacy:\\n        return legacy\\n    return f\\"evidence:{start}..{end}|valid:{valid_from}..{valid_to}\\"\\n\\n\\ndef build_reference_graph(doc: Dict[str, Any]) -> Dict[str, Any]:\\n    papers_by_key = _paper_lookup(doc)\\n    steps = [s for s in (doc.get(\\"steps\\") or []) if isinstance(s, dict)]\\n    manual_nodes: List[Dict[str, Any]] = []\\n    manual_edges: List[Dict[str, Any]] = []\\n    manual_triplets: List[Dict[str, Any]] = []\\n\\n    step_ids: List[int] = []\\n    for idx, step in enumerate(steps, start=1):\\n        sid = int(step.get(\\"step_id\\") or idx)\\n        step_ids.append(sid)\\n        start_date, end_date = _step_time_window(step, papers_by_key)\\n        node_id = f\\"step:{sid}\\"\\n        conditions = step.get(\\"conditions\\") if isinstance(step.get(\\"conditions\\"), dict) else {}\\n        manual_nodes.append(\\n            {\\n                \\"id\\": node_id,\\n                \\"type\\": \\"trajectory_step\\",\\n                \\"label\\": str(step.get(\\"claim\\") or f\\"Шаг {sid}\\"),\\n                \\"step_id\\": sid,\\n                \\"claim\\": str(step.get(\\"claim\\") or \\"\\"),\\n                \\"inference\\": str(step.get(\\"inference\\") or \\"\\"),\\n                \\"next_question\\": str(step.get(\\"next_question\\") or \\"\\"),\\n                \\"conditions\\": conditions,\\n                \\"start_date\\": start_date,\\n                \\"end_date\\": end_date,\\n                \\"valid_from\\": start_date,\\n                \\"valid_to\\": \\"+inf\\" if start_date != \\"unknown\\" else \\"unknown\\",\\n                \\"time_source\\": \\"metadata\\",\\n            }\\n        )\\n        manual_triplets.append(\\n            {\\n                \\"assertion_id\\": f\\"manual-step-{sid}\\",\\n                \\"subject\\": node_id,\\n                \\"predicate\\": \\"states\\",\\n                \\"object\\": str(step.get(\\"claim\\") or \\"\\"),\\n                \\"start_date\\": start_date,\\n                \\"end_date\\": end_date,\\n                \\"valid_from\\": start_date,\\n                \\"valid_to\\": \\"+inf\\" if start_date != \\"unknown\\" else \\"unknown\\",\\n                \\"time_source\\": \\"metadata\\",\\n                \\"time_interval\\": f\\"evidence:{start_date}..{end_date}|valid:{start_date}..{'+inf' if start_date != 'unknown' else 'unknown'}\\",\\n                \\"evidence\\": {\\n                    \\"page\\": None,\\n                    \\"figure_or_table\\": None,\\n                    \\"snippet_or_summary\\": str(step.get(\\"inference\\") or step.get(\\"claim\\") or \\"\\"),\\n                },\\n                \\"verdict\\": \\"accepted\\",\\n                \\"rationale\\": \\"Reference step reconstructed from Task 1 trajectory YAML.\\",\\n            }\\n        )\\n\\n        for src_idx, src in enumerate(step.get(\\"sources\\", []) or [], start=1):\\n            if not isinstance(src, dict):\\n                continue\\n            source_id = f\\"step:{sid}:source:{src_idx:02d}\\"\\n            stype = str(src.get(\\"type\\") or \\"text\\")\\n            source_label = _source_label(src, src_idx)\\n            page_value = src.get(\\"page\\")\\n            try:\\n                page_value = int(page_value) if page_value not in (None, \\"\\") else None\\n            except Exception:\\n                page_value = None\\n            manual_nodes.append(\\n                {\\n                    \\"id\\": source_id,\\n                    \\"type\\": \\"evidence_source\\",\\n                    \\"label\\": source_label,\\n                    \\"source_type\\": stype,\\n                    \\"source_ref\\": str(src.get(\\"source\\") or \\"\\"),\\n                    \\"locator\\": str(src.get(\\"locator\\") or \\"\\"),\\n                    \\"page\\": page_value,\\n                    \\"snippet_or_summary\\": str(src.get(\\"snippet_or_summary\\") or \\"\\"),\\n                    \\"start_date\\": start_date,\\n                    \\"end_date\\": end_date,\\n                    \\"valid_from\\": start_date,\\n                    \\"valid_to\\": end_date,\\n                    \\"time_source\\": \\"metadata\\",\\n                }\\n            )\\n            manual_edges.append(\\n                {\\n                    \\"source\\": source_id,\\n                    \\"target\\": node_id,\\n                    \\"predicate\\": \\"supports\\",\\n                    \\"type\\": \\"evidence_support\\",\\n                    \\"start_date\\": start_date,\\n                    \\"end_date\\": end_date,\\n                }\\n            )\\n            manual_triplets.append(\\n                {\\n                    \\"assertion_id\\": f\\"manual-source-{sid}-{src_idx}\\",\\n                    \\"subject\\": str(src.get(\\"source\\") or source_id),\\n                    \\"predicate\\": \\"supports_step\\",\\n                    \\"object\\": node_id,\\n                    \\"start_date\\": start_date,\\n                    \\"end_date\\": end_date,\\n                    \\"valid_from\\": start_date,\\n                    \\"valid_to\\": end_date,\\n                    \\"time_source\\": \\"metadata\\",\\n                    \\"time_interval\\": f\\"evidence:{start_date}..{end_date}|valid:{start_date}..{end_date}\\",\\n                    \\"evidence\\": {\\n                        \\"page\\": page_value,\\n                        \\"figure_or_table\\": str(src.get(\\"locator\\") or \\"\\") or None,\\n                        \\"snippet_or_summary\\": str(src.get(\\"snippet_or_summary\\") or \\"\\"),\\n                    },\\n                    \\"verdict\\": \\"accepted\\",\\n                    \\"rationale\\": \\"Evidence link reconstructed from Task 1 trajectory YAML.\\",\\n                }\\n            )\\n\\n    raw_edges = doc.get(\\"edges\\") or []\\n    if not raw_edges and len(step_ids) > 1:\\n        raw_edges = [[step_ids[i], step_ids[i + 1]] for i in range(len(step_ids) - 1)]\\n\\n    step_map = {int(step.get(\\"step_id\\") or idx): step for idx, step in enumerate(steps, start=1)}\\n    for edge_idx, edge in enumerate(raw_edges, start=1):\\n        if not isinstance(edge, (list, tuple)) or len(edge) != 2:\\n            continue\\n        try:\\n            src_step = int(edge[0])\\n            dst_step = int(edge[1])\\n        except Exception:\\n            continue\\n        src_node = f\\"step:{src_step}\\"\\n        dst_node = f\\"step:{dst_step}\\"\\n        src_step_obj = step_map.get(src_step, {})\\n        start_date, end_date = _step_time_window(src_step_obj, papers_by_key) if src_step_obj else (\\"unknown\\", \\"unknown\\")\\n        manual_edges.append(\\n            {\\n                \\"source\\": src_node,\\n                \\"target\\": dst_node,\\n                \\"predicate\\": \\"leads_to\\",\\n                \\"type\\": \\"reasoning_transition\\",\\n                \\"start_date\\": start_date,\\n                \\"end_date\\": end_date,\\n            }\\n        )\\n        manual_triplets.append(\\n            {\\n                \\"assertion_id\\": f\\"manual-edge-{edge_idx}\\",\\n                \\"subject\\": src_node,\\n                \\"predicate\\": \\"leads_to\\",\\n                \\"object\\": dst_node,\\n                \\"start_date\\": start_date,\\n                \\"end_date\\": end_date,\\n                \\"valid_from\\": start_date,\\n                \\"valid_to\\": \\"+inf\\" if start_date != \\"unknown\\" else \\"unknown\\",\\n                \\"time_source\\": \\"metadata\\",\\n                \\"time_interval\\": f\\"evidence:{start_date}..{end_date}|valid:{start_date}..{'+inf' if start_date != 'unknown' else 'unknown'}\\",\\n                \\"evidence\\": {\\n                    \\"page\\": None,\\n                    \\"figure_or_table\\": None,\\n                    \\"snippet_or_summary\\": str(src_step_obj.get(\\"next_question\\") or src_step_obj.get(\\"inference\\") or \\"\\"),\\n                },\\n                \\"verdict\\": \\"accepted\\",\\n                \\"rationale\\": \\"Reasoning transition reconstructed from Task 1 trajectory YAML.\\",\\n            }\\n        )\\n\\n    return {\\n        \\"meta\\": {\\n            \\"kind\\": \\"reference_reasoning_graph\\",\\n            \\"artifact_version\\": int(doc.get(\\"artifact_version\\") or 1),\\n            \\"topic\\": str(doc.get(\\"topic\\") or \\"\\"),\\n            \\"domain\\": str(doc.get(\\"domain\\") or \\"\\"),\\n            \\"submission_id\\": str(doc.get(\\"submission_id\\") or \\"\\"),\\n            \\"generated_at\\": _utc_now(),\\n            \\"n_steps\\": len(steps),\\n            \\"n_papers\\": len(doc.get(\\"papers\\") or []),\\n        },\\n        \\"nodes\\": manual_nodes,\\n        \\"edges\\": manual_edges,\\n        \\"triplets\\": manual_triplets,\\n    }\\n\\n\\ndef _paperrecord_from_metadata(meta: PaperMetadata) -> PaperRecord:\\n    text = (meta.abstract or \\"\\").strip() or meta.title\\n    return PaperRecord(\\n        paper_id=meta.id,\\n        title=meta.title,\\n        year=_paper_year(meta),\\n        text=text,\\n        url=meta.url or \\"\\",\\n        source=str(meta.source.value if hasattr(meta.source, \\"value\\") else meta.source),\\n    )\\n\\n\\ndef _load_domain_from_trajectory(doc: Dict[str, Any]) -> DomainConfig:\\n    value = str(doc.get(\\"domain\\") or \\"science\\").strip()\\n    return load_domain_config(value or None)\\n\\n\\ndef _flatten_automatic_graph(kg: TemporalKnowledgeGraph) -> List[Dict[str, Any]]:\\n    rows: List[Dict[str, Any]] = []\\n    kg_json = kg.to_json_dict()\\n    for idx, edge in enumerate(kg_json.get(\\"edges\\", []) or [], start=1):\\n        years = []\\n        try:\\n            years = sorted(int(y) for y in (edge.get(\\"yearly_count\\") or {}).keys())\\n        except Exception:\\n            years = []\\n        start_date = str(years[0]) if years else \\"unknown\\"\\n        end_date = str(years[-1]) if years else \\"unknown\\"\\n        quotes = edge.get(\\"evidence_quotes\\") or []\\n        first_quote = quotes[0] if quotes else {}\\n        rows.append(\\n            {\\n                \\"assertion_id\\": f\\"auto-{idx:05d}\\",\\n                \\"subject\\": str(edge.get(\\"source\\") or \\"\\"),\\n                \\"predicate\\": str(edge.get(\\"predicate\\") or \\"\\"),\\n                \\"object\\": str(edge.get(\\"target\\") or \\"\\"),\\n                \\"start_date\\": start_date,\\n                \\"end_date\\": end_date,\\n                \\"valid_from\\": start_date,\\n                \\"valid_to\\": \\"+inf\\" if start_date != \\"unknown\\" else \\"unknown\\",\\n                \\"time_source\\": \\"metadata\\",\\n                \\"time_interval\\": f\\"evidence:{start_date}..{end_date}|valid:{start_date}..{'+inf' if start_date != 'unknown' else 'unknown'}\\",\\n                \\"score\\": edge.get(\\"score\\"),\\n                \\"mean_confidence\\": edge.get(\\"mean_confidence\\"),\\n                \\"papers\\": edge.get(\\"papers\\") or [],\\n                \\"evidence\\": {\\n                    \\"page\\": None,\\n                    \\"figure_or_table\\": None,\\n                    \\"snippet_or_summary\\": str(first_quote.get(\\"quote\\") or \\"\\"),\\n                    \\"paper_id\\": str(first_quote.get(\\"paper_id\\") or \\"\\"),\\n                },\\n            }\\n        )\\n    return rows\\n\\n\\ndef _prefill_graph_review(doc: Dict[str, Any], automatic_rows: Sequence[Dict[str, Any]]) -> Dict[str, Any]:\\n    reviewer_id = \\"\\"\\n    expert = doc.get(\\"expert\\") if isinstance(doc.get(\\"expert\\"), dict) else {}\\n    if expert:\\n        reviewer_id = str(expert.get(\\"latin_slug\\") or expert.get(\\"full_name\\") or \\"\\")\\n\\n    assertions = []\\n    for row in automatic_rows:\\n        item = dict(row)\\n        item[\\"verdict\\"] = \\"\\"\\n        item[\\"rationale\\"] = \\"\\"\\n        item[\\"time_source_note\\"] = \\"\\"\\n        assertions.append(item)\\n\\n    return {\\n        \\"artifact_version\\": 3,\\n        \\"domain\\": str(doc.get(\\"domain\\") or \\"\\"),\\n        \\"topic\\": str(doc.get(\\"topic\\") or \\"\\"),\\n        \\"trajectory_submission_id\\": str(doc.get(\\"submission_id\\") or \\"\\"),\\n        \\"reviewer_id\\": reviewer_id,\\n        \\"timestamp\\": _utc_now(),\\n        \\"assertions\\": assertions,\\n    }\\n\\n\\ndef _empty_temporal_corrections(doc: Dict[str, Any]) -> Dict[str, Any]:\\n    reviewer_id = \\"\\"\\n    expert = doc.get(\\"expert\\") if isinstance(doc.get(\\"expert\\"), dict) else {}\\n    if expert:\\n        reviewer_id = str(expert.get(\\"latin_slug\\") or expert.get(\\"full_name\\") or \\"\\")\\n\\n    return {\\n        \\"artifact_version\\": 2,\\n        \\"domain\\": str(doc.get(\\"domain\\") or \\"\\"),\\n        \\"paper_id\\": \\"\\",\\n        \\"reviewer_id\\": reviewer_id,\\n        \\"trajectory_submission_id\\": str(doc.get(\\"submission_id\\") or \\"\\"),\\n        \\"corrections\\": [],\\n    }\\n\\n\\ndef _compare_graphs(reference_graph: Dict[str, Any], automatic_rows: Sequence[Dict[str, Any]], resolved_papers: Sequence[PaperMetadata]) -> Dict[str, Any]:\\n    ref_step_labels = {str(n.get(\\"label\\") or \\"\\") for n in reference_graph.get(\\"nodes\\", []) if n.get(\\"type\\") == \\"trajectory_step\\"}\\n    auto_terms = {str(r.get(\\"subject\\") or \\"\\") for r in automatic_rows} | {str(r.get(\\"object\\") or \\"\\") for r in automatic_rows}\\n    paper_ids = {p.id for p in resolved_papers}\\n    auto_papers = set()\\n    for row in automatic_rows:\\n        auto_papers.update(str(x) for x in (row.get(\\"papers\\") or []))\\n\\n    return {\\n        \\"reference_steps\\": len([n for n in reference_graph.get(\\"nodes\\", []) if n.get(\\"type\\") == \\"trajectory_step\\"]),\\n        \\"reference_edges\\": len([e for e in reference_graph.get(\\"edges\\", []) if e.get(\\"predicate\\") == \\"leads_to\\"]),\\n        \\"automatic_edges\\": len(list(automatic_rows)),\\n        \\"automatic_unique_terms\\": len(auto_terms),\\n        \\"resolved_papers\\": len(list(resolved_papers)),\\n        \\"papers_reaching_automatic_graph\\": len(auto_papers & paper_ids),\\n        \\"trajectory_claim_examples\\": sorted(list(ref_step_labels))[:10],\\n        \\"automatic_term_examples\\": sorted(list(auto_terms))[:20],\\n    }\\n\\n\\ndef suggest_link_candidates(\\n    doc: Dict[str, Any],\\n    *,\\n    known_papers: Sequence[PaperMetadata],\\n    max_queries: int = 4,\\n    per_query: int = 8,\\n) -> List[Dict[str, Any]]:\\n    queries: List[str] = []\\n    topic = str(doc.get(\\"topic\\") or \\"\\").strip()\\n    if topic:\\n        queries.append(topic)\\n\\n    for step in doc.get(\\"steps\\", []) or []:\\n        if not isinstance(step, dict):\\n            continue\\n        nq = str(step.get(\\"next_question\\") or \\"\\").strip()\\n        if nq and nq not in queries:\\n            queries.append(nq)\\n        inf = str(step.get(\\"inference\\") or \\"\\").strip()\\n        if inf and len(queries) < max_queries and inf not in queries:\\n            queries.append(inf)\\n        if len(queries) >= max_queries:\\n            break\\n\\n    queries = queries[:max_queries]\\n    known_titles = {_norm_title(p.title) for p in known_papers}\\n    known_ids = {p.id for p in known_papers}\\n    suggestions: Dict[str, Dict[str, Any]] = {}\\n\\n    for q in queries:\\n        try:\\n            found = search_papers(q, limit=per_query, sources=DEFAULT_SEARCH_SOURCES)\\n        except Exception:\\n            found = []\\n        for cand in found:\\n            if cand.id in known_ids or _norm_title(cand.title) in known_titles:\\n                continue\\n            score = 2.0 * _score_title_match(q, cand.title)\\n            if cand.abstract:\\n                score += 0.25\\n            if cand.pdf_url:\\n                score += 0.25\\n            if cand.citation_count:\\n                score += min(float(cand.citation_count) / 500.0, 0.5)\\n            existing = suggestions.get(cand.id)\\n            payload = {\\n                \\"paper_id\\": cand.id,\\n                \\"title\\": cand.title,\\n                \\"year\\": _paper_year(cand),\\n                \\"url\\": cand.url,\\n                \\"pdf_url\\": cand.pdf_url,\\n                \\"source\\": str(cand.source.value if hasattr(cand.source, \\"value\\") else cand.source),\\n                \\"trigger_queries\\": [q],\\n                \\"score\\": round(score, 4),\\n            }\\n            if existing is None:\\n                suggestions[cand.id] = payload\\n            else:\\n                existing[\\"score\\"] = max(float(existing.get(\\"score\\") or 0.0), payload[\\"score\\"])\\n                tq = list(existing.get(\\"trigger_queries\\") or [])\\n                if q not in tq:\\n                    tq.append(q)\\n                existing[\\"trigger_queries\\"] = tq\\n\\n    ranked = sorted(suggestions.values(), key=lambda x: float(x.get(\\"score\\") or 0.0), reverse=True)\\n    return ranked[:20]\\n\\n\\ndef prepare_task2_validation_bundle(\\n    trajectory_yaml: Path,\\n    *,\\n    out_dir: Path,\\n    include_multimodal: bool = True,\\n    run_vlm: bool = True,\\n    edge_mode: str = \\"auto\\",\\n    suggest_links: bool = True,\\n    max_papers: int = 0,\\n    max_link_queries: int = 4,\\n) -> Path:\\n    doc = yaml.safe_load(trajectory_yaml.read_text(encoding=\\"utf-8\\")) or {}\\n    if not isinstance(doc, dict):\\n        raise ValueError(\\"Trajectory YAML must contain a top-level object.\\")\\n\\n    run_name = str(doc.get(\\"submission_id\\") or _slugify(str(doc.get(\\"topic\\") or trajectory_yaml.stem)))\\n    out = out_dir / run_name\\n    out.mkdir(parents=True, exist_ok=True)\\n\\n    shutil.copy2(trajectory_yaml, out / trajectory_yaml.name)\\n\\n    reference_graph = build_reference_graph(doc)\\n    (out / \\"reference_graph.json\\").write_text(json.dumps(reference_graph, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    (out / \\"reference_triplets.json\\").write_text(json.dumps(reference_graph.get(\\"triplets\\") or [], ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n\\n    domain_cfg = _load_domain_from_trajectory(doc)\\n\\n    resolved = resolve_papers_from_trajectory(doc)\\n    if max_papers and max_papers > 0:\\n        resolved = resolved[:max_papers]\\n\\n    (out / \\"papers_resolved.json\\").write_text(\\n        json.dumps([p.model_dump(mode=\\"json\\") for p in resolved], ensure_ascii=False, indent=2),\\n        encoding=\\"utf-8\\",\\n    )\\n\\n    acquire_dir = out / \\"automatic_graph\\"\\n    raw_dir = acquire_dir / \\"raw_pdfs\\"\\n    meta_dir = acquire_dir / \\"raw_meta\\"\\n    processed_dir = acquire_dir / \\"processed_papers\\"\\n    processed_dir.mkdir(parents=True, exist_ok=True)\\n\\n    acq: List[AcquireResult] = acquire_pdfs(resolved, raw_dir=raw_dir, meta_dir=meta_dir)\\n    (out / \\"acquire_results.json\\").write_text(\\n        json.dumps(\\n            [asdict(a) | {\\"pdf_path\\": str(a.pdf_path) if a.pdf_path else None, \\"meta_path\\": str(a.meta_path)} for a in acq],\\n            ensure_ascii=False,\\n            indent=2,\\n        ),\\n        encoding=\\"utf-8\\",\\n    )\\n\\n    ingested_ids: List[str] = []\\n    for meta, a in zip(resolved, acq):\\n        if not a.pdf_path:\\n            continue\\n        meta_json = meta.model_dump(mode=\\"json\\")\\n        try:\\n            if include_multimodal:\\n                ingest_pdf_multimodal_auto(a.pdf_path, meta_json, processed_dir, run_vlm=run_vlm)\\n            else:\\n                ingest_pdf_auto(a.pdf_path, meta_json, processed_dir)\\n            ingested_ids.append(meta.id)\\n        except Exception as e:\\n            console.print(f\\"[yellow]Ingest failed for {meta.id}: {e}[/yellow]\\")\\n\\n    processed_records = load_papers_from_processed(processed_dir)\\n    processed_by_id = {p.paper_id: p for p in processed_records}\\n    paper_records: List[PaperRecord] = []\\n    for meta in resolved:\\n        if meta.id in processed_by_id:\\n            paper_records.append(processed_by_id[meta.id])\\n        else:\\n            paper_records.append(_paperrecord_from_metadata(meta))\\n\\n    (out / \\"paper_records.json\\").write_text(\\n        json.dumps([asdict(r) for r in paper_records], ensure_ascii=False, indent=2),\\n        encoding=\\"utf-8\\",\\n    )\\n\\n    kg = build_temporal_kg(\\n        paper_records,\\n        domain=domain_cfg,\\n        query=str(doc.get(\\"topic\\") or \\"\\"),\\n        edge_mode=edge_mode,  # type: ignore[arg-type]\\n        expert_overrides_path=None,\\n    )\\n    kg.dump_json(out / \\"automatic_graph\\" / \\"temporal_kg.json\\")\\n\\n    automatic_rows = _flatten_automatic_graph(kg)\\n    (out / \\"automatic_triplets.json\\").write_text(json.dumps(automatic_rows, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n\\n    prefill_review = _prefill_graph_review(doc, automatic_rows)\\n    review_dir = out / \\"review_templates\\"\\n    review_dir.mkdir(parents=True, exist_ok=True)\\n    (review_dir / \\"graph_review_prefill.json\\").write_text(json.dumps(prefill_review, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    (review_dir / \\"temporal_corrections_template.json\\").write_text(json.dumps(_empty_temporal_corrections(doc), ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n\\n    comparison = _compare_graphs(reference_graph, automatic_rows, resolved)\\n    (out / \\"comparison_summary.json\\").write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n\\n    if suggest_links:\\n        suggestions = suggest_link_candidates(doc, known_papers=resolved, max_queries=max_link_queries)\\n        scout_dir = out / \\"scout\\"\\n        scout_dir.mkdir(parents=True, exist_ok=True)\\n        (scout_dir / \\"suggested_links.json\\").write_text(json.dumps(suggestions, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n\\n    manifest = {\\n        \\"generated_at\\": _utc_now(),\\n        \\"trajectory_file\\": trajectory_yaml.name,\\n        \\"submission_id\\": str(doc.get(\\"submission_id\\") or \\"\\"),\\n        \\"topic\\": str(doc.get(\\"topic\\") or \\"\\"),\\n        \\"domain\\": str(doc.get(\\"domain\\") or \\"\\"),\\n        \\"resolved_papers\\": len(resolved),\\n        \\"ingested_pdfs\\": len(ingested_ids),\\n        \\"automatic_edges\\": len(automatic_rows),\\n        \\"reference_steps\\": len([n for n in reference_graph.get(\\"nodes\\", []) if n.get(\\"type\\") == \\"trajectory_step\\"]),\\n        \\"artifacts\\": {\\n            \\"reference_graph\\": \\"reference_graph.json\\",\\n            \\"reference_triplets\\": \\"reference_triplets.json\\",\\n            \\"automatic_graph\\": \\"automatic_graph/temporal_kg.json\\",\\n            \\"automatic_triplets\\": \\"automatic_triplets.json\\",\\n            \\"papers_resolved\\": \\"papers_resolved.json\\",\\n            \\"comparison_summary\\": \\"comparison_summary.json\\",\\n            \\"review_prefill\\": \\"review_templates/graph_review_prefill.json\\",\\n        },\\n    }\\n    (out / \\"manifest.json\\").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    return out\\n", "scripts/ci/validate_expert_artifacts.py": "#!/usr/bin/env python3\\n\\"\\"\\"\\nValidate expert artifacts (YAML/JSON) for required fields:\\n- evidence present\\n- conditions present\\n- time_scope / time_interval present (where applicable)\\n\\nExit code:\\n- 0 if all good\\n- 1 if any problems found\\n\\nDesigned to be used in CI on pull requests.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport re\\nimport sys\\nfrom pathlib import Path\\nfrom typing import Any, Dict, List, Tuple\\n\\ntry:\\n    import yaml  # type: ignore\\nexcept Exception:\\n    print(\\"ERROR: Missing dependency PyYAML. Install with `pip install pyyaml`.\\", file=sys.stderr)\\n    raise\\n\\nREPO_ROOT = Path(__file__).resolve().parents[2]\\n\\nTRAJ_DIR = REPO_ROOT / \\"data\\" / \\"experts\\" / \\"trajectories\\"\\nGRAPH_REV_DIR = REPO_ROOT / \\"data\\" / \\"experts\\" / \\"graph_reviews\\"\\nHYP_REV_DIR = REPO_ROOT / \\"data\\" / \\"experts\\" / \\"hypothesis_reviews\\"\\nMM_REV_DIR = REPO_ROOT / \\"data\\" / \\"experts\\" / \\"mm_reviews\\"\\nTEMP_CORR_DIR = REPO_ROOT / \\"data\\" / \\"experts\\" / \\"temporal_corrections\\"\\n\\n\\ndef _load_yaml(path: Path) -> Dict[str, Any]:\\n    with path.open(\\"r\\", encoding=\\"utf-8\\") as f:\\n        return yaml.safe_load(f) or {}\\n\\n\\ndef _load_json(path: Path) -> Dict[str, Any]:\\n    with path.open(\\"r\\", encoding=\\"utf-8\\") as f:\\n        return json.load(f)\\n\\n\\ndef _is_empty(v: Any) -> bool:\\n    if v is None:\\n        return True\\n    if isinstance(v, str) and not v.strip():\\n        return True\\n    if isinstance(v, (list, dict)) and len(v) == 0:\\n        return True\\n    return False\\n\\n\\nTIME_TOKEN_RE = re.compile(r\\"^(?:\\\\d{4}|\\\\d{4}-\\\\d{2}|\\\\d{4}-\\\\d{2}-\\\\d{2}|unknown|\\\\+inf|-inf)$\\")\\n\\n\\ndef _is_time_token(v: Any) -> bool:\\n    if v is None:\\n        return False\\n    return bool(TIME_TOKEN_RE.fullmatch(str(v).strip()))\\n\\n\\ndef _has_structured_temporal_fields(a: Dict[str, Any]) -> bool:\\n    return any(not _is_empty(a.get(k)) for k in (\\"start_date\\", \\"end_date\\", \\"valid_from\\", \\"valid_to\\"))\\n\\n\\ndef _validate_temporal_fields(a: Dict[str, Any], prefix: str) -> List[str]:\\n    errs: List[str] = []\\n    if _has_structured_temporal_fields(a):\\n        # Evidence interval is the primary temporal axis in graph reviews.\\n        if _is_empty(a.get(\\"start_date\\")):\\n            errs.append(f\\"{prefix}: missing start_date (use 'unknown' or '-inf' if needed)\\")\\n        elif not _is_time_token(a.get(\\"start_date\\")):\\n            errs.append(f\\"{prefix}: invalid start_date '{a.get('start_date')}'\\")\\n\\n        if _is_empty(a.get(\\"end_date\\")):\\n            errs.append(f\\"{prefix}: missing end_date (use publication date, 'unknown' or '+inf' if needed)\\")\\n        elif not _is_time_token(a.get(\\"end_date\\")):\\n            errs.append(f\\"{prefix}: invalid end_date '{a.get('end_date')}'\\")\\n\\n        for k in (\\"valid_from\\", \\"valid_to\\"):\\n            if not _is_empty(a.get(k)) and not _is_time_token(a.get(k)):\\n                errs.append(f\\"{prefix}: invalid {k} '{a.get(k)}'\\")\\n    return errs\\n\\n\\ndef _resolve_domain_config(domain_value: str) -> Dict[str, Any]:\\n    \\"\\"\\"Resolve domain config from configs/domains/*.yaml.\\n\\n    Supports both:\\n    - legacy domain id (e.g. \\"science\\")\\n    - Wikidata QID stored in trajectory artifact (e.g. \\"Q336\\")\\n    \\"\\"\\"\\n    dv = (domain_value or \\"\\").strip()\\n    if not dv:\\n        return {}\\n\\n    qid = dv.upper() if re.fullmatch(r\\"Q\\\\d+\\", dv.upper()) else \\"\\"\\n\\n    domain_dir = REPO_ROOT / \\"configs\\" / \\"domains\\"\\n    for cfg_path in sorted(domain_dir.glob(\\"*.yaml\\")):\\n        try:\\n            cfg = yaml.safe_load(cfg_path.read_text(encoding=\\"utf-8\\")) or {}\\n        except Exception:\\n            continue\\n\\n        # QID match\\n        if qid:\\n            if str(cfg.get(\\"wikidata_qid\\") or \\"\\").strip().upper() == qid:\\n                return cfg\\n            continue\\n\\n        # legacy match\\n        domain_id = str(cfg.get(\\"domain_id\\") or \\"\\").strip().lower()\\n        stem = cfg_path.stem.strip().lower()\\n        dv_norm = dv.strip().lower()\\n        if dv_norm in {domain_id, stem}:\\n            return cfg\\n\\n    return {}\\n\\n\\ndef _required_condition_keys(domain: str) -> List[str]:\\n    \\"\\"\\"Domain-specific condition requirements for trajectories.\\n\\n    For artifact v2 the trajectory stores `domain` as a Wikidata QID (e.g. Q336).\\n    We map it back to configs/domains/*.yaml via `wikidata_qid`.\\n    \\"\\"\\"\\n    cfg = _resolve_domain_config(domain)\\n    av = (cfg.get(\\"artifact_validation\\") or {}) if isinstance(cfg, dict) else {}\\n    req = av.get(\\"trajectory_required_conditions\\") or []\\n    if isinstance(req, list):\\n        return [str(x) for x in req]\\n    return []\\n\\n\\n\\ndef validate_trajectory(path: Path) -> List[str]:\\n    errs: List[str] = []\\n    doc = _load_yaml(path)\\n\\n    artifact_version = int(doc.get(\\"artifact_version\\") or 1)\\n    domain = str(doc.get(\\"domain\\") or \\"\\").strip()\\n\\n    if _is_empty(domain):\\n        errs.append(\\"domain required\\")\\n\\n    steps = doc.get(\\"steps\\", [])\\n    if not isinstance(steps, list) or len(steps) == 0:\\n        errs.append(\\"steps[] must be a non-empty list\\")\\n        return errs\\n\\n    n_steps = len(steps)\\n    required_keys = _required_condition_keys(domain) if domain else []\\n\\n    for i, step in enumerate(steps, start=1):\\n        if not isinstance(step, dict):\\n            errs.append(f\\"step {i}: must be object\\")\\n            continue\\n\\n        if _is_empty(step.get(\\"claim\\")):\\n            errs.append(f\\"step {i}: missing claim\\")\\n\\n        # conditions\\n        conditions = step.get(\\"conditions\\", {})\\n        if not isinstance(conditions, dict) or _is_empty(conditions):\\n            errs.append(f\\"step {i}: missing conditions\\")\\n        else:\\n            for k in required_keys:\\n                if _is_empty(conditions.get(k)):\\n                    errs.append(f\\"step {i}: conditions.{k} required for domain={domain}\\")\\n\\n        # inference / next_question (recommended to be non-empty)\\n        if _is_empty(step.get(\\"inference\\")):\\n            errs.append(f\\"step {i}: missing inference\\")\\n        if _is_empty(step.get(\\"next_question\\")):\\n            errs.append(f\\"step {i}: missing next_question\\")\\n\\n        if artifact_version >= 2:\\n            # v2: sources[]\\n            sources = step.get(\\"sources\\", [])\\n            if not isinstance(sources, list) or len(sources) == 0:\\n                errs.append(f\\"step {i}: missing sources[] (must be non-empty list)\\")\\n            else:\\n                for j, src in enumerate(sources, start=1):\\n                    if not isinstance(src, dict):\\n                        errs.append(f\\"step {i} source {j}: must be object\\")\\n                        continue\\n                    stype = str(src.get(\\"type\\") or \\"\\").strip().lower()\\n                    if stype == \\"figure\\":\\n                        stype = \\"image\\"\\n                    if stype not in {\\"text\\", \\"image\\", \\"table\\"}:\\n                        errs.append(f\\"step {i} source {j}: invalid type='{src.get('type')}' (use text/image/table)\\")\\n                    if _is_empty(src.get(\\"source\\")):\\n                        errs.append(f\\"step {i} source {j}: source required\\")\\n                    if _is_empty(src.get(\\"snippet_or_summary\\")):\\n                        errs.append(f\\"step {i} source {j}: snippet_or_summary required\\")\\n        else:\\n            # v1 legacy: evidence{}\\n            evidence = step.get(\\"evidence\\", {})\\n            if not isinstance(evidence, dict) or _is_empty(evidence):\\n                errs.append(f\\"step {i}: missing evidence\\")\\n            else:\\n                if _is_empty(evidence.get(\\"source\\")):\\n                    errs.append(f\\"step {i}: evidence.source required\\")\\n                if _is_empty(evidence.get(\\"page\\")):\\n                    errs.append(f\\"step {i}: evidence.page required\\")\\n                if _is_empty(evidence.get(\\"snippet_or_summary\\")):\\n                    errs.append(f\\"step {i}: evidence.snippet_or_summary required\\")\\n                etype = str(evidence.get(\\"type\\", \\"\\")).strip().lower()\\n                if etype in {\\"figure\\", \\"table\\"} and _is_empty(evidence.get(\\"figure_or_table\\")):\\n                    errs.append(f\\"step {i}: evidence.figure_or_table required for {etype}\\")\\n\\n    # edges (optional): directed graph as ordered pairs [from, to]\\n    edges = doc.get(\\"edges\\", [])\\n    if edges is None:\\n        edges = []\\n    if not _is_empty(edges):\\n        if not isinstance(edges, list):\\n            errs.append(\\"edges must be a list of [from,to]\\")\\n        else:\\n            seen: set[tuple[int, int]] = set()\\n            for k, e in enumerate(edges, start=1):\\n                if not isinstance(e, (list, tuple)) or len(e) != 2:\\n                    errs.append(f\\"edge {k}: must be [from, to]\\")\\n                    continue\\n                a, b = e[0], e[1]\\n                try:\\n                    a_i = int(a)\\n                    b_i = int(b)\\n                except Exception:\\n                    errs.append(f\\"edge {k}: from/to must be integers\\")\\n                    continue\\n                if a_i == b_i:\\n                    errs.append(f\\"edge {k}: self-loop {a_i}->{b_i} is not allowed\\")\\n                    continue\\n                if not (1 <= a_i <= n_steps) or not (1 <= b_i <= n_steps):\\n                    errs.append(f\\"edge {k}: out of range (steps are 1..{n_steps})\\")\\n                    continue\\n                t = (a_i, b_i)\\n                if t in seen:\\n                    errs.append(f\\"edge {k}: duplicate edge {a_i}->{b_i}\\")\\n                else:\\n                    seen.add(t)\\n\\n    return errs\\n\\n\\n\\ndef validate_graph_review(path: Path) -> List[str]:\\n    errs: List[str] = []\\n    doc = _load_json(path)\\n\\n    assertions = doc.get(\\"assertions\\", [])\\n    if not isinstance(assertions, list) or len(assertions) == 0:\\n        return [\\"assertions[] must be a non-empty list\\"]\\n\\n    for i, a in enumerate(assertions, start=1):\\n        if not isinstance(a, dict):\\n            errs.append(f\\"assertion {i}: must be object\\")\\n            continue\\n\\n        for k in (\\"subject\\", \\"predicate\\", \\"object\\"):\\n            if _is_empty(a.get(k)):\\n                errs.append(f\\"assertion {i}: missing {k}\\")\\n\\n        has_legacy_interval = not _is_empty(a.get(\\"time_interval\\"))\\n        if has_legacy_interval:\\n            # Legacy free-form field remains supported for backward compatibility.\\n            pass\\n        elif not _has_structured_temporal_fields(a):\\n            errs.append(\\n                f\\"assertion {i}: missing temporal information (provide time_interval or start_date/end_date; use 'unknown' if not stated)\\"\\n            )\\n\\n        errs.extend(_validate_temporal_fields(a, f\\"assertion {i}\\"))\\n\\n        ev = a.get(\\"evidence\\", {})\\n        if not isinstance(ev, dict) or _is_empty(ev):\\n            errs.append(f\\"assertion {i}: missing evidence\\")\\n        else:\\n            if _is_empty(ev.get(\\"snippet_or_summary\\")):\\n                errs.append(f\\"assertion {i}: evidence.snippet_or_summary required\\")\\n            has_locator = not _is_empty(ev.get(\\"page\\")) or not _is_empty(ev.get(\\"figure_or_table\\")) or not _is_empty(ev.get(\\"paper_id\\")) or not _is_empty(ev.get(\\"source\\"))\\n            if not has_locator:\\n                errs.append(f\\"assertion {i}: evidence should include page, figure_or_table, paper_id, or source\\")\\n\\n        verdict = str(a.get(\\"verdict\\", \\"\\")).strip()\\n        if verdict not in {\\"accepted\\", \\"rejected\\", \\"needs_time_fix\\", \\"needs_evidence_fix\\", \\"added\\"}:\\n            errs.append(f\\"assertion {i}: invalid verdict '{verdict}'\\")\\n\\n        if _is_empty(a.get(\\"rationale\\")):\\n            errs.append(f\\"assertion {i}: missing rationale\\")\\n\\n    return errs\\n\\n\\ndef validate_hypothesis_review(path: Path) -> List[str]:\\n    errs: List[str] = []\\n    doc = _load_json(path)\\n\\n    if _is_empty(doc.get(\\"time_scope\\")):\\n        errs.append(\\"missing time_scope (conditions/time applicability)\\")\\n\\n    scores = doc.get(\\"scores\\")\\n    if not isinstance(scores, dict):\\n        errs.append(\\"missing scores object\\")\\n    else:\\n        for k in (\\"novelty\\", \\"soundness\\", \\"testability\\"):\\n            if k not in scores:\\n                errs.append(f\\"missing scores.{k}\\")\\n\\n    if \\"accept\\" not in doc:\\n        errs.append(\\"missing accept (true/false)\\")\\n    if _is_empty(doc.get(\\"major_issues\\")):\\n        errs.append(\\"missing major_issues[] (at least 1, or ['none'] if truly clean)\\")\\n    if _is_empty(doc.get(\\"required_experiments\\")):\\n        errs.append(\\"missing required_experiments (how to test)\\")\\n\\n    return errs\\n\\n\\ndef validate_mm_review(path: Path) -> List[str]:\\n    errs: List[str] = []\\n    doc = _load_json(path)\\n\\n    items = doc.get(\\"items\\", [])\\n    if not isinstance(items, list) or len(items) == 0:\\n        return [\\"items[] must be a non-empty list\\"]\\n\\n    for i, it in enumerate(items, start=1):\\n        if not isinstance(it, dict):\\n            errs.append(f\\"item {i}: must be object\\")\\n            continue\\n\\n        if it.get(\\"page\\") is None:\\n            errs.append(f\\"item {i}: missing page\\")\\n        if _is_empty(it.get(\\"verdict\\")):\\n            errs.append(f\\"item {i}: missing verdict\\")\\n        if _is_empty(it.get(\\"rationale\\")):\\n            errs.append(f\\"item {i}: missing rationale\\")\\n\\n        # At least one modality field should be present\\n        if _is_empty(it.get(\\"vlm_caption\\")) and _is_empty(it.get(\\"tables_md\\")) and _is_empty(it.get(\\"equations_md\\")):\\n            errs.append(f\\"item {i}: provide at least one of vlm_caption/tables_md/equations_md\\")\\n\\n        v = str(it.get(\\"verdict\\", \\"\\")).strip()\\n        if v not in {\\"accepted\\", \\"needs_fix\\", \\"rejected\\"}:\\n            errs.append(f\\"item {i}: invalid verdict '{v}'\\")\\n\\n    return errs\\n\\n\\ndef validate_temporal_correction(path: Path) -> List[str]:\\n    errs: List[str] = []\\n    doc = _load_json(path)\\n\\n    corrections = doc.get(\\"corrections\\", [])\\n    if not isinstance(corrections, list) or len(corrections) == 0:\\n        return [\\"corrections[] must be a non-empty list\\"]\\n\\n    for i, c in enumerate(corrections, start=1):\\n        if not isinstance(c, dict):\\n            errs.append(f\\"correction {i}: must be object\\")\\n            continue\\n        if _is_empty(c.get(\\"assertion_id\\")):\\n            errs.append(f\\"correction {i}: missing assertion_id\\")\\n        if _is_empty(c.get(\\"rationale\\")):\\n            errs.append(f\\"correction {i}: missing rationale\\")\\n\\n        ct = c.get(\\"corrected_time\\")\\n        if not isinstance(ct, dict) or _is_empty(ct):\\n            errs.append(f\\"correction {i}: missing corrected_time object\\")\\n        else:\\n            if _is_empty(ct.get(\\"granularity\\")):\\n                errs.append(f\\"correction {i}: corrected_time.granularity required\\")\\n            if _is_empty(ct.get(\\"start\\")) and _is_empty(ct.get(\\"end\\")):\\n                errs.append(f\\"correction {i}: corrected_time must include start and/or end\\")\\n            for key in (\\"start\\", \\"end\\"):\\n                if not _is_empty(ct.get(key)) and not _is_time_token(ct.get(key)):\\n                    errs.append(f\\"correction {i}: invalid corrected_time.{key} '{ct.get(key)}'\\")\\n\\n        ot = c.get(\\"original_time\\")\\n        if isinstance(ot, dict) and not _is_empty(ot):\\n            for key in (\\"start\\", \\"end\\"):\\n                if not _is_empty(ot.get(key)) and not _is_time_token(ot.get(key)):\\n                    errs.append(f\\"correction {i}: invalid original_time.{key} '{ot.get(key)}'\\")\\n\\n    return errs\\n\\n\\ndef _collect_files() -> List[Tuple[str, Path]]:\\n    files: List[Tuple[str, Path]] = []\\n    for p in sorted(TRAJ_DIR.glob(\\"**/*.y*ml\\")):\\n        files.append((\\"trajectory\\", p))\\n    for p in sorted(GRAPH_REV_DIR.glob(\\"**/*.json\\")):\\n        files.append((\\"graph_review\\", p))\\n    for p in sorted(HYP_REV_DIR.glob(\\"**/*.json\\")):\\n        files.append((\\"hypothesis_review\\", p))\\n    for p in sorted(MM_REV_DIR.glob(\\"**/*.json\\")):\\n        files.append((\\"mm_review\\", p))\\n    for p in sorted(TEMP_CORR_DIR.glob(\\"**/*.json\\")):\\n        files.append((\\"temporal_correction\\", p))\\n    return files\\n\\n\\ndef main() -> int:\\n    problems: List[str] = []\\n\\n    for kind, path in _collect_files():\\n        try:\\n            if kind == \\"trajectory\\":\\n                errs = validate_trajectory(path)\\n            elif kind == \\"graph_review\\":\\n                errs = validate_graph_review(path)\\n            elif kind == \\"hypothesis_review\\":\\n                errs = validate_hypothesis_review(path)\\n            elif kind == \\"mm_review\\":\\n                errs = validate_mm_review(path)\\n            else:\\n                errs = validate_temporal_correction(path)\\n        except Exception as e:\\n            problems.append(f\\"{kind} {path}: failed to parse ({e})\\")\\n            continue\\n\\n        for e in errs:\\n            problems.append(f\\"{kind} {path}: {e}\\")\\n\\n    if problems:\\n        print(\\"❌ Expert artifacts validation failed:\\\\n\\")\\n        for p in problems:\\n            print(\\" - \\" + p)\\n        print(f\\"\\\\nTotal problems: {len(problems)}\\")\\n        return 1\\n\\n    print(\\"✅ Expert artifacts validation passed.\\")\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "data/experts/graph_reviews/_template.json": "{\\n  \\"artifact_version\\": 3,\\n  \\"domain\\": \\"example_domain_or_wikidata_qid\\",\\n  \\"topic\\": \\"example topic\\",\\n  \\"paper_id\\": \\"doi:...\\",\\n  \\"reviewer_id\\": \\"expert_001\\",\\n  \\"timestamp\\": \\"2026-02-01T12:00:00Z\\",\\n  \\"assertions\\": [\\n    {\\n      \\"assertion_id\\": \\"A-000123\\",\\n      \\"subject\\": \\"Method X\\",\\n      \\"predicate\\": \\"improves\\",\\n      \\"object\\": \\"Outcome Y\\",\\n      \\"start_date\\": \\"2021\\",\\n      \\"end_date\\": \\"2024\\",\\n      \\"valid_from\\": \\"2021\\",\\n      \\"valid_to\\": \\"+inf\\",\\n      \\"time_source\\": \\"explicit_text\\",\\n      \\"time_source_note\\": \\"В статье явно указан период исследования 2021-2024; claim считается актуальным до появления опровержения.\\",\\n      \\"time_interval\\": \\"evidence:2021..2024|valid:2021..+inf\\",\\n      \\"evidence\\": {\\n        \\"paper_id\\": \\"doi:10.xxxx/xxxxx\\",\\n        \\"page\\": 6,\\n        \\"figure_or_table\\": \\"Figure 5\\",\\n        \\"source\\": \\"doi:10.xxxx/xxxxx\\",\\n        \\"snippet_or_summary\\": \\"Коротко: что именно утверждается источником\\"\\n      },\\n      \\"verdict\\": \\"accepted\\",\\n      \\"rationale\\": \\"Почему так (границы применимости, альтернативы, риски)\\"\\n    }\\n  ]\\n}\\n", "data/experts/temporal_corrections/_template.json": "{\\n  \\"artifact_version\\": 2,\\n  \\"domain\\": \\"example_domain\\",\\n  \\"paper_id\\": \\"arxiv:2401.01234\\",\\n  \\"reviewer_id\\": \\"anon\\",\\n  \\"trajectory_submission_id\\": \\"example_submission\\",\\n  \\"corrections\\": [\\n    {\\n      \\"assertion_id\\": \\"arxiv:2401.01234|Li plating|is triggered by|high overpotential|supports|year:2018:2018\\",\\n      \\"original_time\\": {\\"start\\": \\"2018\\", \\"end\\": \\"2018\\", \\"granularity\\": \\"year\\"},\\n      \\"corrected_time\\": {\\"start\\": \\"2016\\", \\"end\\": \\"+inf\\", \\"granularity\\": \\"year\\"},\\n      \\"evidence_quote\\": \\"...\\",\\n      \\"rationale\\": \\"Откуда следует правильный период; допускаются -inf/+inf для открытых интервалов.\\"\\n    }\\n  ]\\n}\\n", "docs/course/experts/docs_experts/task2_graph_verification_temporal_mm.md": "# Task 2 — Валидация темпорального графа знаний\\n\\n## Что делает эксперт\\nЭксперт загружает YAML из **Task 1** и получает **два графа**:\\n\\n1. **Reference graph** — точное восстановление вручную размеченной reasoning-схемы из YAML.\\n2. **Automatic temporal KG** — автоматически построенный темпоральный граф по статьям из YAML через CLI `top-papers-graph prepare-task2-validation`.\\n\\nДальше эксперт сравнивает оба графа, просматривает триплеты, оценивает временные поля и при необходимости вносит правки в `graph_review_prefill.json` и `temporal_corrections_template.json`.\\n\\n## Temporal-схема\\nВ Task 2 используется **двойная темпоральность**:\\n\\n- `start_date` — начало интервала, к которому относится утверждение / наблюдение.\\n- `end_date` — конец интервала, обычно дата публикации или последний год, подтверждённый источником.\\n- `valid_from` — момент, с которого утверждение считаем валидным в графе.\\n- `valid_to` — момент, до которого утверждение считается валидным; если опровержения нет, допустимо `+inf`.\\n- `time_source` — откуда взято время: `explicit_text`, `figure_or_table`, `metadata`, `expert_inference`, `unknown`.\\n- `time_source_note` — свободный комментарий эксперта.\\n\\nПоддерживаются специальные значения: `unknown`, `-inf`, `+inf`.\\nПоле `time_interval` сохраняется только ради обратной совместимости с существующим пайплайном и формируется автоматически.\\n\\n## Как интерпретировать время\\nРекомендуемый приоритет:\\n\\n1. **explicit_text** — время явно указано в тексте статьи.\\n2. **figure_or_table** — время извлечено из подписи, оси, таблицы.\\n3. **metadata** — время взято из года публикации / внешних метаданных.\\n4. **expert_inference** — время реконструировано экспертом, если другого источника нет.\\n5. **unknown** — если время определить нельзя.\\n\\n## Вердикты эксперта\\n- `accepted`\\n- `rejected`\\n- `needs_time_fix`\\n- `needs_evidence_fix`\\n- `added`\\n\\n## Дополнительный scouting\\nCLI также создаёт `scout/suggested_links.json` — список дополнительных статей-кандидатов, найденных по `topic` и `next_question` из trajectory. Это отдельный слой поддержки эксперта: ссылки предлагает система, а решение об их использовании принимает эксперт.\\n\\n## Где смотреть результаты\\nПосле запуска `prepare-task2-validation` в bundle появляются:\\n\\n- `reference_graph.json`\\n- `reference_triplets.json`\\n- `automatic_graph/temporal_kg.json`\\n- `automatic_triplets.json`\\n- `comparison_summary.json`\\n- `review_templates/graph_review_prefill.json`\\n- `review_templates/temporal_corrections_template.json`\\n- `scout/suggested_links.json`\\n", "docs/course/experts/templates/graph_review_template.json": "{\\n  \\"artifact_version\\": 3,\\n  \\"domain\\": \\"example_domain_or_wikidata_qid\\",\\n  \\"topic\\": \\"example topic\\",\\n  \\"paper_id\\": \\"doi:...\\",\\n  \\"reviewer_id\\": \\"expert_001\\",\\n  \\"timestamp\\": \\"2026-02-01T12:00:00Z\\",\\n  \\"assertions\\": [\\n    {\\n      \\"assertion_id\\": \\"A-000123\\",\\n      \\"subject\\": \\"Method X\\",\\n      \\"predicate\\": \\"improves\\",\\n      \\"object\\": \\"Outcome Y\\",\\n      \\"start_date\\": \\"2021\\",\\n      \\"end_date\\": \\"2024\\",\\n      \\"valid_from\\": \\"2021\\",\\n      \\"valid_to\\": \\"+inf\\",\\n      \\"time_source\\": \\"explicit_text\\",\\n      \\"time_source_note\\": \\"Период 2021-2024 прямо указан в статье; claim считаем валидным до опровержения.\\",\\n      \\"time_interval\\": \\"evidence:2021..2024|valid:2021..+inf\\",\\n      \\"evidence\\": {\\n        \\"paper_id\\": \\"doi:10.xxxx/xxxxx\\",\\n        \\"page\\": 6,\\n        \\"figure_or_table\\": \\"Figure 5\\",\\n        \\"source\\": \\"doi:10.xxxx/xxxxx\\",\\n        \\"snippet_or_summary\\": \\"Коротко: что именно утверждается источником\\"\\n      },\\n      \\"verdict\\": \\"accepted\\",\\n      \\"rationale\\": \\"Почему так (границы применимости, альтернативы, риски)\\"\\n    }\\n  ]\\n}\\n", "README_TASK2_VALIDATION.md": "# Task 2 validation bundle\\n\\nThis repository snapshot includes:\\n\\n- patched CLI command `top-papers-graph prepare-task2-validation`\\n- temporal review schema v3 (`start_date`, `end_date`, `valid_from`, `valid_to`)\\n- backward compatibility with legacy `time_interval`\\n- Colab notebook `notebooks/task2_temporal_graph_validation_colab.ipynb`\\n- sample Task 1 YAML inputs under `examples/task2_validation_inputs/`\\n\\nRecommended entrypoint for experts: open the notebook and run it top-to-bottom.\\n"}""")


def run_cmd(cmd, cwd=None, env=None, check=True):
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed with code {{proc.returncode}}')
    return proc


def write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')


def apply_task2_patches(repo_dir: Path):
    for rel_path, content in PATCHES.items():
        write_text(repo_dir / rel_path, content)
    print(f'Applied {{len(PATCHES)}} Task 2 patch files.')


def clone_and_setup_repo(repo_url: str, branch: str, repo_dir: Path):
    if repo_dir.exists():
        print(f'Removing existing directory: {{repo_dir}}')
        shutil.rmtree(repo_dir)
    run_cmd(['git', 'clone', '--depth', '1', '--branch', branch, repo_url, str(repo_dir)])
    apply_task2_patches(repo_dir)
    # editable install + repo extras
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev,g4f,mm,temporal]'], cwd=str(repo_dir))
    SESSION['repo_dir'] = str(repo_dir)
    return repo_dir


def _save_upload_value(upload_value, target_dir: Path) -> Path:
    target_dir.mkdir(parents=True, exist_ok=True)
    if isinstance(upload_value, dict):
        items = list(upload_value.items())
    else:
        items = list(upload_value)
    if not items:
        raise ValueError('Сначала загрузите YAML файл.')
    first = items[0][1] if isinstance(items[0], tuple) else items[0]
    name = first.get('name') or first.get('metadata', {{}}).get('name') or 'task1.yaml'
    content = first.get('content')
    if content is None:
        raise ValueError('Не удалось прочитать content из upload widget.')
    out_path = target_dir / name
    if isinstance(content, memoryview):
        content = content.tobytes()
    out_path.write_bytes(content)
    return out_path


def load_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))


def load_yaml_doc(path: Path):
    return yaml.safe_load(path.read_text(encoding='utf-8'))


def create_probe_image(path: Path):
    from PIL import Image, ImageDraw
    img = Image.new('RGB', (256, 160), color='white')
    d = ImageDraw.Draw(img)
    d.rectangle((10, 10, 246, 150), outline='black', width=2)
    d.text((20, 30), 'Task 2 graph validation', fill='black')
    d.text((20, 70), 'Does the model see this?', fill='black')
    img.save(path)
    return path


def try_g4f_multimodal(model_name, image_path: Path, provider=None):
    from g4f.client import Client
    data_url = 'data:image/png;base64,' + base64.b64encode(image_path.read_bytes()).decode('ascii')
    client = Client()
    errors = []
    text = None
    try:
        resp = client.chat.completions.create(
            model=model_name,
            provider=provider,
            messages=[{{
                'role': 'user',
                'content': [
                    {{'type': 'text', 'text': 'Describe the uploaded test image in one short sentence.'}},
                    {{'type': 'image_url', 'image_url': {{'url': data_url}}}},
                ],
            }}],
        )
        if getattr(resp, 'choices', None):
            text = resp.choices[0].message.content
    except Exception as e:
        errors.append(f'openai_style={{type(e).__name__}}: {{e}}')

    if not text:
        try:
            resp = client.chat.completions.create(
                model=model_name,
                provider=provider,
                messages=[{{'role': 'user', 'content': 'Describe the uploaded test image in one short sentence.'}}],
                images=[[data_url, image_path.name]],
            )
            if getattr(resp, 'choices', None):
                text = resp.choices[0].message.content
        except Exception as e:
            errors.append(f'images_arg={{type(e).__name__}}: {{e}}')

    return text, errors


def probe_g4f_models():
    candidate_models = [
        'gpt-4o-mini',
        'gpt-4o',
        'gemini-2.0-flash',
        'gemini-2.5-flash',
        'gemini-1.5-flash',
        'claude-3.5-sonnet',
        'qwen-2.5-vl-72b-instruct',
        'qwen-2.5-vl-7b-instruct',
        'qwen-2.5-vl-3b-instruct',
        'llama-3.2-11b-vision-instruct',
    ]
    probe_dir = Path(SESSION['repo_dir']) / '.colab_probe'
    probe_dir.mkdir(parents=True, exist_ok=True)
    img_path = create_probe_image(probe_dir / 'probe.png')
    report = []
    best = None
    for model in candidate_models:
        text, errors = try_g4f_multimodal(model, img_path)
        ok = bool(text and str(text).strip())
        item = {{'model': model, 'ok': ok, 'text': text or '', 'errors': errors}}
        report.append(item)
        if ok and best is None:
            best = item
    SESSION['g4f_probe'] = report
    return best, report


def ensure_qwen_fallback_requirements():
    # По рекомендациям HF для Qwen2.5-VL надёжнее брать свежий transformers.
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/huggingface/transformers', 'accelerate', 'sentencepiece'])


def persist_env_settings(repo_dir: Path, settings_map: dict):
    env_path = repo_dir / '.env.task2_colab'
    lines = [f'{{k}}={{v}}' for k, v in settings_map.items()]
    env_path.write_text('
'.join(lines) + '
', encoding='utf-8')
    return env_path


def choose_vlm_backend(repo_dir: Path):
    best, report = probe_g4f_models()
    if best is not None:
        os.environ['VLM_BACKEND'] = 'g4f'
        os.environ['VLM_MODEL_ID'] = best['model']
        SESSION['vlm_mode'] = 'g4f'
        SESSION['vlm_model'] = best['model']
        env_path = persist_env_settings(repo_dir, {{
            'VLM_BACKEND': 'g4f',
            'VLM_MODEL_ID': best['model'],
        }})
        return {{'mode': 'g4f', 'model': best['model'], 'report': report, 'env_path': str(env_path)}}

    ensure_qwen_fallback_requirements()
    os.environ['VLM_BACKEND'] = 'qwen2_vl'
    os.environ['VLM_MODEL_ID'] = 'Qwen/Qwen2.5-VL-3B-Instruct'
    SESSION['vlm_mode'] = 'qwen2_vl'
    SESSION['vlm_model'] = 'Qwen/Qwen2.5-VL-3B-Instruct'
    env_path = persist_env_settings(repo_dir, {{
        'VLM_BACKEND': 'qwen2_vl',
        'VLM_MODEL_ID': 'Qwen/Qwen2.5-VL-3B-Instruct',
    }})
    return {{'mode': 'qwen2_vl', 'model': 'Qwen/Qwen2.5-VL-3B-Instruct', 'report': report, 'env_path': str(env_path)}}


def build_task2_bundle(repo_dir: Path, trajectory_path: Path, out_dir: Path, multimodal=True, vlm=True, suggest_links=True, edge_mode='auto', max_papers=0):
    env = os.environ.copy()
    cmd = [
        sys.executable, '-m', 'scireason.cli', 'prepare-task2-validation',
        '--trajectory', str(trajectory_path),
        '--out-dir', str(out_dir),
        '--edge-mode', edge_mode,
        '--max-papers', str(max_papers),
    ]
    cmd += ['--multimodal', 'true' if multimodal else 'false']
    cmd += ['--vlm', 'true' if vlm else 'false']
    cmd += ['--suggest-links', 'true' if suggest_links else 'false']
    run_cmd(cmd, cwd=str(repo_dir), env=env)

    doc = load_yaml_doc(trajectory_path)
    run_name = doc.get('submission_id') or trajectory_path.stem
    bundle_dir = out_dir / run_name
    SESSION['bundle_dir'] = str(bundle_dir)
    return bundle_dir


def triplets_to_df(rows):
    items = []
    for r in rows:
        ev = r.get('evidence') if isinstance(r.get('evidence'), dict) else {{}}
        items.append({{
            'assertion_id': r.get('assertion_id'),
            'subject': r.get('subject'),
            'predicate': r.get('predicate'),
            'object': r.get('object'),
            'start_date': r.get('start_date'),
            'end_date': r.get('end_date'),
            'valid_from': r.get('valid_from'),
            'valid_to': r.get('valid_to'),
            'time_source': r.get('time_source'),
            'snippet': (ev.get('snippet_or_summary') or '')[:220],
            'paper_id': ev.get('paper_id') or '',
        }})
    return pd.DataFrame(items)


def graph_from_triplets(rows, title='graph'):
    G = nx.MultiDiGraph(title=title)
    for r in rows:
        s = str(r.get('subject') or '')
        p = str(r.get('predicate') or '')
        o = str(r.get('object') or '')
        if not s or not o:
            continue
        G.add_node(s, label=s)
        G.add_node(o, label=o)
        G.add_edge(s, o, predicate=p, title=f"{{p}} | {{r.get('start_date')}}..{{r.get('end_date')}}")
    return G


def render_pyvis_graph(rows, html_path: Path, title='Graph'):
    from pyvis.network import Network
    net = Network(height='720px', width='100%', directed=True, notebook=False)
    net.force_atlas_2based()
    G = graph_from_triplets(rows, title=title)
    for node, data in G.nodes(data=True):
        net.add_node(node, label=str(node), title=str(node))
    edge_counter = 0
    for u, v, data in G.edges(data=True):
        edge_counter += 1
        label = str(data.get('predicate') or '')
        tooltip = str(data.get('title') or label)
        net.add_edge(u, v, label=label, title=tooltip)
    html_path.parent.mkdir(parents=True, exist_ok=True)
    net.save_graph(str(html_path))
    return html_path


def render_hvplot(rows, title='Graph'):
    G = graph_from_triplets(rows, title=title)
    if len(G.nodes) == 0:
        return None
    pos = nx.spring_layout(G, seed=42)
    return hvnx.draw(G, pos=pos, with_labels=True, width=900, height=650)


def display_bundle(bundle_dir: Path):
    manifest = load_json(bundle_dir / 'manifest.json')
    comparison = load_json(bundle_dir / 'comparison_summary.json')
    ref_triplets = load_json(bundle_dir / 'reference_triplets.json')
    auto_triplets = load_json(bundle_dir / 'automatic_triplets.json')
    suggestions_path = bundle_dir / 'scout' / 'suggested_links.json'
    suggestions = load_json(suggestions_path) if suggestions_path.exists() else []

    display(Markdown('## Summary'))
    display(pd.DataFrame([manifest]))
    display(pd.DataFrame([comparison]))

    ref_df = triplets_to_df(ref_triplets)
    auto_df = triplets_to_df(auto_triplets)

    display(Markdown('## Reference triplets'))
    display(ref_df)

    display(Markdown('## Automatic triplets'))
    display(auto_df)

    vis_dir = bundle_dir / 'visualizations'
    ref_html = render_pyvis_graph(ref_triplets, vis_dir / 'reference_graph.html', title='Reference graph')
    auto_html = render_pyvis_graph(auto_triplets, vis_dir / 'automatic_graph.html', title='Automatic temporal KG')

    display(Markdown('## Reference graph — advanced HTML view'))
    display(IFrame(src=str(ref_html), width='100%', height='760'))
    hv_ref = render_hvplot(ref_triplets, title='Reference graph')
    if hv_ref is not None:
        display(hv_ref)

    display(Markdown('## Automatic graph — advanced HTML view'))
    display(IFrame(src=str(auto_html), width='100%', height='760'))
    hv_auto = render_hvplot(auto_triplets, title='Automatic temporal KG')
    if hv_auto is not None:
        display(hv_auto)

    if suggestions:
        display(Markdown('## Scout: дополнительные статьи-кандидаты'))
        display(pd.DataFrame(suggestions))

    print('Bundle directory:', bundle_dir)
PATCHES = json.loads(r'''{"src/scireason/config.py": "from __future__ import annotations\n\nfrom pydantic_settings import BaseSettings, SettingsConfigDict\n\n\nclass Settings(BaseSettings):\n    model_config = SettingsConfigDict(env_file=\".env\", env_file_encoding=\"utf-8\", extra=\"ignore\")\n\n    # ===== LLM =====\n    # Default: `auto` for classroom robustness.\n    # auto -> try local Ollama (if reachable) -> g4f (if installed) -> LiteLLM providers -> mock (offline)\n    llm_provider: str = \"auto\"  # auto|mock|ollama|g4f|openai|anthropic|...\n    llm_model: str = \"auto\"\n    ollama_base_url: str = \"http://localhost:11434\"\n\n    # g4f fine-tuning (optional)\n    g4f_providers: str | None = None\n    g4f_api_key: str | None = None\n\n    # ===== Embeddings =====\n    embed_provider: str = \"hash\"  # hash|sentence-transformers|openai|ollama|...\n    embed_model: str = \"sentence-transformers/all-MiniLM-L6-v2\"\n    hash_embed_dim: int = 384\n\n    # ===== Infra (optional services) =====\n    neo4j_uri: str = \"bolt://localhost:7687\"\n    neo4j_user: str = \"neo4j\"\n    neo4j_password: str = \"please_change_me\"\n\n    qdrant_url: str = \"http://localhost:6333\"\n    qdrant_api_key: str | None = None\n\n    grobid_url: str = \"http://localhost:8070\"\n\n    # ===== HTTP Client (cache + rate limiting) =====\n    http_cache_dir: str = \".cache/http\"\n    http_cache_ttl_seconds: int = 7 * 24 * 3600\n    http_timeout_seconds: int = 30\n\n    # ===== External APIs =====\n    contact_email: str | None = None\n    user_agent: str | None = None\n\n    # Semantic Scholar\n    s2_api_key: str | None = None\n\n    # NCBI / PubMed\n    ncbi_api_key: str | None = None\n    ncbi_tool: str = \"top-papers-graph\"\n    ncbi_email: str | None = None\n\n    # Crossref / OpenAlex\n    crossref_mailto: str | None = None\n    openalex_mailto: str | None = None\n    openalex_api_key: str | None = None\n\n    # ===== Demo store (retrieval few-shot) =====\n    demo_enabled: bool = True\n    demo_schema_version: str = \"1.0\"\n    demo_quality: str = \"gold\"  # gold|silver\n    demo_top_k_triplets: int = 3\n    demo_top_k_hypothesis: int = 2\n    demo_max_chars_total: int = 3500\n    demo_collection_triplets: str = \"demos_temporal_triplets\"\n    demo_collection_hypothesis: str = \"demos_hypothesis_test\"\n\n    # ===== Multimodal (VL + MM embeddings) =====\n    vlm_backend: str = \"none\"  # none|qwen2_vl|llava|phi3_vision\n    vlm_model_id: str = \"Qwen/Qwen2.5-VL-3B-Instruct\"\n    mm_embed_backend: str = \"none\"  # none|open_clip\n    open_clip_model: str = \"ViT-B-32\"\n    open_clip_pretrained: str = \"laion2b_s34b_b79k\"\n    pdf_render_dpi: int = 150\n\n    # ===== Temporal GraphRAG =====\n    temporal_default_granularity: str = \"year\"  # year|month|day\n\n    # ===== Domain =====\n    domain_id: str = \"science\"\n    domain_config_path: str | None = None\n\n    # ===== Agentic hypothesis generation (code-writing agent) =====\n    hyp_agent_enabled: bool = True\n    # internal|smolagents\n    # - internal: lightweight built-in code agent (works fully offline with --llm-provider mock)\n    # - smolagents: Hugging Face smolagents CodeAgent (optional dependency)\n    hyp_agent_backend: str = \"internal\"\n    hyp_agent_max_steps: int = 4\n    hyp_agent_timeout_seconds: int = 20\n\n    # ===== smolagents integration (optional) =====\n    # Only used when HYP_AGENT_BACKEND=smolagents.\n    #\n    # smol_model_backend:\n    # - scireason: wrap this project's LLM router (LLM_PROVIDER=auto|g4f|ollama|...) into a smolagents Model\n    # - transformers: smolagents.TransformersModel for local HF models (requires smolagents[transformers])\n    # - g4f: direct smolagents Model that calls g4f client (requires g4f)\n    smol_model_backend: str = \"scireason\"\n    smol_model_id: str = \"HuggingFaceTB/SmolLM2-1.7B-Instruct\"\n    smol_max_new_tokens: int = 768\n    smol_device_map: str | None = None\n    smol_torch_dtype: str | None = None  # e.g. float16|bfloat16\n    smol_g4f_model: str = \"auto\"\n    smol_executor: str = \"local\"  # local|docker (docker requires Docker)\n    smol_print_steps: bool = False\n\n    # ===== Temporal link prediction / TGNN =====\n    # Enabled by default: the project should prefer event-stream temporal prediction over\n    # static graph link prediction whenever the temporal KG is available.\n    hyp_tgnn_enabled: bool = True\n    hyp_tgnn_recent_window_years: int = 3\n    hyp_tgnn_half_life_years: float = 2.0\n    hyp_tgnn_min_candidate_score: float = 0.05\n\n    # ===== Static GNN baseline (PyTorch Geometric) =====\n    # Kept as an optional baseline for ablations and course comparisons.\n    hyp_gnn_enabled: bool = False\n    hyp_gnn_epochs: int = 80\n    hyp_gnn_hidden_dim: int = 64\n    hyp_gnn_lr: float = 0.01\n    # To keep training fast for classroom-sized runs, we restrict to an induced\n    # subgraph of the most connected terms.\n    hyp_gnn_node_cap: int = 300\n    hyp_gnn_seed: int = 7\n\n    # ===== Neo4j vector indexing =====\n    neo4j_vector_enabled: bool = True\n    neo4j_vector_chunk_dimensions: int = 384\n    neo4j_vector_assertion_dimensions: int = 384\n\n\nsettings = Settings()\n", "src/scireason/domain.py": "from __future__ import annotations\n\n\"\"\"Domain configuration utilities.\n\nWhy this exists\n---------------\nSciReason is designed to support many scientific domains (battery, bio, ...). Most things that are\ndomain-sensitive (seed queries, ontologies, experiment backends, validation rules for expert\nartifacts) should be configured via YAML rather than hard-coded.\n\nThe loader below is intentionally lightweight: it tries to load a YAML config file from the\nrepository (when running from source) but also works when the config is missing (e.g. when the\npackage is installed without the `configs/` directory).\n\"\"\"\n\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional\n\nimport yaml  # type: ignore\n\nfrom .config import settings\n\n\n@dataclass\nclass DomainConfig:\n    domain_id: str\n    title: str = \"Science\"\n    keywords: List[str] = field(default_factory=list)\n    seed_queries: List[str] = field(default_factory=list)\n\n    # Optional sections (free-form dictionaries)\n    kg: Dict[str, Any] = field(default_factory=dict)\n    evaluation: Dict[str, Any] = field(default_factory=dict)\n    artifact_validation: Dict[str, Any] = field(default_factory=dict)\n\n    # Term graph / temporal KG knobs (optional)\n    term_graph: Dict[str, Any] = field(default_factory=dict)\n\n\ndef _candidate_paths(domain_id: str) -> List[Path]:\n    \"\"\"Resolve config candidate paths.\n\n    Priority:\n    1) explicit settings.domain_config_path\n    2) configs/domains/<domain_id>.yaml relative to CWD\n    3) configs/domains/<domain_id>.yaml relative to repository root (best effort)\n    \"\"\"\n    paths: List[Path] = []\n    if settings.domain_config_path:\n        paths.append(Path(settings.domain_config_path))\n\n    paths.append(Path(\"configs\") / \"domains\" / f\"{domain_id}.yaml\")\n\n    # When running from source, this file lives in <repo>/src/scireason/domain.py\n    try:\n        repo_root = Path(__file__).resolve().parents[2]\n        paths.append(repo_root / \"configs\" / \"domains\" / f\"{domain_id}.yaml\")\n    except Exception:\n        pass\n\n    # Unique, existing\n    seen: set[str] = set()\n    out: List[Path] = []\n    for p in paths:\n        key = str(p)\n        if key in seen:\n            continue\n        seen.add(key)\n        if p.exists():\n            out.append(p)\n    return out\n\n\ndef load_domain_config(domain_id: Optional[str] = None) -> DomainConfig:\n    \"\"\"Load domain configuration.\n\n    Supports both canonical `domain_id` values (for example `science`) and Wikidata QIDs\n    stored in expert trajectory artifacts (for example `Q336`).\n\n    If the config file is missing, returns a minimal DomainConfig so the system stays usable.\n    \"\"\"\n    did = (domain_id or settings.domain_id or \"science\").strip()\n\n    def _from_data(data: dict[str, Any], fallback: str) -> DomainConfig:\n        return DomainConfig(\n            domain_id=str(data.get(\"domain_id\") or fallback),\n            title=str(data.get(\"title\") or fallback),\n            keywords=list(data.get(\"keywords\") or []),\n            seed_queries=list(data.get(\"seed_queries\") or []),\n            kg=dict(data.get(\"kg\") or {}),\n            evaluation=dict(data.get(\"evaluation\") or {}),\n            artifact_validation=dict(data.get(\"artifact_validation\") or {}),\n            term_graph=dict(data.get(\"term_graph\") or {}),\n        )\n\n    for path in _candidate_paths(did):\n        data = yaml.safe_load(path.read_text(encoding=\"utf-8\")) or {}\n        return _from_data(data, did)\n\n    # Expert trajectories store a Wikidata QID in `domain`. Support that form directly.\n    qid = did.upper()\n    if qid.startswith(\"Q\") and qid[1:].isdigit():\n        candidate_dirs: list[Path] = []\n        try:\n            repo_root = Path(__file__).resolve().parents[2]\n            candidate_dirs.append(repo_root / \"configs\" / \"domains\")\n        except Exception:\n            pass\n        candidate_dirs.append(Path(\"configs\") / \"domains\")\n\n        seen: set[str] = set()\n        for directory in candidate_dirs:\n            key = str(directory)\n            if key in seen or not directory.exists():\n                continue\n            seen.add(key)\n            for cfg_path in sorted(directory.glob(\"*.yaml\")):\n                try:\n                    data = yaml.safe_load(cfg_path.read_text(encoding=\"utf-8\")) or {}\n                except Exception:\n                    continue\n                if str(data.get(\"wikidata_qid\") or \"\").strip().upper() == qid:\n                    return _from_data(data, did)\n\n    # Fallback\n    return DomainConfig(domain_id=did, title=did)\n", "src/scireason/graph/review_applier.py": "from __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Tuple\n\n\n@dataclass\nclass ReviewStats:\n    accepted: int = 0\n    rejected: int = 0\n    needs_fix: int = 0\n    added: int = 0\n\n\ndef _iter_review_files(graph_reviews_dir: Path) -> Iterable[Path]:\n    yield from sorted(graph_reviews_dir.glob(\"**/*.json\"))\n\n\ndef _weight_for(verdict: str) -> float:\n    if verdict == \"accepted\":\n        return 1.0\n    if verdict == \"rejected\":\n        return -1.0\n    if verdict in {\"needs_time_fix\", \"needs_evidence_fix\"}:\n        return -0.25\n    if verdict == \"added\":\n        return 0.75\n    return 0.0\n\n\ndef _time_token(v: Any, *, default: str = \"unknown\") -> str:\n    if v is None:\n        return default\n    s = str(v).strip()\n    return s or default\n\n\ndef _canonical_time_interval(a: Dict[str, Any]) -> str:\n    legacy = str(a.get(\"time_interval\") or \"\").strip()\n    if legacy:\n        return legacy\n\n    start = _time_token(a.get(\"start_date\"), default=\"unknown\")\n    end = _time_token(a.get(\"end_date\"), default=\"unknown\")\n    valid_from = _time_token(a.get(\"valid_from\"), default=start)\n    valid_to = _time_token(a.get(\"valid_to\"), default=\"+inf\")\n    return f\"evidence:{start}..{end}|valid:{valid_from}..{valid_to}\"\n\n\ndef _normalize_assertions(doc: Dict[str, Any], source_path: Path) -> List[Dict[str, Any]]:\n    \"\"\"Support both new and legacy expert formats.\n\n    New format (preferred):\n      { assertions: [ {subject,predicate,object,time_interval,evidence,verdict,...}, ... ] }\n\n    Legacy format (kept for backward compatibility in early course phases):\n      { accepted_edges: [...], rejected_edges: [...], added_edges: [...] }\n    \"\"\"\n    assertions = doc.get(\"assertions\")\n    if isinstance(assertions, list) and assertions:\n        return assertions\n\n    out: List[Dict[str, Any]] = []\n\n    for a in doc.get(\"accepted_edges\", []) or []:\n        out.append({\n            \"assertion_id\": a.get(\"assertion_id\") or \"legacy\",\n            \"subject\": a.get(\"subject\"),\n            \"predicate\": a.get(\"predicate\"),\n            \"object\": a.get(\"object\"),\n            \"time_interval\": a.get(\"time_interval\", \"unknown\"),\n            \"start_date\": a.get(\"start_date\"),\n            \"end_date\": a.get(\"end_date\"),\n            \"valid_from\": a.get(\"valid_from\"),\n            \"valid_to\": a.get(\"valid_to\"),\n            \"time_source\": a.get(\"time_source\") or \"legacy\",\n            \"evidence\": {\n                \"page\": a.get(\"evidence\", {}).get(\"page\"),\n                \"figure_or_table\": a.get(\"evidence\", {}).get(\"figure_or_table\"),\n                \"snippet_or_summary\": a.get(\"evidence\", {}).get(\"text_snippet\") or a.get(\"evidence\", {}).get(\"snippet_or_summary\"),\n            },\n            \"verdict\": \"accepted\",\n            \"rationale\": a.get(\"rationale\") or \"legacy accepted_edges\",\n        })\n\n    for a in doc.get(\"rejected_edges\", []) or []:\n        out.append({\n            \"assertion_id\": a.get(\"assertion_id\") or \"legacy\",\n            \"subject\": a.get(\"subject\"),\n            \"predicate\": a.get(\"predicate\"),\n            \"object\": a.get(\"object\"),\n            \"time_interval\": a.get(\"time_interval\", \"unknown\"),\n            \"start_date\": a.get(\"start_date\"),\n            \"end_date\": a.get(\"end_date\"),\n            \"valid_from\": a.get(\"valid_from\"),\n            \"valid_to\": a.get(\"valid_to\"),\n            \"time_source\": a.get(\"time_source\") or \"legacy\",\n            \"evidence\": {\n                \"page\": None,\n                \"figure_or_table\": None,\n                \"snippet_or_summary\": None,\n            },\n            \"verdict\": \"rejected\",\n            \"rationale\": a.get(\"reason\") or \"legacy rejected_edges\",\n        })\n\n    for a in doc.get(\"added_edges\", []) or []:\n        out.append({\n            \"assertion_id\": a.get(\"assertion_id\") or \"legacy\",\n            \"subject\": a.get(\"subject\"),\n            \"predicate\": a.get(\"predicate\"),\n            \"object\": a.get(\"object\"),\n            \"time_interval\": a.get(\"time_interval\", \"unknown\"),\n            \"start_date\": a.get(\"start_date\"),\n            \"end_date\": a.get(\"end_date\"),\n            \"valid_from\": a.get(\"valid_from\"),\n            \"valid_to\": a.get(\"valid_to\"),\n            \"time_source\": a.get(\"time_source\") or \"legacy\",\n            \"evidence\": {\n                \"page\": None,\n                \"figure_or_table\": None,\n                \"snippet_or_summary\": a.get(\"evidence_hint\") or None,\n            },\n            \"verdict\": \"added\",\n            \"rationale\": a.get(\"reason\") or \"legacy added_edges\",\n        })\n\n    return out\n\n\ndef compile_overrides(graph_reviews_dir: Path, out_path: Path) -> ReviewStats:\n    \"\"\"Compile expert graph reviews into a DB-agnostic overrides file (JSONL).\n\n    Each line:\n      {assertion_id, verdict, weight, subject, predicate, object, time_interval, source_review}\n\n    The overrides file is used by:\n    - retriever weighting (future)\n    - rule-based reward now (immediate behavior changes)\n    - optional Neo4j tagging via CLI (`apply-graph-reviews --to-neo4j`)\n    \"\"\"\n    stats = ReviewStats()\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n\n    with out_path.open(\"w\", encoding=\"utf-8\") as out:\n        for f in _iter_review_files(graph_reviews_dir):\n            doc = json.loads(f.read_text(encoding=\"utf-8\"))\n            assertions = _normalize_assertions(doc, f)\n            for a in assertions:\n                verdict = str(a.get(\"verdict\", \"\")).strip()\n                rec = {\n                    \"assertion_id\": a.get(\"assertion_id\") or \"new\",\n                    \"verdict\": verdict,\n                    \"weight\": _weight_for(verdict),\n                    \"subject\": a.get(\"subject\"),\n                    \"predicate\": a.get(\"predicate\"),\n                    \"object\": a.get(\"object\"),\n                    \"start_date\": _time_token(a.get(\"start_date\"), default=\"unknown\"),\n                    \"end_date\": _time_token(a.get(\"end_date\"), default=\"unknown\"),\n                    \"valid_from\": _time_token(a.get(\"valid_from\"), default=_time_token(a.get(\"start_date\"), default=\"unknown\")),\n                    \"valid_to\": _time_token(a.get(\"valid_to\"), default=\"+inf\"),\n                    \"time_source\": a.get(\"time_source\") or \"unknown\",\n                    \"time_interval\": _canonical_time_interval(a),\n                    \"source_review\": str(f),\n                }\n                out.write(json.dumps(rec, ensure_ascii=False) + \"\\n\")\n\n                if verdict == \"accepted\":\n                    stats.accepted += 1\n                elif verdict == \"rejected\":\n                    stats.rejected += 1\n                elif verdict in {\"needs_time_fix\", \"needs_evidence_fix\"}:\n                    stats.needs_fix += 1\n                elif verdict == \"added\":\n                    stats.added += 1\n\n    return stats\n", "src/scireason/graph/temporal_neo4j_store.py": "from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom hashlib import sha1\nfrom typing import Any, Dict, Iterable, Optional\n\ntry:  # pragma: no cover\n    from neo4j import GraphDatabase\nexcept Exception:  # pragma: no cover\n    GraphDatabase = None  # type: ignore[assignment]\n\nfrom ..config import settings\nfrom ..temporal.schemas import TemporalEvent, TimeInterval, TemporalTriplet\n\n\ndef _assertion_id(paper_id: str, t: TemporalTriplet) -> str:\n    key = (\n        f\"{paper_id}|{t.subject}|{t.predicate}|{t.object}|{t.polarity}|\"\n        f\"{t.time.start if t.time else ''}|{t.time.end if t.time else ''}\"\n    )\n    return sha1(key.encode(\"utf-8\")).hexdigest()[:16]\n\n\n@dataclass\nclass Neo4jTemporalStore:\n    uri: str = settings.neo4j_uri\n    user: str = settings.neo4j_user\n    password: str = settings.neo4j_password\n\n    def __post_init__(self) -> None:\n        if GraphDatabase is None:\n            raise RuntimeError(\n                \"neo4j python driver is not installed. Install base dependencies with neo4j support enabled.\"\n            )\n        self._driver = GraphDatabase.driver(self.uri, auth=(self.user, self.password))\n\n    def close(self) -> None:\n        self._driver.close()\n\n    def ensure_schema(self) -> None:\n        cypher = [\n            \"CREATE CONSTRAINT paper_id IF NOT EXISTS FOR (p:Paper) REQUIRE p.id IS UNIQUE\",\n            \"CREATE CONSTRAINT entity_name IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE\",\n            \"CREATE CONSTRAINT time_key IF NOT EXISTS FOR (t:Time) REQUIRE t.key IS UNIQUE\",\n            \"CREATE CONSTRAINT assertion_id IF NOT EXISTS FOR (a:Assertion) REQUIRE a.id IS UNIQUE\",\n            \"CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (c:Chunk) REQUIRE c.id IS UNIQUE\",\n            \"CREATE CONSTRAINT event_id IF NOT EXISTS FOR (e:Event) REQUIRE e.id IS UNIQUE\",\n        ]\n        with self._driver.session() as s:\n            for q in cypher:\n                s.run(q)\n\n    def ensure_vector_indexes(\n        self,\n        *,\n        chunk_dimensions: Optional[int] = None,\n        assertion_dimensions: Optional[int] = None,\n        entity_dimensions: Optional[int] = None,\n    ) -> None:\n        \"\"\"Best-effort vector indexes for Neo4j as graph + vector DB.\n\n        Compatible with current Neo4j vector index syntax. Older servers may reject these\n        statements; callers should treat failures as non-fatal.\n        \"\"\"\n\n        queries = []\n        if chunk_dimensions:\n            queries.append(\n                (\n                    \"chunk_embedding_idx\",\n                    f\"CREATE VECTOR INDEX chunk_embedding_idx IF NOT EXISTS \"\n                    f\"FOR (c:Chunk) ON c.embedding \"\n                    f\"OPTIONS {{indexConfig: {{`vector.dimensions`: {int(chunk_dimensions)}, `vector.similarity_function`: 'cosine'}}}}\",\n                )\n            )\n        if assertion_dimensions:\n            queries.append(\n                (\n                    \"assertion_embedding_idx\",\n                    f\"CREATE VECTOR INDEX assertion_embedding_idx IF NOT EXISTS \"\n                    f\"FOR (a:Assertion) ON a.embedding \"\n                    f\"OPTIONS {{indexConfig: {{`vector.dimensions`: {int(assertion_dimensions)}, `vector.similarity_function`: 'cosine'}}}}\",\n                )\n            )\n        if entity_dimensions:\n            queries.append(\n                (\n                    \"entity_embedding_idx\",\n                    f\"CREATE VECTOR INDEX entity_embedding_idx IF NOT EXISTS \"\n                    f\"FOR (e:Entity) ON e.embedding \"\n                    f\"OPTIONS {{indexConfig: {{`vector.dimensions`: {int(entity_dimensions)}, `vector.similarity_function`: 'cosine'}}}}\",\n                )\n            )\n        with self._driver.session() as s:\n            for _, q in queries:\n                try:\n                    s.run(q)\n                except Exception:\n                    # Best effort only: older Neo4j versions or restricted deployments may not support vector indexes.\n                    continue\n\n    def upsert_paper(self, paper: Dict[str, Any]) -> None:\n        q = \"\"\"\n        MERGE (p:Paper {id: $id})\n        SET p.title = $title,\n            p.year = $year,\n            p.source = $source,\n            p.url = $url\n        \"\"\"\n        with self._driver.session() as s:\n            s.run(q, **paper)\n\n    def upsert_chunk(\n        self,\n        paper_id: str,\n        chunk_id: str,\n        text: str,\n        chunk_index: Optional[int] = None,\n        embedding: Optional[list[float]] = None,\n    ) -> None:\n        \"\"\"Upsert a text chunk node for provenance and optional vector retrieval.\"\"\"\n        t = (text or \"\").strip()\n        if len(t) > 4000:\n            t = t[:4000] + \"…\"\n\n        q = \"\"\"\n        MATCH (p:Paper {id:$paper_id})\n        MERGE (c:Chunk {id:$chunk_id})\n        SET c.paper_id=$paper_id,\n            c.idx=$chunk_index,\n            c.text=$text,\n            c.embedding=$embedding\n        MERGE (p)-[:HAS_CHUNK]->(c)\n        \"\"\"\n        with self._driver.session() as s:\n            s.run(\n                q,\n                paper_id=paper_id,\n                chunk_id=chunk_id,\n                chunk_index=chunk_index,\n                text=t,\n                embedding=embedding,\n            )\n\n    def upsert_time(self, interval: TimeInterval) -> str:\n        start = interval.start or \"\"\n        end = interval.end or start\n        key = f\"{interval.granularity}:{start}:{end}\"\n\n        q = \"\"\"\n        MERGE (t:Time {key: $key})\n        SET t.granularity = $granularity,\n            t.start = $start,\n            t.end = $end\n        RETURN t.key as key\n        \"\"\"\n        with self._driver.session() as s:\n            rec = s.run(q, key=key, granularity=interval.granularity, start=start, end=end).single()\n            assert rec\n            return rec[\"key\"]\n\n    def upsert_assertion(\n        self,\n        paper_id: str,\n        t: TemporalTriplet,\n        chunk_id: Optional[str] = None,\n        evidence_quote: Optional[str] = None,\n        embedding: Optional[list[float]] = None,\n        extraction_method: str = \"llm_triplet\",\n        review_status: str = \"pending\",\n    ) -> str:\n        aid = _assertion_id(paper_id, t)\n        time_key = None\n        if t.time:\n            time_key = self.upsert_time(t.time)\n\n        q = \"\"\"\n        MERGE (s:Entity {name: $subj})\n        MERGE (o:Entity {name: $obj})\n        MATCH (p:Paper {id: $paper_id})\n        MERGE (a:Assertion {id: $aid})\n        SET a.predicate = $pred,\n            a.confidence = $confidence,\n            a.polarity = $polarity,\n            a.paper_id = $paper_id,\n            a.evidence_quote = $evidence_quote,\n            a.embedding = $embedding,\n            a.extraction_method = $extraction_method,\n            a.review_status = $review_status,\n            a.object = $obj,\n            a.subject = $subj\n        MERGE (a)-[:SUBJECT]->(s)\n        MERGE (a)-[:OBJECT]->(o)\n        MERGE (p)-[:HAS_ASSERTION]->(a)\n        MERGE (a)-[:ASSERTED_IN]->(p)\n        WITH a\n        OPTIONAL MATCH (t:Time {key: $time_key})\n        FOREACH (_ IN CASE WHEN t IS NULL THEN [] ELSE [1] END |\n            MERGE (a)-[:AT_TIME]->(t)\n        )\n        RETURN a.id as aid\n        \"\"\"\n        with self._driver.session() as s:\n            rec = s.run(\n                q,\n                aid=aid,\n                subj=t.subject,\n                obj=t.object,\n                pred=t.predicate,\n                confidence=float(t.confidence),\n                polarity=t.polarity,\n                paper_id=paper_id,\n                time_key=time_key,\n                evidence_quote=(evidence_quote or t.evidence_quote or None),\n                embedding=embedding,\n                extraction_method=extraction_method,\n                review_status=review_status,\n            ).single()\n            assert rec\n            assertion_id = rec[\"aid\"]\n\n        if chunk_id:\n            q_ev = \"\"\"\n            MATCH (a:Assertion {id:$aid})\n            MERGE (c:Chunk {id:$chunk_id})\n            MERGE (a)-[e:EVIDENCE]->(c)\n            SET e.quote = $quote\n            \"\"\"\n            quote = (evidence_quote or t.evidence_quote or \"\").strip()\n            if len(quote) > 200:\n                quote = quote[:200] + \"…\"\n            with self._driver.session() as s:\n                s.run(q_ev, aid=assertion_id, chunk_id=chunk_id, quote=quote)\n\n        return assertion_id\n\n    def upsert_event(self, event: TemporalEvent) -> str:\n        event_id = event.stable_id()\n        time_key = self.upsert_time(\n            TimeInterval(start=event.ts_start, end=event.ts_end, granularity=event.granularity)\n        )\n        q = \"\"\"\n        MERGE (s:Entity {name:$subj})\n        MERGE (o:Entity {name:$obj})\n        MATCH (p:Paper {id:$paper_id})\n        MATCH (t:Time {key:$time_key})\n        MERGE (e:Event {id:$event_id})\n        SET e.paper_id=$paper_id,\n            e.chunk_id=$chunk_id,\n            e.assertion_id=$assertion_id,\n            e.predicate=$pred,\n            e.confidence=$confidence,\n            e.polarity=$polarity,\n            e.ts_start=$ts_start,\n            e.ts_end=$ts_end,\n            e.granularity=$granularity,\n            e.split=$split,\n            e.event_type=$event_type,\n            e.extraction_method=$extraction_method,\n            e.weight=$weight,\n            e.evidence_quote=$evidence_quote\n        MERGE (e)-[:SOURCE_ENTITY]->(s)\n        MERGE (e)-[:TARGET_ENTITY]->(o)\n        MERGE (e)-[:FROM_PAPER]->(p)\n        MERGE (e)-[:AT_TIME]->(t)\n        WITH e\n        OPTIONAL MATCH (a:Assertion {id:$assertion_id})\n        FOREACH (_ IN CASE WHEN a IS NULL THEN [] ELSE [1] END |\n            MERGE (e)-[:ASSERTS]->(a)\n        )\n        RETURN e.id as event_id\n        \"\"\"\n        with self._driver.session() as s:\n            rec = s.run(\n                q,\n                event_id=event_id,\n                paper_id=event.paper_id,\n                chunk_id=event.chunk_id,\n                assertion_id=event.assertion_id,\n                subj=event.subject,\n                obj=event.object,\n                pred=event.predicate,\n                confidence=float(event.confidence),\n                polarity=event.polarity,\n                ts_start=event.ts_start,\n                ts_end=event.ts_end,\n                granularity=event.granularity,\n                split=event.split,\n                event_type=event.event_type,\n                extraction_method=event.extraction_method,\n                weight=float(event.weight),\n                evidence_quote=event.evidence_quote,\n                time_key=time_key,\n            ).single()\n            assert rec\n            return str(rec[\"event_id\"])\n\n    def export_event_stream(self, limit: int = 1000) -> list[dict[str, Any]]:\n        q = \"\"\"\n        MATCH (e:Event)-[:SOURCE_ENTITY]->(s:Entity)\n        MATCH (e)-[:TARGET_ENTITY]->(o:Entity)\n        OPTIONAL MATCH (e)-[:AT_TIME]->(t:Time)\n        RETURN e.id as event_id,\n               e.paper_id as paper_id,\n               e.chunk_id as chunk_id,\n               e.assertion_id as assertion_id,\n               s.name as subject,\n               e.predicate as predicate,\n               o.name as object,\n               e.confidence as confidence,\n               e.polarity as polarity,\n               coalesce(e.ts_start, t.start) as ts_start,\n               coalesce(e.ts_end, t.end) as ts_end,\n               coalesce(e.granularity, t.granularity) as granularity,\n               e.split as split,\n               e.event_type as event_type,\n               e.extraction_method as extraction_method,\n               e.weight as weight,\n               e.evidence_quote as evidence_quote\n        ORDER BY coalesce(e.ts_start, t.start) ASC, e.id ASC\n        LIMIT $limit\n        \"\"\"\n        with self._driver.session() as s:\n            return [dict(r) for r in s.run(q, limit=int(limit))]\n\n    def search_chunks_by_vector(self, index_name: str, query_embedding: list[float], limit: int = 8) -> list[dict[str, Any]]:\n        q = \"\"\"\n        CALL db.index.vector.queryNodes($index_name, $limit, $embedding)\n        YIELD node, score\n        RETURN node.id as id, node.paper_id as paper_id, node.text as text, score\n        ORDER BY score DESC\n        \"\"\"\n        with self._driver.session() as s:\n            return [dict(r) for r in s.run(q, index_name=index_name, limit=int(limit), embedding=query_embedding)]\n\n    def query_assertions(self, entity: str, time: Optional[TimeInterval] = None, limit: int = 50):\n        \"\"\"Простой поиск утверждений вокруг сущности с (опциональным) фильтром по времени.\"\"\"\n        time_key = None\n        if time:\n            start = time.start or \"\"\n            end = time.end or start\n            time_key = f\"{time.granularity}:{start}:{end}\"\n\n        q = \"\"\"\n        MATCH (e:Entity {name:$entity})\n        MATCH (a:Assertion)-[:SUBJECT|OBJECT]->(e)\n        OPTIONAL MATCH (a)-[:AT_TIME]->(t:Time)\n        WITH a, t\n        WHERE coalesce(a.status, 'active') <> 'replaced'\n          AND ($time_key IS NULL OR (t.key = $time_key))\n        RETURN a.id as id, a.predicate as predicate, a.confidence as confidence, a.polarity as polarity,\n               t.start as t_start, t.end as t_end, t.granularity as granularity\n        ORDER BY a.confidence DESC\n        LIMIT $limit\n        \"\"\"\n        with self._driver.session() as s:\n            return [dict(r) for r in s.run(q, entity=entity, time_key=time_key, limit=limit)]\n\n    def apply_expert_override(\n        self,\n        subj: str,\n        pred: str,\n        obj: str,\n        verdict: str,\n        weight: float,\n        time_interval: str = \"unknown\",\n        *,\n        start_date: str = \"unknown\",\n        end_date: str = \"unknown\",\n        valid_from: str = \"unknown\",\n        valid_to: str = \"+inf\",\n        time_source: str = \"unknown\",\n    ) -> None:\n        q = \"\"\"\n        MATCH (a:Assertion)-[:SUBJECT]->(s:Entity {name: $subj})\n        MATCH (a)-[:OBJECT]->(o:Entity {name: $obj})\n        WHERE a.predicate = $pred\n        SET a.expert_verdict = $verdict,\n            a.expert_weight = $weight,\n            a.expert_time_interval = $time_interval,\n            a.expert_start_date = $start_date,\n            a.expert_end_date = $end_date,\n            a.expert_valid_from = $valid_from,\n            a.expert_valid_to = $valid_to,\n            a.expert_time_source = $time_source\n        \"\"\"\n        with self._driver.session() as s:\n            s.run(\n                q,\n                subj=subj,\n                obj=obj,\n                pred=pred,\n                verdict=verdict,\n                weight=float(weight),\n                time_interval=time_interval,\n                start_date=start_date,\n                end_date=end_date,\n                valid_from=valid_from,\n                valid_to=valid_to,\n                time_source=time_source,\n            )\n\n    def get_assertion_details(self, assertion_id: str) -> Optional[dict]:\n        q = \"\"\"\n        MATCH (p:Paper)-[:HAS_ASSERTION]->(a:Assertion {id:$aid})\n        MATCH (a)-[:SUBJECT]->(s:Entity)\n        MATCH (a)-[:OBJECT]->(o:Entity)\n        OPTIONAL MATCH (a)-[:AT_TIME]->(t:Time)\n        RETURN p.id as paper_id,\n               s.name as subject,\n               a.predicate as predicate,\n               o.name as object,\n               a.polarity as polarity,\n               a.confidence as confidence,\n               a.evidence_quote as evidence_quote,\n               t.start as t_start,\n               t.end as t_end,\n               t.granularity as granularity\n        \"\"\"\n        with self._driver.session() as s:\n            rec = s.run(q, aid=assertion_id).single()\n            return dict(rec) if rec else None\n\n    def link_replacement(self, old_id: str, new_id: str, *, rationale: str = \"\", reviewer_id: str = \"\") -> None:\n        q = \"\"\"\n        MATCH (old:Assertion {id:$old_id})\n        MATCH (new:Assertion {id:$new_id})\n        MERGE (old)-[r:REPLACED_BY]->(new)\n        SET old.status = 'replaced',\n            r.rationale = $rationale,\n            r.reviewer_id = $reviewer_id\n        \"\"\"\n        with self._driver.session() as s:\n            s.run(q, old_id=old_id, new_id=new_id, rationale=rationale, reviewer_id=reviewer_id)\n", "src/scireason/cli.py": "from __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional\n\nimport typer\nfrom rich.console import Console\nfrom rich.table import Table\n\nfrom .config import settings\nfrom .domain import load_domain_config\n\nfrom .connectors.arxiv import search as arxiv_search\nfrom .connectors.openalex import search_works as openalex_search\nfrom .connectors.semantic_scholar import search_papers as s2_search\n\nfrom .connectors.crossref import search_works as crossref_search\nfrom .connectors.pubmed import search as pubmed_search\nfrom .connectors.europe_pmc import search as europepmc_search\nfrom .connectors.biorxiv import (\n    details_by_doi as biorxiv_details_by_doi,\n    details_by_interval as biorxiv_details_by_interval,\n    normalize_record as biorxiv_normalize_record,\n)\n\nfrom .ingest.pipeline import ingest_pdf\nfrom .ingest.mm_pipeline import ingest_pdf_multimodal\nfrom .ingest.one_click import one_click_ingest_arxiv, normalize_arxiv_id\n\nfrom .graph.build_kg import build_from_paper_dir\nfrom .graph.build_tg_mmkg import build_temporal_and_multimodal\nfrom .graph.graphrag_query import retrieve_context\nfrom .graph.review_applier import compile_overrides\nfrom .graph.temporal_neo4j_store import Neo4jTemporalStore\n\nfrom .temporal.schemas import TemporalTriplet, TimeInterval\n\nfrom .agents.debate_graph import run_debate\nfrom .agents.hypothesis_tester import load_hypothesis_from_json, test_hypothesis\n\nfrom .pipeline.e2e import run_pipeline\nfrom .pipeline.demo import run_demo_pipeline\nfrom .pipeline.task2_validation import prepare_task2_validation_bundle\n\n\napp = typer.Typer(help=\"top-papers-graph CLI (ex SciReason)\", add_completion=False)\nconsole = Console()\n\ndef _user_agent() -> str:\n    \"\"\"Build a polite User-Agent for external APIs.\n\n    Many scholarly APIs (Crossref/OpenAlex/arXiv/NCBI) recommend identifying your client\n    and providing a contact email.\n    \"\"\"\n    if settings.user_agent:\n        return settings.user_agent\n    if settings.contact_email:\n        return f\"top-papers-graph (mailto:{settings.contact_email})\"\n    return \"top-papers-graph\"\n\n\n\ndef _normalize_meta(meta: Dict[str, Any], *, fallback_id: str, source: str = \"\") -> Dict[str, Any]:\n    \"\"\"Normalize metadata to the minimal fields used across the repo.\"\"\"\n    m = dict(meta)\n\n    # id\n    if not m.get(\"id\"):\n        # prefer DOI if present, else fallback\n        if m.get(\"doi\"):\n            m[\"id\"] = f\"doi:{m['doi']}\"\n        else:\n            m[\"id\"] = fallback_id\n\n    # title\n    m.setdefault(\"title\", \"\")\n\n    # year\n    if not m.get(\"year\"):\n        published = str(m.get(\"published\") or \"\")\n        if len(published) >= 4 and published[:4].isdigit():\n            m[\"year\"] = int(published[:4])\n\n    # source / url\n    if source:\n        m.setdefault(\"source\", source)\n    m.setdefault(\"url\", m.get(\"id\") if isinstance(m.get(\"id\"), str) and m[\"id\"].startswith(\"http\") else \"\")\n\n    return m\n\n\n@app.command()\ndef doctor() -> None:\n    \"\"\"Проверка окружения/настроек.\"\"\"\n    domain_cfg = load_domain_config()\n\n    t = Table(title=\"top-papers-graph doctor\")\n    t.add_column(\"Key\")\n    t.add_column(\"Value\")\n\n    t.add_row(\"Domain\", f\"{domain_cfg.domain_id} — {domain_cfg.title}\")\n    t.add_row(\"LLM provider\", settings.llm_provider)\n    t.add_row(\"LLM model\", settings.llm_model)\n    t.add_row(\"Embed provider\", settings.embed_provider)\n    t.add_row(\"Embed model\", settings.embed_model)\n    t.add_row(\"Neo4j\", settings.neo4j_uri)\n    t.add_row(\"Qdrant\", settings.qdrant_url)\n    t.add_row(\"GROBID\", settings.grobid_url)\n    t.add_row(\"CONTACT_EMAIL\", str(settings.contact_email or \"\"))\n    t.add_row(\"USER_AGENT\", _user_agent())\n    t.add_row(\"NCBI_EMAIL\", str((settings.ncbi_email or settings.contact_email) or \"\"))\n    t.add_row(\"VLM backend\", settings.vlm_backend)\n    t.add_row(\"MM embed backend\", settings.mm_embed_backend)\n\n    console.print(t)\n    console.print(\"[green]Если сервисы подняты через docker compose — вы готовы.[/green]\")\n\n    # g4f sanity: list working models from g4f/models.py (no network call)\n    try:\n        import g4f  # type: ignore\n        from g4f import models as gm  # type: ignore\n\n        models_list = []\n        try:\n            Model = getattr(gm, \"Model\", None)\n            if Model is not None and hasattr(Model, \"__all__\"):\n                cand = Model.__all__()  # type: ignore\n                if isinstance(cand, (list, tuple)):\n                    models_list = list(cand)\n        except Exception:\n            models_list = []\n\n        if not models_list:\n            models_list = list(getattr(gm, \"_all_models\", []) or [])\n\n        console.print(f\"g4f: {getattr(g4f, '__version__', 'unknown')} | models (working): {len(models_list)}\")\n        if models_list:\n            console.print(\"g4f sample models: \" + \", \".join(models_list[:10]))\n    except Exception as e:\n        console.print(f\"g4f: not available ({e})\")\n\n\n\n@app.command()\ndef fetch(\n    query: str,\n    source: str = typer.Option(\n        \"arxiv\",\n        help=\"arxiv|openalex|s2|crossref|pubmed|europepmc|biorxiv|medrxiv\",\n    ),\n    limit: int = typer.Option(10, help=\"Сколько результатов вернуть (где поддерживается)\"),\n    out: Path = typer.Option(Path(\"data/papers/search.json\"), help=\"Куда сохранить JSON\"),\n    with_abstract: bool = typer.Option(False, help=\"PubMed: подтянуть абстракт (EFetch).\"),\n    cursor: int = typer.Option(0, help=\"biorxiv/medrxiv: cursor для пагинации (по 100 записей).\"),\n    category: Optional[str] = typer.Option(None, help=\"biorxiv/medrxiv: фильтр subject category.\"),\n    normalize: bool = typer.Option(False, help=\"Нормализовать к единому PaperMetadata schema (Pydantic)\"),\n) -> None:\n    \"\"\"Поиск статей (метаданные) в одном из источников.\n\n    Источники:\n    - arxiv: arXiv Atom API (пример query: \"all:graph rag\" или \"cat:cs.AI AND all:retrieval\")\n    - openalex: OpenAlex works search\n    - s2: Semantic Scholar Graph API\n    - crossref: Crossref works search (часто полезно для кандидатов DOI)\n    - pubmed: NCBI PubMed E-utilities (ESearch + ESummary; опц. EFetch для абстрактов)\n    - europepmc: Europe PMC (агрегатор PubMed + PMC + preprints и др.)\n    - biorxiv/medrxiv: bioRxiv details API (query = DOI \"10.1101/...\" или interval \"YYYY-MM-DD/YYYY-MM-DD\" / \"Nd\" / \"N\")\n    \"\"\"\n    src = source.lower().strip()\n    ua = _user_agent()\n\n    # Unified normalization layer: returns PaperMetadata[] for all sources.\n    if normalize:\n        from scireason.papers import PaperSource as _PS, search_papers as _search\n\n        srcs = None\n        if src not in (\"all\", \"*\"):\n            parts = [p.strip() for p in src.split(\",\") if p.strip()]\n            srcs = []\n            for p in parts:\n                # map legacy shorthand\n                if p in (\"s2\", \"semanticscholar\"):\n                    p = \"semantic_scholar\"\n                if p in (\"europepmc\", \"europe_pmc\"):\n                    p = \"europe_pmc\"\n                try:\n                    srcs.append(_PS(p))\n                except Exception:\n                    continue\n\n        papers = _search(query, limit=limit, sources=srcs, with_abstracts=with_abstract)\n        # PaperMetadata contains `published_date: date` → use Pydantic JSON mode\n        # so standard `json.dumps(...)` does not fail.\n        data = [p.model_dump(mode=\"json\") for p in papers]\n\n        out.parent.mkdir(parents=True, exist_ok=True)\n        out.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n        console.print(f\"[green]Saved (normalized):[/green] {out}\")\n        return\n\n    # Legacy raw mode (source-specific outputs)\n    crossref_mailto = settings.crossref_mailto or settings.contact_email\n    openalex_mailto = settings.openalex_mailto or settings.contact_email\n    ncbi_email = settings.ncbi_email or settings.contact_email\n\n    if src == \"arxiv\":\n        data = arxiv_search(query=query, max_results=limit, user_agent=ua)\n    elif src == \"openalex\":\n        data = openalex_search(query=query, per_page=limit, mailto=openalex_mailto, api_key=settings.openalex_api_key, user_agent=ua)\n    elif src in (\"s2\", \"semanticscholar\", \"semantic_scholar\"):\n        data = s2_search(query=query, limit=limit, api_key=settings.s2_api_key)\n    elif src == \"crossref\":\n        data = crossref_search(query=query, rows=limit, mailto=crossref_mailto, user_agent=ua)\n    elif src == \"pubmed\":\n        data = pubmed_search(\n            query,\n            retmax=limit,\n            api_key=settings.ncbi_api_key,\n            tool=settings.ncbi_tool,\n            email=ncbi_email,\n            with_abstract=with_abstract,\n        )\n    elif src in (\"europepmc\", \"europe_pmc\"):\n        data = europe_pmc_search(query=query, page_size=limit, user_agent=ua)\n    elif src in (\"biorxiv\", \"medrxiv\"):\n        recs = biorxiv_search_details(\n            query=query,\n            server=src,\n            cursor=cursor,\n            category=category,\n            user_agent=ua,\n        )\n        data = recs\n    else:\n        raise typer.BadParameter(\n            \"source must be one of: arxiv, openalex, s2, crossref, pubmed, europepmc, biorxiv, medrxiv\"\n        )\n\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    console.print(f\"[green]Saved:[/green] {out}\")\n@app.command()\ndef parse(\n\n    pdf: Path = typer.Option(..., help=\"Путь к PDF\"),\n    meta: Path = typer.Option(..., help=\"Путь к meta.json\"),\n    out_dir: Path = typer.Option(Path(\"data/processed/papers\"), help=\"Корневая папка для paper_dir\"),\n) -> None:\n    \"\"\"Парсинг PDF через GROBID и сохранение чанков.\"\"\"\n    meta_obj = json.loads(meta.read_text(encoding=\"utf-8\"))\n    # best-effort normalize\n    meta_obj = _normalize_meta(meta_obj, fallback_id=pdf.stem, source=str(meta_obj.get(\"source\") or \"\"))\n    paper_dir = ingest_pdf(pdf_path=pdf, meta=meta_obj, out_dir=out_dir)\n    console.print(f\"[green]Paper stored:[/green] {paper_dir}\")\n\n\n@app.command(\"parse-mm\")\ndef parse_mm(\n    pdf: Path = typer.Option(..., help=\"Путь к PDF\"),\n    meta: Path = typer.Option(..., help=\"Путь к meta.json\"),\n    out_dir: Path = typer.Option(Path(\"data/processed/papers\"), help=\"Корневая папка для paper_dir\"),\n    vlm: bool = typer.Option(True, help=\"Включить VLM подписи/таблицы/формулы\"),\n) -> None:\n    \"\"\"Парсинг PDF + мультимодальность (страницы/картинки).\"\"\"\n    meta_obj = json.loads(meta.read_text(encoding=\"utf-8\"))\n    meta_obj = _normalize_meta(meta_obj, fallback_id=pdf.stem, source=str(meta_obj.get(\"source\") or \"\"))\n    paper_dir = ingest_pdf_multimodal(pdf_path=pdf, meta=meta_obj, out_dir=out_dir, run_vlm=vlm)\n    console.print(f\"[green]Paper stored (mm):[/green] {paper_dir}\")\n\n\n@app.command(\"ingest-arxiv\")\ndef ingest_arxiv(\n    arxiv_id: str = typer.Argument(..., help=\"arXiv id или URL (например 2401.01234 или https://arxiv.org/abs/2401.01234)\"),\n    raw_dir: Path = typer.Option(Path(\"data/raw/papers\"), help=\"Куда скачать PDF\"),\n    meta_dir: Path = typer.Option(Path(\"data/raw/metadata\"), help=\"Куда сохранить metadata JSON\"),\n    processed_dir: Path = typer.Option(Path(\"data/processed/papers\"), help=\"Куда сохранить обработанные paper_dir\"),\n    multimodal: bool = typer.Option(True, help=\"Использовать мультимодальный ingest (страницы/таблицы/формулы).\"),\n    build_graph: bool = typer.Option(True, help=\"После ingest собрать temporal+mm граф (TG-MMKG).\"),\n    collection_text: Optional[str] = typer.Option(None, help=\"Qdrant коллекция (текст). По умолчанию из domain config.\"),\n    collection_mm: Optional[str] = typer.Option(None, help=\"Qdrant коллекция (multimodal). По умолчанию <text>_mm.\"),\n) -> None:\n    \"\"\"Ingestion “в один клик”: скачать arXiv PDF + metadata resolver + ingest pipeline.\"\"\"\n    arxiv_norm = normalize_arxiv_id(arxiv_id)\n    pdf_path, meta_path = one_click_ingest_arxiv(arxiv_id=arxiv_norm, raw_dir=raw_dir, meta_dir=meta_dir)\n\n    console.print(f\"[green]Downloaded:[/green] {pdf_path}\")\n    console.print(f\"[green]Metadata:[/green] {meta_path}\")\n\n    meta_obj = json.loads(meta_path.read_text(encoding=\"utf-8\"))\n    meta_obj = _normalize_meta(\n        meta_obj,\n        fallback_id=f\"arxiv:{arxiv_norm}\",\n        source=\"arxiv\",\n    )\n    # make id stable and compact\n    meta_obj[\"id\"] = meta_obj.get(\"doi\") and f\"doi:{meta_obj['doi']}\" or f\"arxiv:{arxiv_norm}\"\n\n    if multimodal:\n        paper_dir = ingest_pdf_multimodal(pdf_path=pdf_path, meta=meta_obj, out_dir=processed_dir, run_vlm=True)\n    else:\n        paper_dir = ingest_pdf(pdf_path=pdf_path, meta=meta_obj, out_dir=processed_dir)\n\n    console.print(f\"[green]Ingested:[/green] {paper_dir}\")\n\n    if build_graph:\n        domain_cfg = load_domain_config()\n        ct = collection_text or (domain_cfg.kg.get(\"collection\") if domain_cfg.kg else None) or \"demo\"\n        cm = collection_mm\n        if cm is None:\n            cm = f\"{ct}_mm\"\n        build_temporal_and_multimodal(\n            paper_dir=paper_dir,\n            collection_text=ct,\n            collection_mm=cm if multimodal else None,\n            domain=domain_cfg.title,\n        )\n        console.print(\"[green]TG-MMKG built.[/green]\")\n\n\n@app.command(\"build-tg-mmkg\")\ndef build_tg_mmkg(\n    paper_dir: Path = typer.Option(..., help=\"Папка paper_dir (meta.json + chunks.jsonl + optional mm/)\"),\n    collection_text: str = typer.Option(\"demo\", help=\"Qdrant коллекция (текст)\"),\n    collection_mm: Optional[str] = typer.Option(None, help=\"Qdrant коллекция (multimodal). None => skip mm index.\"),\n    domain: str = typer.Option(\"Science\", help=\"Доменные подсказки для LLM при извлечении утверждений\"),\n    max_chunks_for_triplets: int = typer.Option(16, help=\"Сколько чанков использовать для извлечения темпоральных триплетов\"),\n) -> None:\n    \"\"\"Строит Temporal KG + (опционально) Multimodal индекс для одной статьи.\"\"\"\n    build_temporal_and_multimodal(\n        paper_dir=paper_dir,\n        collection_text=collection_text,\n        collection_mm=collection_mm,\n        domain=domain,\n        max_chunks_for_triplets=max_chunks_for_triplets,\n    )\n    console.print(\"[green]Done.[/green]\")\n\n\n@app.command(\"build-corpus\")\ndef build_corpus(\n    papers_dir: Path = typer.Option(Path(\"data/processed/papers\"), help=\"Корневая папка с множеством paper_dir\"),\n    collection_text: Optional[str] = typer.Option(None, help=\"Qdrant коллекция (текст). Default: domain config.\"),\n    collection_mm: Optional[str] = typer.Option(None, help=\"Qdrant коллекция (multimodal). Default: <text>_mm.\"),\n    domain: Optional[str] = typer.Option(None, help=\"Domain hint. Default: domain config title.\"),\n    max_papers: int = typer.Option(0, help=\"Если >0 — ограничить число обработанных статей\"),\n    max_chunks_for_triplets: int = typer.Option(16, help=\"Сколько чанков на статью использовать для извлечения триплетов\"),\n) -> None:\n    \"\"\"Build TG-MMKG for *all* processed papers in a directory.\n\n    This is the recommended entry point for batch ingestion once you have a folder of `paper_dir`.\n    \"\"\"\n    domain_cfg = load_domain_config()\n    ct = collection_text or (domain_cfg.kg.get(\"collection\") if domain_cfg.kg else None) or \"demo\"\n    cm = collection_mm\n    if cm is None:\n        cm = f\"{ct}_mm\"\n    dom = domain or domain_cfg.title\n\n    paper_dirs = sorted([p for p in papers_dir.iterdir() if p.is_dir() and (p / \"meta.json\").exists()])\n    if max_papers and max_papers > 0:\n        paper_dirs = paper_dirs[:max_papers]\n\n    if not paper_dirs:\n        console.print(f\"[yellow]No paper_dir found in {papers_dir}[/yellow]\")\n        raise typer.Exit(code=1)\n\n    console.print(f\"[cyan]Building corpus:[/cyan] papers={len(paper_dirs)} text_collection={ct} mm_collection={cm}\")\n\n    for i, pd in enumerate(paper_dirs, start=1):\n        console.print(f\"[dim]({i}/{len(paper_dirs)})[/dim] {pd}\")\n        try:\n            pages_path = pd / \"mm\" / \"pages.jsonl\"\n            has_mm = pages_path.exists()\n            build_temporal_and_multimodal(\n                paper_dir=pd,\n                collection_text=ct,\n                collection_mm=(cm if has_mm else None),\n                domain=dom,\n                max_chunks_for_triplets=max_chunks_for_triplets,\n            )\n        except Exception as e:\n            console.print(f\"[yellow]Skip {pd.name}: {e}[/yellow]\")\n\n    console.print(\"[green]Corpus build finished.[/green]\")\n\n\n@app.command(\"build-kg\")\ndef build_kg(\n    paper_dir: Path = typer.Option(..., help=\"Папка paper_dir (meta.json + chunks.jsonl)\"),\n    collection: str = typer.Option(\"demo\", help=\"Qdrant коллекция (текст)\"),\n    domain: str = typer.Option(\"Science\", help=\"Доменные подсказки для LLM\"),\n) -> None:\n    \"\"\"Строит обычный KG в Neo4j и эмбеддинги в Qdrant из paper_dir.\"\"\"\n    build_from_paper_dir(paper_dir=paper_dir, collection=collection, domain=domain)\n    console.print(\"[green]Done.[/green]\")\n\n\n@app.command()\ndef debate(\n    query: str,\n    collection: str = typer.Option(\"demo\", help=\"Qdrant коллекция (текст)\"),\n    domain: str = typer.Option(\"Science\", help=\"Domain hint for agents\"),\n    k: int = typer.Option(8, help=\"Сколько документов достать в контекст\"),\n    max_rounds: int = typer.Option(3, help=\"Сколько раундов дебатов\"),\n    allow_empty_context: bool = typer.Option(\n        False,\n        help=(\n            \"Разрешить запуск дебатов без найденного контекста (например, если коллекция ещё не создана). \"\n            \"По умолчанию команда подскажет, как собрать коллекцию, и завершится с ошибкой.\"\n        ),\n    ),\n) -> None:\n    \"\"\"GraphRAG: достать контекст + дебаты агентов -> гипотеза.\"\"\"\n    try:\n        ctx = retrieve_context(collection=collection, query=query, limit=k)\n    except Exception as e:\n        console.print(\n            \"[red]Failed to retrieve context from Qdrant.[/red] \"\n            \"Make sure Qdrant is running and the collection is built (parse + build-kg).\"\n        )\n        console.print(f\"[dim]{e}[/dim]\")\n        if not allow_empty_context:\n            raise typer.Exit(code=1)\n        ctx = []\n\n    if not ctx and not allow_empty_context:\n        console.print(\n            \"[yellow]No context chunks were found.[/yellow] \"\n            \"Run `top-papers-graph parse ...` and `top-papers-graph build-kg ...` first, \"\n            \"or pass --allow-empty-context to proceed without retrieval.\"\n        )\n        raise typer.Exit(code=1)\n    context_text = \"\\n\\n\".join(\n        [f\"[{c['payload'].get('paper_id')}] {c['payload'].get('text')}\" for c in ctx]\n    )\n    res = run_debate(domain=domain, context=context_text, max_rounds=max_rounds)\n    console.print(res.model_dump_json(indent=2, ensure_ascii=False))\n\n\n@app.command(\"test-hypothesis\")\ndef test_hypothesis_cmd(\n    hypothesis_json: Path = typer.Option(..., help=\"Путь к JSON (HypothesisDraft или DebateResult из команды debate)\"),\n    collection: str = typer.Option(\"demo\", help=\"Qdrant коллекция (текст)\"),\n    domain: Optional[str] = typer.Option(None, help=\"Domain hint. Default: domain config title.\"),\n    k: int = typer.Option(12, help=\"Сколько чанков извлечь для проверки\"),\n) -> None:\n    \"\"\"Проверить (verification) гипотезу на основе литературы и темпорального KG.\"\"\"\n    domain_cfg = load_domain_config()\n    dom = domain or domain_cfg.title\n\n    hyp = load_hypothesis_from_json(hypothesis_json)\n    result = test_hypothesis(domain=dom, hypothesis=hyp, collection_text=collection, k=k)\n    console.print(result.model_dump_json(indent=2, ensure_ascii=False))\n\n\n@app.command(\"refresh-feedback\")\ndef refresh_feedback(\n    graph_reviews_dir: Path = typer.Option(Path(\"data/experts/graph_reviews\"), help=\"Папка с graph_reviews (JSON).\"),\n    out_path: Path = typer.Option(Path(\"data/derived/expert_overrides.jsonl\"), help=\"Куда сохранить overrides (JSONL).\"),\n) -> None:\n    \"\"\"Graph reviews → overrides (для мгновенного эффекта на reward/retriever).\"\"\"\n    stats = compile_overrides(graph_reviews_dir, out_path)\n    console.print(f\"[green]Compiled overrides:[/green] {out_path}\")\n    console.print(\n        f\"accepted={stats.accepted} rejected={stats.rejected} needs_fix={stats.needs_fix} added={stats.added}\"\n    )\n\n\n@app.command(\"apply-graph-reviews\")\ndef apply_graph_reviews(\n    graph_reviews_dir: Path = typer.Option(Path(\"data/experts/graph_reviews\"), help=\"Папка с graph_reviews (JSON).\"),\n    overrides_path: Path = typer.Option(Path(\"data/derived/expert_overrides.jsonl\"), help=\"Куда сохранить overrides (JSONL).\"),\n    to_neo4j: bool = typer.Option(False, help=\"Проставить вердикты/веса на Assertion-нодах в Neo4j.\"),\n) -> None:\n    \"\"\"Собрать overrides и (опционально) применить к Neo4j.\"\"\"\n    stats = compile_overrides(graph_reviews_dir, overrides_path)\n    console.print(f\"[green]Compiled overrides:[/green] {overrides_path}\")\n    console.print(\n        f\"accepted={stats.accepted} rejected={stats.rejected} needs_fix={stats.needs_fix} added={stats.added}\"\n    )\n\n    if not to_neo4j:\n        return\n\n    try:\n        tneo = Neo4jTemporalStore()\n        tneo.ensure_schema()\n\n        count = 0\n        for line in overrides_path.read_text(encoding=\"utf-8\").splitlines():\n            if not line.strip():\n                continue\n            rec = json.loads(line)\n            tneo.apply_expert_override(\n                subj=str(rec.get(\"subject\")),\n                pred=str(rec.get(\"predicate\")),\n                obj=str(rec.get(\"object\")),\n                verdict=str(rec.get(\"verdict\")),\n                weight=float(rec.get(\"weight\", 0.0)),\n                time_interval=str(rec.get(\"time_interval\", \"unknown\")),\n                start_date=str(rec.get(\"start_date\", \"unknown\")),\n                end_date=str(rec.get(\"end_date\", \"unknown\")),\n                valid_from=str(rec.get(\"valid_from\", rec.get(\"start_date\", \"unknown\"))),\n                valid_to=str(rec.get(\"valid_to\", \"+inf\")),\n                time_source=str(rec.get(\"time_source\", \"unknown\")),\n            )\n            count += 1\n        tneo.close()\n        console.print(f\"[green]Applied to Neo4j:[/green] {count} overrides\")\n    except Exception as e:\n        console.print(f\"[red]Neo4j apply failed:[/red] {e}\")\n\n\n@app.command(\"apply-temporal-corrections\")\ndef apply_temporal_corrections(\n    corrections_dir: Path = typer.Option(\n        Path(\"data/experts/temporal_corrections\"), help=\"Папка с temporal_corrections (JSON).\"\n    ),\n    dry_run: bool = typer.Option(False, help=\"Не писать в Neo4j, только показать план изменений.\"),\n) -> None:\n    \"\"\"Применить temporal_corrections к Neo4j Temporal KG.\n\n    Т.к. assertion_id включает время, исправление времени реализовано как:\n    old_assertion -[:REPLACED_BY]-> new_assertion.\n    \"\"\"\n\n    paths = sorted(corrections_dir.glob(\"**/*.json\"))\n    if not paths:\n        console.print(f\"[yellow]No JSON files found in {corrections_dir}[/yellow]\")\n        return\n\n    tneo = Neo4jTemporalStore()\n    tneo.ensure_schema()\n\n    total = 0\n    created = 0\n    replaced = 0\n\n    for p in paths:\n        doc = json.loads(p.read_text(encoding=\"utf-8\"))\n        reviewer_id = str(doc.get(\"reviewer_id\", \"\"))\n        for corr in doc.get(\"corrections\", []):\n            total += 1\n            old_id = str(corr.get(\"assertion_id\", \"\")).strip()\n            if not old_id:\n                continue\n\n            details = tneo.get_assertion_details(old_id)\n            if not details:\n                console.print(f\"[yellow]Skip:[/yellow] cannot find assertion {old_id}\")\n                continue\n\n            try:\n                corrected_time = TimeInterval.model_validate(corr.get(\"corrected_time\"))\n            except Exception as e:\n                console.print(f\"[yellow]Skip:[/yellow] invalid corrected_time for {old_id} ({e})\")\n                continue\n\n            t = TemporalTriplet(\n                subject=str(details.get(\"subject\")),\n                predicate=str(details.get(\"predicate\")),\n                object=str(details.get(\"object\")),\n                confidence=float(details.get(\"confidence\") or 0.5),\n                polarity=str(details.get(\"polarity\") or \"unknown\"),\n                time=corrected_time,\n                evidence_quote=str(corr.get(\"evidence_quote\") or details.get(\"evidence_quote\") or \"\").strip() or None,\n            )\n\n            paper_id = str(details.get(\"paper_id\"))\n            if dry_run:\n                console.print(\n                    f\"[cyan]DRY RUN[/cyan] {old_id} -> time {corrected_time.start}-{corrected_time.end} ({corrected_time.granularity})\"\n                )\n                continue\n\n            new_id = tneo.upsert_assertion(paper_id=paper_id, t=t, evidence_quote=t.evidence_quote)\n            created += 1\n            tneo.link_replacement(\n                old_id,\n                new_id,\n                rationale=str(corr.get(\"rationale\", \"\")).strip(),\n                reviewer_id=reviewer_id,\n            )\n            replaced += 1\n\n    tneo.close()\n    console.print(\n        f\"[green]Temporal corrections processed[/green]: total={total}, new_assertions={created}, replaced_links={replaced}\"\n    )\n\n\n@app.command(\"prepare-task2-validation\")\ndef prepare_task2_validation(\n    trajectory: Path = typer.Option(..., help=\"Путь к YAML артефакту Task 1 (trajectory).\"),\n    out_dir: Path = typer.Option(Path(\"runs/task2_validation\"), help=\"Куда сохранить bundle для эксперта.\"),\n    multimodal: bool = typer.Option(True, help=\"Пробовать мультимодальный ingest PDF (страницы + VLM при наличии).\"),\n    vlm: bool = typer.Option(True, help=\"Запускать VLM на страницах PDF, если VLM backend настроен.\"),\n    edge_mode: str = typer.Option(\"auto\", help=\"auto|llm_triplets|cooccurrence\"),\n    suggest_links: bool = typer.Option(True, help=\"Добавить scout/suggested_links.json для поиска дополнительных ссылок.\"),\n    max_papers: int = typer.Option(0, help=\"Если >0 — ограничить число статей из trajectory YAML.\"),\n    max_link_queries: int = typer.Option(4, help=\"Сколько topic/next_question запросов использовать для scout.\"),\n) -> None:\n    \"\"\"Task 2 orchestrator: trajectory YAML -> reference graph + automatic temporal KG + review templates.\n\n    Command is designed for Google Colab / notebook usage and does not require Neo4j/Qdrant.\n    \"\"\"\n    bundle_dir = prepare_task2_validation_bundle(\n        trajectory,\n        out_dir=out_dir,\n        include_multimodal=multimodal,\n        run_vlm=vlm,\n        edge_mode=edge_mode,\n        suggest_links=suggest_links,\n        max_papers=max_papers,\n        max_link_queries=max_link_queries,\n    )\n    console.print(f\"[green]Task 2 bundle prepared:[/green] {bundle_dir}\")\n\n\n@app.command(\"pybamm-fastcharge\")\ndef pybamm_fastcharge(\n    profile: str = typer.Option(\"baseline_cc\", help=\"baseline_cc|proposed_two_stage|...\"),\n    profiles_dir: Path | None = typer.Option(\n        None,\n        \"--profiles-dir\",\n        envvar=\"CHARGING_PROFILES_DIR\",\n        help=\"Папка с YAML профилями зарядки (можно задать через CHARGING_PROFILES_DIR)\",\n    ),\n    out_dir: Path = typer.Option(Path(\"results/pybamm/run\"), help=\"Куда сохранить результаты\"),\n) -> None:\n    \"\"\"Запуск симуляции PyBaMM для профиля зарядки (пример: battery_fastcharge).\"\"\"\n    from .experiments.pybamm_fastcharge import run_simulation\n\n    out = run_simulation(profile_name=profile, out_dir=out_dir, config_dir=profiles_dir)\n    console.print(f\"[green]Saved metrics:[/green] {out}\")\n\n\n@app.command(\"import-top-papers\")\ndef import_top_papers(\n    inp: Path = typer.Option(..., help=\"JSON файл, который выдаёт top-papers-bot\"),\n    out_dir: Path = typer.Option(Path(\"configs/top_papers_meta\"), help=\"Куда сохранить meta-файлы\"),\n) -> None:\n    \"\"\"Импорт JSON из top-papers-bot в meta-файлы SciReason.\"\"\"\n    from .integrations.top_papers_import import export_meta_files\n\n    files = export_meta_files(inp, out_dir)\n    console.print(f\"[green]Generated meta files:[/green] {len(files)} → {out_dir}\")\n\n\n@app.command(\"run\")\ndef run_cmd(\n    query: str = typer.Option(..., help=\"Пользовательский запрос (topic/query).\"),\n    domain_id: str = typer.Option(\n        None,  # type: ignore[arg-type]\n        help=\"ID домена (configs/domains/<id>.yaml). По умолчанию берётся из .env (DOMAIN_ID) или science.\",\n    ),\n    sources: str = typer.Option(\n        \"all\",\n        help=\"Источники через запятую: all|openalex,semantic_scholar,crossref,arxiv,pubmed,europe_pmc,biorxiv.\",\n    ),\n    search_limit: int = typer.Option(50, help=\"Сколько результатов запросить у источников.\"),\n    top_papers: int = typer.Option(20, help=\"Сколько лучших статей взять в пайплайн.\"),\n    out_dir: Path = typer.Option(Path(\"runs\"), help=\"Куда сохранить артефакты запуска.\"),\n    multimodal: bool = typer.Option(False, help=\"Извлекать страницы/картинки (MM) при наличии зависимостей.\"),\n    no_llm_hypotheses: bool = typer.Option(False, help=\"Не использовать LLM для переформулировки гипотез.\"),\n\n    # --- LLM overrides (CLI > env/config defaults) ---\n    llm: Optional[str] = typer.Option(\n        None,\n        \"--llm\",\n        help=(\n            \"Переопределить LLM одним флагом. Форматы: 'g4f:deepseek-r1', 'g4f:gpt-4o-mini', \"\n            \"'local:llama3.2' (Ollama), 'ollama:llama3.2', или 'openai/gpt-4o-mini' (LiteLLM).\"\n        ),\n    ),\n    g4f_model: Optional[str] = typer.Option(\n        None,\n        \"--g4f-model\",\n        help=\"Явно использовать g4f с указанной моделью (например deepseek-r1).\",\n    ),\n    local_model: Optional[str] = typer.Option(\n        None,\n        \"--local-model\",\n        help=\"Явно использовать локальную Ollama модель (например llama3.2).\",\n    ),\n    llm_provider: Optional[str] = typer.Option(\n        None,\n        \"--llm-provider\",\n        help=\"Явно задать провайдера (g4f|ollama|openai|anthropic|...).\",\n    ),\n    llm_model: Optional[str] = typer.Option(\n        None,\n        \"--llm-model\",\n        help=\"Явно задать имя модели провайдера.\",\n    ),\n    smol_model_backend: Optional[str] = typer.Option(\n        None,\n        \"--smol-model-backend\",\n        help=\"smolagents model backend (scireason|transformers|g4f). Overrides SMOL_MODEL_BACKEND.\",\n    ),\n    smol_model_id: Optional[str] = typer.Option(\n        None,\n        \"--smol-model-id\",\n        help=\"HF model id/path for smolagents TransformersModel. Overrides SMOL_MODEL_ID.\",\n    ),\n) -> None:\n    \"\"\"Полностью автоматический пайплайн: query → papers → temporal KG → hypotheses.\"\"\"\n    # ---- Apply LLM overrides ----\n    def _apply_llm_overrides() -> None:\n        # 1) single-flag format\n        if llm:\n            raw = llm.strip()\n\n            # Accept provider/model as \"provider:model\" or \"provider/model\"\n            if \":\" in raw:\n                prov, model = raw.split(\":\", 1)\n            elif \"/\" in raw:\n                prov, model = raw.split(\"/\", 1)\n            else:\n                # No separator -> assume g4f model\n                prov, model = \"g4f\", raw\n\n            prov = prov.strip().lower()\n            model = model.strip()\n\n            if prov in {\"local\", \"ollama\"}:\n                settings.llm_provider = \"ollama\"\n                settings.llm_model = model\n            elif prov == \"g4f\":\n                settings.llm_provider = \"g4f\"\n                settings.llm_model = model\n            else:\n                # LiteLLM-style provider/model\n                settings.llm_provider = prov\n                settings.llm_model = model\n            return\n\n        # 2) convenience flags\n        if local_model:\n            settings.llm_provider = \"ollama\"\n            settings.llm_model = local_model.strip()\n            return\n\n        if g4f_model:\n            settings.llm_provider = \"g4f\"\n            settings.llm_model = g4f_model.strip()\n            return\n\n        # 3) explicit provider/model flags\n        if llm_provider:\n            settings.llm_provider = llm_provider.strip()\n        if llm_model:\n            settings.llm_model = llm_model.strip()\n\n    # Apply overrides (CLI > env/config defaults)\n    _apply_llm_overrides()\n\n    # smolagents model overrides (CLI > env)\n    if smol_model_backend:\n        settings.smol_model_backend = smol_model_backend.strip()\n    if smol_model_id:\n        settings.smol_model_id = smol_model_id.strip()\n\n    console.print(\n        f\"[bold]LLM:[/bold] {settings.llm_provider}/{settings.llm_model}  |  \"\n        f\"[bold]Embeddings:[/bold] {getattr(settings, 'embed_provider', 'hash')}\"\n    )\n\n    did = domain_id or settings.domain_id or \"science\"\n    src_list = None\n    if sources.strip().lower() != \"all\":\n        src_list = [s.strip() for s in sources.split(\",\") if s.strip()]\n\n    run_path = run_pipeline(\n        query=query,\n        domain_id=did,\n        sources=src_list,\n        search_limit=search_limit,\n        top_papers=top_papers,\n        run_dir=out_dir,\n        include_multimodal=multimodal,\n        use_llm_for_hypotheses=not no_llm_hypotheses,\n    )\n    console.print(f\"[bold green]Artifacts saved:[/bold green] {run_path}\")\n\n\n@app.command(\"export-temporal-events\")\ndef export_temporal_events(\n    out: Path = typer.Option(Path(\"runs/temporal_events.json\"), help=\"Where to save the exported event stream JSON.\"),\n    limit: int = typer.Option(5000, help=\"Maximum number of Neo4j events to export.\"),\n) -> None:\n    \"\"\"Export the Event layer from Neo4j for temporal model training/evaluation.\"\"\"\n    store = Neo4jTemporalStore()\n    try:\n        rows = store.export_event_stream(limit=limit)\n    finally:\n        store.close()\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    console.print(f\"[green]Exported events:[/green] {out} (n={len(rows)})\")\n\n\n@app.command(\"train-tgn\")\ndef train_tgn(\n    temporal_kg_json: Path = typer.Option(..., help=\"Path to temporal_kg.json produced by the pipeline.\"),\n    out: Path = typer.Option(Path(\"runs/tgn_predictions.json\"), help=\"Where to save top temporal link predictions.\"),\n    top_k: int = typer.Option(20, help=\"Number of temporal link predictions to keep.\"),\n) -> None:\n    \"\"\"Train/evaluate the lightweight TGNN-style predictor on a temporal KG artifact.\"\"\"\n    from .temporal.temporal_kg_builder import TemporalKnowledgeGraph, EdgeStats, NodeStats\n    from .tgnn.event_dataset import build_event_stream, chronological_split, event_stats\n    from .tgnn.tgn_link_prediction import TGNLinkPredConfig, tgn_link_prediction\n\n    raw = json.loads(temporal_kg_json.read_text(encoding=\"utf-8\"))\n    kg = TemporalKnowledgeGraph(meta=dict(raw.get(\"meta\") or {}))\n    for n in raw.get(\"nodes\", []):\n        term = str(n.get(\"term\") or \"\")\n        if not term:\n            continue\n        kg.nodes[term] = NodeStats(term=term, doc_freq=int(n.get(\"doc_freq\") or 0), yearly_doc_freq=dict(n.get(\"yearly_doc_freq\") or {}))\n    for e in raw.get(\"edges\", []):\n        edge = EdgeStats(\n            source=str(e.get(\"source\") or \"\"),\n            target=str(e.get(\"target\") or \"\"),\n            predicate=str(e.get(\"predicate\") or \"may_relate_to\"),\n            directed=bool(e.get(\"directed\", True)),\n            total_count=int(e.get(\"total_count\") or 0),\n            yearly_count={int(k): int(v) for k, v in dict(e.get(\"yearly_count\") or {}).items()},\n            confidence_sum=float(e.get(\"mean_confidence\") or 0.0) * max(1, int(e.get(\"total_count\") or 1)),\n            confidence_n=max(1, int(e.get(\"total_count\") or 1)),\n            polarity_counts=dict(e.get(\"polarity_counts\") or {\"supports\": 0, \"contradicts\": 0, \"unknown\": 0}),\n            papers=set(e.get(\"papers\") or []),\n            evidence_quotes=list(e.get(\"evidence_quotes\") or []),\n            features=dict(e.get(\"features\") or {}),\n            score=float(e.get(\"score\") or 0.0),\n        )\n        kg.edges.append(edge)\n\n    events = build_event_stream(kg)\n    train_events, valid_events, test_events = chronological_split(events)\n    preds = tgn_link_prediction(\n        train_events + valid_events,\n        top_k=top_k,\n        config=TGNLinkPredConfig(\n            recent_window_years=int(getattr(settings, \"hyp_tgnn_recent_window_years\", 3) or 3),\n            recency_half_life_years=float(getattr(settings, \"hyp_tgnn_half_life_years\", 2.0) or 2.0),\n            min_candidate_score=float(getattr(settings, \"hyp_tgnn_min_candidate_score\", 0.05) or 0.05),\n        ),\n    )\n    payload = {\n        \"stats\": {\n            **event_stats(events),\n            \"train_events\": len(train_events),\n            \"valid_events\": len(valid_events),\n            \"test_events\": len(test_events),\n        },\n        \"predictions\": [\n            {\"source\": u, \"target\": v, \"score\": score}\n            for u, v, score in preds\n        ],\n    }\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    console.print(f\"[green]Saved TGNN predictions:[/green] {out}\")\n\n\n@app.command(\"demo-run\")\ndef demo_run_cmd(\n    query: str = typer.Option(\"temporal knowledge graph hypothesis\", help=\"Demo query (offline).\"),\n    edge_mode: str = typer.Option(\"cooccurrence\", help=\"cooccurrence|llm_triplets\"),\n    out_dir: Path = typer.Option(Path(\"runs\"), help=\"Where to write demo artifacts.\"),\n    domain_id: str = typer.Option(None, help=\"Domain config id (defaults to env DOMAIN_ID or science).\"),\n    no_llm_hypotheses: bool = typer.Option(False, help=\"Disable LLM rewriting for hypotheses.\"),\n    tgnn: bool = typer.Option(True, help=\"Enable TGNN/TGN-style temporal link prediction (default on).\"),\n    gnn: bool = typer.Option(False, help=\"Enable optional static GNN baseline (requires '.[gnn]').\"),\n    agent_backend: Optional[str] = typer.Option(\n        None,\n        help=\"Override HYP_AGENT_BACKEND for this run (internal|smolagents).\",\n    ),\n    llm_provider: Optional[str] = typer.Option(None, help=\"Override LLM_PROVIDER for this run (e.g. mock).\"),\n    llm_model: Optional[str] = typer.Option(None, help=\"Override LLM_MODEL for this run.\"),\n    smol_model_backend: Optional[str] = typer.Option(\n        None,\n        \"--smol-model-backend\",\n        help=\"smolagents model backend (scireason|transformers|g4f). Overrides SMOL_MODEL_BACKEND.\",\n    ),\n    smol_model_id: Optional[str] = typer.Option(\n        None,\n        \"--smol-model-id\",\n        help=\"HF model id/path for smolagents TransformersModel. Overrides SMOL_MODEL_ID.\",\n    ),\n) -> None:\n    \"\"\"Offline demo pipeline: build temporal KG + hypotheses from a tiny built-in corpus.\n\n    This command is used for smoke tests and for the first classroom run without network/services.\n    \"\"\"\n\n    if llm_provider:\n        settings.llm_provider = llm_provider.strip()\n    if llm_model:\n        settings.llm_model = llm_model.strip()\n    if smol_model_backend:\n        settings.smol_model_backend = smol_model_backend.strip()\n    if smol_model_id:\n        settings.smol_model_id = smol_model_id.strip()\n\n    settings.hyp_tgnn_enabled = bool(tgnn)\n    if gnn:\n        settings.hyp_gnn_enabled = True\n\n    if agent_backend:\n        settings.hyp_agent_backend = agent_backend.strip()\n\n    did = domain_id or settings.domain_id or \"science\"\n    run_path = run_demo_pipeline(\n        query=query,\n        domain_id=did,\n        edge_mode=edge_mode,\n        out_dir=out_dir,\n        use_llm_for_hypotheses=not no_llm_hypotheses,\n    )\n    console.print(f\"[bold green]Demo artifacts saved:[/bold green] {run_path}\")\n\n\n@app.command(\"smoke-all\")\ndef smoke_all(\n    out_dir: Path = typer.Option(Path(\"runs\"), help=\"Where to write artifacts.\"),\n    include_g4f: bool = typer.Option(\n        False,\n        help=\"Also run smoke with LLM_PROVIDER=g4f (requires '.[g4f]' and internet; can be unstable).\",\n    ),\n    smol_model_backend: Optional[str] = typer.Option(\n        None,\n        \"--smol-model-backend\",\n        help=\"smolagents model backend for smolagents runs (scireason|transformers|g4f).\",\n    ),\n    smol_model_id: Optional[str] = typer.Option(\n        None,\n        \"--smol-model-id\",\n        help=\"HF model id/path for smolagents TransformersModel.\",\n    ),\n) -> None:\n    \"\"\"Run an offline smoke matrix for key pipeline branches.\"\"\"\n\n    # Prefer deterministic offline mode by default.\n    llm_providers = [\"mock\"]\n    if include_g4f:\n        llm_providers.append(\"g4f\")\n\n    # Try both agent backends if smolagents is available.\n    import importlib.util\n\n    agent_backends = [\"internal\"]\n    if importlib.util.find_spec(\"smolagents\") is not None:\n        agent_backends.append(\"smolagents\")\n\n    # smolagents model overrides (CLI > env)\n    if smol_model_backend:\n        settings.smol_model_backend = smol_model_backend.strip()\n    if smol_model_id:\n        settings.smol_model_id = smol_model_id.strip()\n\n    combos = [\n        (\"cooccurrence\", True, False),\n        (\"cooccurrence\", False, False),\n        (\"llm_triplets\", True, False),\n        (\"llm_triplets\", False, False),\n        # Optional GNN branch (best-effort; will fall back if PyG isn't installed)\n        (\"cooccurrence\", True, True),\n        (\"cooccurrence\", False, True),\n    ]\n    for llm_provider in llm_providers:\n        settings.llm_provider = llm_provider\n        settings.llm_model = \"mock\" if llm_provider == \"mock\" else (settings.llm_model or \"auto\")\n\n        for agent_backend in agent_backends:\n            settings.hyp_agent_backend = agent_backend\n            for edge_mode, no_llm, gnn in combos:\n                settings.hyp_gnn_enabled = bool(gnn)\n                console.print(\n                    f\"[cyan]Smoke[/cyan] llm_provider={llm_provider} agent_backend={agent_backend} edge_mode={edge_mode} no_llm_hypotheses={no_llm} gnn={gnn}\"\n                )\n                rp = run_demo_pipeline(\n                    query=\"demo smoke\",\n                    domain_id=settings.domain_id or \"science\",\n                    edge_mode=edge_mode,\n                    out_dir=out_dir,\n                    use_llm_for_hypotheses=not no_llm,\n                )\n                # Ensure key artifacts exist\n                for f in [\"paper_records.json\", \"temporal_kg.json\", \"hypotheses.json\"]:\n                    p = rp / f\n                    if not p.exists():\n                        raise RuntimeError(f\"Smoke failed: missing {p}\")\n    console.print(\"[bold green]Smoke-all: OK[/bold green]\")\n\n\nif __name__ == \"__main__\":\n    app()\n", "src/scireason/mm/vlm.py": "from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom typing import Optional, Literal\n\nfrom ..config import settings\n\n\nBackend = Literal[\"none\", \"qwen2_vl\", \"llava\", \"phi3_vision\", \"g4f\"]\n\n\n@dataclass\nclass VLMResult:\n    \"\"\"Результат работы VL-модели по изображению страницы/фигуры.\"\"\"\n    caption: str\n    extracted_tables_md: Optional[str] = None\n    extracted_equations_md: Optional[str] = None\n\n\ndef _require(pkg: str) -> None:\n    raise RuntimeError(\n        f\"Для VLM backend нужна зависимость '{pkg}'.\\n\"\n        \"Установите extras: pip install -e '.[mm]'\\n\"\n        \"Или отключите VLM: export VLM_BACKEND=none\"\n    )\n\n\n@lru_cache(maxsize=1)\ndef _load_transformers_vlm(model_id: str):\n    \"\"\"Load and cache a HF vision-language model.\n\n    Qwen2.5-VL имеет собственный класс в Transformers, поэтому для него\n    используем официальный путь через `Qwen2_5_VLForConditionalGeneration`.\n    Для остальных backends оставляем AutoModelForVision2Seq fallback.\n    \"\"\"\n    try:\n        import torch  # type: ignore\n        from transformers import AutoProcessor  # type: ignore\n    except Exception:\n        _require(\"transformers/torch\")\n\n    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)\n\n    if \"Qwen/Qwen2.5-VL\" in model_id or \"Qwen2.5-VL\" in model_id:\n        try:\n            from transformers import Qwen2_5_VLForConditionalGeneration  # type: ignore\n        except Exception:\n            _require(\"transformers>=Qwen2.5-VL support\")\n        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(\n            model_id,\n            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\n            device_map=\"auto\" if torch.cuda.is_available() else None,\n            trust_remote_code=True,\n        )\n        return processor, model, \"qwen2_5_vl\"\n\n    try:\n        from transformers import AutoModelForVision2Seq  # type: ignore\n    except Exception:\n        _require(\"transformers/torch\")\n\n    model = AutoModelForVision2Seq.from_pretrained(\n        model_id,\n        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\n        device_map=\"auto\" if torch.cuda.is_available() else None,\n        trust_remote_code=True,\n    )\n    return processor, model, \"generic\"\n\n\ndef _describe_image_g4f(image_path: Path, prompt: str, model_id: str) -> VLMResult:\n    try:\n        import base64\n        from g4f.client import Client as G4FClient  # type: ignore\n    except Exception as e:  # pragma: no cover\n        raise RuntimeError(\"Для VLM backend='g4f' нужен пакет g4f (pip install -e '.[g4f]').\") from e\n\n    client = G4FClient(api_key=getattr(settings, \"g4f_api_key\", None) or None)\n    providers = None\n    raw_providers = getattr(settings, \"g4f_providers\", None)\n    if raw_providers:\n        providers = [p.strip() for p in str(raw_providers).split(\",\") if p.strip()]\n        if len(providers) == 1:\n            providers = providers[0]\n\n    mime = \"image/png\"\n    suffix = image_path.suffix.lower()\n    if suffix in {\".jpg\", \".jpeg\"}:\n        mime = \"image/jpeg\"\n    elif suffix == \".webp\":\n        mime = \"image/webp\"\n\n    data_url = f\"data:{mime};base64,\" + base64.b64encode(image_path.read_bytes()).decode(\"ascii\")\n\n    text = None\n    errors = []\n\n    try:\n        resp = client.chat.completions.create(\n            model=model_id,\n            provider=providers,\n            messages=[\n                {\n                    \"role\": \"user\",\n                    \"content\": [\n                        {\"type\": \"text\", \"text\": prompt},\n                        {\"type\": \"image_url\", \"image_url\": {\"url\": data_url}},\n                    ],\n                }\n            ],\n        )\n        text = resp.choices[0].message.content if getattr(resp, \"choices\", None) else None\n    except Exception as e:\n        errors.append(f\"openai_style={type(e).__name__}: {e}\")\n\n    if not text:\n        try:\n            resp = client.chat.completions.create(\n                model=model_id,\n                provider=providers,\n                messages=[{\"role\": \"user\", \"content\": prompt}],\n                images=[[data_url, image_path.name]],\n            )\n            text = resp.choices[0].message.content if getattr(resp, \"choices\", None) else None\n        except Exception as e:\n            errors.append(f\"images_arg={type(e).__name__}: {e}\")\n\n    if not text:\n        raise RuntimeError(\"; \".join(errors) or \"g4f multimodal call returned empty response\")\n\n    return VLMResult(caption=str(text).strip())\n\n\ndef _describe_image_qwen(image_path: Path, prompt: str, model_id: str, max_new_tokens: int) -> VLMResult:\n    try:\n        import torch  # type: ignore\n        from PIL import Image  # type: ignore\n    except Exception:\n        _require(\"torch/pillow\")\n\n    processor, model, family = _load_transformers_vlm(model_id)\n    img = Image.open(image_path).convert(\"RGB\")\n    full_prompt = (\n        \"Ты — научный ассистент. \"\n        \"1) Дай краткую подпись к изображению (1-3 предложения). \"\n        \"2) Если на изображении есть таблицы — извлеки их в Markdown. \"\n        \"3) Если есть формулы — перепиши их в LaTeX. \"\n        f\"\\n\\nЗадача/контекст: {prompt}\"\n    )\n\n    if family == \"qwen2_5_vl\":\n        messages = [\n            {\n                \"role\": \"user\",\n                \"content\": [\n                    {\"type\": \"image\", \"image\": img},\n                    {\"type\": \"text\", \"text\": full_prompt},\n                ],\n            }\n        ]\n        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        inputs = processor(text=[text], images=[img], padding=True, return_tensors=\"pt\")\n        if torch.cuda.is_available():\n            inputs = {k: v.to(model.device) for k, v in inputs.items()}\n        with torch.no_grad():\n            generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)\n        prompt_len = inputs[\"input_ids\"].shape[1]\n        trimmed = [out_ids[prompt_len:] for out_ids in generated_ids]\n        raw_text = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()\n    else:\n        inputs = processor(images=img, text=full_prompt, return_tensors=\"pt\")\n        if torch.cuda.is_available():\n            inputs = {k: v.to(model.device) for k, v in inputs.items()}\n        with torch.no_grad():\n            out = model.generate(**inputs, max_new_tokens=max_new_tokens)\n        raw_text = processor.batch_decode(out, skip_special_tokens=True)[0].strip()\n\n    caption = raw_text\n    tables_md = None\n    equations_md = None\n    m = caption.split(\"TABLES:\", 1)\n    if len(m) == 2:\n        caption, rest = m[0].strip(), m[1].strip()\n        m2 = rest.split(\"EQUATIONS:\", 1)\n        if len(m2) == 2:\n            tables_md, equations_md = m2[0].strip(), m2[1].strip()\n        else:\n            tables_md = rest\n\n    return VLMResult(caption=caption, extracted_tables_md=tables_md, extracted_equations_md=equations_md)\n\n\ndef describe_image(\n    image_path: Path,\n    prompt: str,\n    backend: Optional[Backend] = None,\n    model_id: Optional[str] = None,\n    max_new_tokens: int = 512,\n) -> VLMResult:\n    \"\"\"Описывает изображение (страница PDF / figure / table) через VL-модель.\"\"\"\n    backend = backend or settings.vlm_backend  # type: ignore[attr-defined]\n    model_id = model_id or settings.vlm_model_id  # type: ignore[attr-defined]\n\n    if backend == \"none\":\n        return VLMResult(caption=\"\")\n\n    if backend == \"g4f\":\n        return _describe_image_g4f(image_path=image_path, prompt=prompt, model_id=model_id)\n\n    if backend == \"qwen2_vl\":\n        return _describe_image_qwen(image_path=image_path, prompt=prompt, model_id=model_id, max_new_tokens=max_new_tokens)\n\n    return _describe_image_qwen(image_path=image_path, prompt=prompt, model_id=model_id, max_new_tokens=max_new_tokens)\n", "src/scireason/pipeline/task2_validation.py": "from __future__ import annotations\n\nimport json\nimport re\nimport shutil\nfrom dataclasses import asdict\nfrom datetime import datetime\nfrom difflib import SequenceMatcher\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple\nfrom urllib.parse import urlparse, unquote\n\nimport yaml  # type: ignore\nfrom rich.console import Console\n\nfrom ..domain import DomainConfig, load_domain_config\nfrom ..ingest.acquire import AcquireResult, acquire_pdfs\nfrom ..ingest.mm_pipeline import ingest_pdf_multimodal_auto\nfrom ..ingest.pipeline import ingest_pdf_auto\nfrom ..papers.schema import ExternalIds, PaperMetadata, PaperSource\nfrom ..papers.service import get_paper_by_doi, search_papers\nfrom ..temporal.temporal_kg_builder import PaperRecord, TemporalKnowledgeGraph, build_temporal_kg, load_papers_from_processed\n\n\nconsole = Console()\n\n\nDOI_RE = re.compile(r\"10\\.\\d{4,9}/[-._;()/:A-Z0-9]+\", re.IGNORECASE)\nARXIV_RE = re.compile(r\"(?:arxiv\\.org/(?:abs|pdf)/|arxiv:)?([a-z\\-]+/\\d{7}|\\d{4}\\.\\d{4,5})(?:v\\d+)?\", re.IGNORECASE)\nOPENALEX_RE = re.compile(r\"(?:openalex\\.org/)?(W\\d+)\", re.IGNORECASE)\nPMID_RE = re.compile(r\"(?:pubmed(?:\\.ncbi\\.nlm\\.nih\\.gov)?/)?(\\d{5,10})\", re.IGNORECASE)\nPMCID_RE = re.compile(r\"(PMC\\d+)\", re.IGNORECASE)\nDATE_TOKEN_RE = re.compile(r\"^(?:\\d{4}|\\d{4}-\\d{2}|\\d{4}-\\d{2}-\\d{2}|unknown|\\+inf|-inf)$\")\n\n\nDEFAULT_SEARCH_SOURCES = [\n    PaperSource.semantic_scholar,\n    PaperSource.openalex,\n    PaperSource.crossref,\n    PaperSource.pubmed,\n    PaperSource.europe_pmc,\n    PaperSource.arxiv,\n    PaperSource.biorxiv,\n]\n\n\ndef _slugify(text: str) -> str:\n    s = re.sub(r\"[^a-z0-9\\-\\s_]+\", \"\", (text or \"\").strip().lower())\n    s = re.sub(r\"\\s+\", \"-\", s).strip(\"-\")\n    return s[:80] or \"task2\"\n\n\ndef _utc_now() -> str:\n    return datetime.utcnow().strftime(\"%Y-%m-%dT%H:%M:%SZ\")\n\n\ndef _norm_ws(text: str) -> str:\n    return \" \".join((text or \"\").split())\n\n\ndef _norm_title(text: str) -> str:\n    s = _norm_ws(text).lower()\n    s = re.sub(r\"\\s+\", \" \", s)\n    s = re.sub(r\"[^\\w\\s]+\", \"\", s, flags=re.UNICODE)\n    return s.strip()\n\n\ndef _looks_like_url(text: str) -> bool:\n    try:\n        u = urlparse((text or \"\").strip())\n    except Exception:\n        return False\n    return bool(u.scheme and u.netloc)\n\n\ndef _extract_doi(text: str) -> Optional[str]:\n    s = unquote((text or \"\").strip())\n    m = DOI_RE.search(s)\n    if not m:\n        return None\n    doi = m.group(0).rstrip(\").,;]\")\n    return doi\n\n\ndef _extract_arxiv(text: str) -> Optional[str]:\n    s = unquote((text or \"\").strip())\n    m = ARXIV_RE.search(s)\n    if not m:\n        return None\n    return m.group(1)\n\n\ndef _extract_openalex(text: str) -> Optional[str]:\n    s = unquote((text or \"\").strip())\n    m = OPENALEX_RE.search(s)\n    if not m:\n        return None\n    return m.group(1).upper()\n\n\ndef _extract_pmid(text: str) -> Optional[str]:\n    s = unquote((text or \"\").strip())\n    m = PMID_RE.search(s)\n    if not m:\n        return None\n    return m.group(1)\n\n\ndef _extract_pmcid(text: str) -> Optional[str]:\n    s = unquote((text or \"\").strip())\n    m = PMCID_RE.search(s)\n    if not m:\n        return None\n    return m.group(1).upper()\n\n\ndef _paper_year(meta: PaperMetadata) -> Optional[int]:\n    if meta.year is not None:\n        try:\n            return int(meta.year)\n        except Exception:\n            return None\n    if meta.published_date is not None:\n        try:\n            return int(meta.published_date.year)\n        except Exception:\n            return None\n    return None\n\n\ndef _score_title_match(a: str, b: str) -> float:\n    na = _norm_title(a)\n    nb = _norm_title(b)\n    if not na or not nb:\n        return 0.0\n    if na == nb:\n        return 1.0\n    return SequenceMatcher(None, na, nb).ratio()\n\n\ndef _pick_best_candidate(entry: Dict[str, Any], candidates: Sequence[PaperMetadata]) -> Optional[PaperMetadata]:\n    title = str(entry.get(\"title\") or \"\")\n    year_raw = entry.get(\"year\")\n    try:\n        entry_year = int(year_raw) if year_raw not in (None, \"\") else None\n    except Exception:\n        entry_year = None\n    raw_id = str(entry.get(\"id\") or \"\")\n    doi_hint = _extract_doi(raw_id)\n    arxiv_hint = _extract_arxiv(raw_id)\n    openalex_hint = _extract_openalex(raw_id)\n\n    scored: List[Tuple[float, PaperMetadata]] = []\n    for cand in candidates:\n        score = 0.0\n        if title:\n            score += 4.0 * _score_title_match(title, cand.title)\n        cy = _paper_year(cand)\n        if entry_year is not None and cy is not None:\n            score += max(0.0, 1.0 - 0.25 * abs(entry_year - cy))\n        if doi_hint and cand.ids and cand.ids.doi and cand.ids.doi.lower() == doi_hint.lower():\n            score += 5.0\n        if arxiv_hint and cand.ids and cand.ids.arxiv and cand.ids.arxiv.lower() == arxiv_hint.lower():\n            score += 4.0\n        if openalex_hint and cand.ids and cand.ids.openalex and cand.ids.openalex.upper() == openalex_hint.upper():\n            score += 4.0\n        if raw_id and cand.id.lower() == raw_id.lower():\n            score += 5.0\n        if cand.pdf_url:\n            score += 0.25\n        if cand.abstract:\n            score += 0.1\n        scored.append((score, cand))\n\n    if not scored:\n        return None\n    scored.sort(key=lambda x: x[0], reverse=True)\n    best_score, best = scored[0]\n    if best_score < 1.75:\n        return None\n    return best\n\n\ndef _fallback_paper(entry: Dict[str, Any]) -> PaperMetadata:\n    raw_id = str(entry.get(\"id\") or \"\").strip()\n    title = str(entry.get(\"title\") or raw_id or \"Untitled paper\").strip()\n    try:\n        year = int(entry.get(\"year\")) if entry.get(\"year\") not in (None, \"\") else None\n    except Exception:\n        year = None\n\n    canonical_id = raw_id\n    if not canonical_id:\n        canonical_id = f\"manual:{_slugify(title)}\"\n    elif _extract_doi(canonical_id):\n        canonical_id = f\"doi:{_extract_doi(canonical_id)}\"\n    elif _extract_arxiv(canonical_id):\n        canonical_id = f\"arxiv:{_extract_arxiv(canonical_id)}\"\n    elif _extract_openalex(canonical_id):\n        canonical_id = f\"openalex:{_extract_openalex(canonical_id)}\"\n    elif _extract_pmcid(canonical_id):\n        canonical_id = f\"pmc:{_extract_pmcid(canonical_id)}\"\n    elif _extract_pmid(canonical_id) and not _looks_like_url(canonical_id):\n        canonical_id = f\"pmid:{_extract_pmid(canonical_id)}\"\n    elif _looks_like_url(canonical_id):\n        canonical_id = canonical_id\n    else:\n        canonical_id = f\"manual:{_slugify(canonical_id)}\"\n\n    ids = ExternalIds(\n        doi=_extract_doi(raw_id),\n        arxiv=_extract_arxiv(raw_id),\n        openalex=_extract_openalex(raw_id),\n        pmid=_extract_pmid(raw_id) if not _looks_like_url(raw_id) else None,\n        pmcid=_extract_pmcid(raw_id),\n    )\n\n    return PaperMetadata(\n        id=canonical_id,\n        source=PaperSource.unknown,\n        title=title,\n        abstract=None,\n        year=year,\n        url=raw_id if _looks_like_url(raw_id) else None,\n        pdf_url=None,\n        ids=ids,\n    )\n\n\ndef _resolve_entry_by_exact_identifier(entry: Dict[str, Any]) -> Optional[PaperMetadata]:\n    raw_id = str(entry.get(\"id\") or \"\").strip()\n    if not raw_id:\n        return None\n\n    doi = _extract_doi(raw_id)\n    if doi:\n        try:\n            meta = get_paper_by_doi(doi)\n            if meta is not None:\n                return meta\n        except Exception:\n            pass\n\n    title = str(entry.get(\"title\") or \"\").strip()\n    try:\n        candidates = search_papers(title or raw_id, limit=8, sources=DEFAULT_SEARCH_SOURCES)\n    except Exception:\n        candidates = []\n\n    best = _pick_best_candidate(entry, candidates)\n    if best is not None:\n        return best\n\n    return None\n\n\ndef resolve_papers_from_trajectory(doc: Dict[str, Any], *, search_limit: int = 8) -> List[PaperMetadata]:\n    resolved: List[PaperMetadata] = []\n    seen_ids: set[str] = set()\n\n    for entry in doc.get(\"papers\", []) or []:\n        if not isinstance(entry, dict):\n            continue\n        meta = _resolve_entry_by_exact_identifier(entry)\n        if meta is None:\n            title = str(entry.get(\"title\") or \"\").strip()\n            if title:\n                try:\n                    candidates = search_papers(title, limit=search_limit, sources=DEFAULT_SEARCH_SOURCES)\n                except Exception:\n                    candidates = []\n                meta = _pick_best_candidate(entry, candidates)\n        if meta is None:\n            meta = _fallback_paper(entry)\n\n        # Fill missing fields from YAML entry when resolver found only partial metadata.\n        if not meta.title and entry.get(\"title\"):\n            meta.title = str(entry.get(\"title\") or \"\")\n        if meta.year is None and entry.get(\"year\") not in (None, \"\"):\n            try:\n                meta.year = int(entry.get(\"year\"))\n            except Exception:\n                pass\n        if not meta.url and _looks_like_url(str(entry.get(\"id\") or \"\")):\n            meta.url = str(entry.get(\"id\") or \"\")\n\n        if meta.id in seen_ids:\n            continue\n        seen_ids.add(meta.id)\n        resolved.append(meta)\n\n    return resolved\n\n\ndef _source_label(src: Dict[str, Any], idx: int) -> str:\n    s = str(src.get(\"source\") or f\"source_{idx}\")\n    loc = str(src.get(\"locator\") or src.get(\"page\") or \"\").strip()\n    return f\"{s} {loc}\".strip()\n\n\ndef _paper_lookup(doc: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:\n    out: Dict[str, Dict[str, Any]] = {}\n    for p in doc.get(\"papers\", []) or []:\n        if not isinstance(p, dict):\n            continue\n        pid = str(p.get(\"id\") or \"\").strip()\n        if pid:\n            out[pid] = p\n            doi = _extract_doi(pid)\n            if doi:\n                out[doi] = p\n                out[f\"doi:{doi}\"] = p\n        title = str(p.get(\"title\") or \"\").strip()\n        if title:\n            out[_norm_title(title)] = p\n    return out\n\n\ndef _year_from_source(source_ref: str, papers_by_key: Dict[str, Dict[str, Any]]) -> Optional[int]:\n    ref = (source_ref or \"\").strip()\n    candidates = [ref]\n    doi = _extract_doi(ref)\n    if doi:\n        candidates.extend([doi, f\"doi:{doi}\"])\n    arxiv = _extract_arxiv(ref)\n    if arxiv:\n        candidates.extend([arxiv, f\"arxiv:{arxiv}\"])\n    oa = _extract_openalex(ref)\n    if oa:\n        candidates.extend([oa, f\"openalex:{oa}\"])\n    if _norm_title(ref):\n        candidates.append(_norm_title(ref))\n\n    for key in candidates:\n        paper = papers_by_key.get(key)\n        if not isinstance(paper, dict):\n            continue\n        try:\n            y = int(paper.get(\"year\")) if paper.get(\"year\") not in (None, \"\") else None\n        except Exception:\n            y = None\n        if y is not None:\n            return y\n    return None\n\n\ndef _step_time_window(step: Dict[str, Any], papers_by_key: Dict[str, Dict[str, Any]]) -> Tuple[str, str]:\n    years: List[int] = []\n    for src in step.get(\"sources\", []) or []:\n        if not isinstance(src, dict):\n            continue\n        y = _year_from_source(str(src.get(\"source\") or \"\"), papers_by_key)\n        if y is not None:\n            years.append(y)\n    if not years:\n        return (\"unknown\", \"unknown\")\n    return (str(min(years)), str(max(years)))\n\n\ndef _coerce_time_token(value: Any, *, default: str = \"unknown\") -> str:\n    if value is None:\n        return default\n    s = str(value).strip()\n    if not s:\n        return default\n    if DATE_TOKEN_RE.match(s):\n        return s\n    return default\n\n\ndef _canonical_time_interval(rec: Dict[str, Any]) -> str:\n    start = _coerce_time_token(rec.get(\"start_date\"), default=\"unknown\")\n    end = _coerce_time_token(rec.get(\"end_date\"), default=\"unknown\")\n    valid_from = _coerce_time_token(rec.get(\"valid_from\"), default=start)\n    valid_to = _coerce_time_token(rec.get(\"valid_to\"), default=\"+inf\")\n    legacy = str(rec.get(\"time_interval\") or \"\").strip()\n    if legacy:\n        return legacy\n    return f\"evidence:{start}..{end}|valid:{valid_from}..{valid_to}\"\n\n\ndef build_reference_graph(doc: Dict[str, Any]) -> Dict[str, Any]:\n    papers_by_key = _paper_lookup(doc)\n    steps = [s for s in (doc.get(\"steps\") or []) if isinstance(s, dict)]\n    manual_nodes: List[Dict[str, Any]] = []\n    manual_edges: List[Dict[str, Any]] = []\n    manual_triplets: List[Dict[str, Any]] = []\n\n    step_ids: List[int] = []\n    for idx, step in enumerate(steps, start=1):\n        sid = int(step.get(\"step_id\") or idx)\n        step_ids.append(sid)\n        start_date, end_date = _step_time_window(step, papers_by_key)\n        node_id = f\"step:{sid}\"\n        conditions = step.get(\"conditions\") if isinstance(step.get(\"conditions\"), dict) else {}\n        manual_nodes.append(\n            {\n                \"id\": node_id,\n                \"type\": \"trajectory_step\",\n                \"label\": str(step.get(\"claim\") or f\"Шаг {sid}\"),\n                \"step_id\": sid,\n                \"claim\": str(step.get(\"claim\") or \"\"),\n                \"inference\": str(step.get(\"inference\") or \"\"),\n                \"next_question\": str(step.get(\"next_question\") or \"\"),\n                \"conditions\": conditions,\n                \"start_date\": start_date,\n                \"end_date\": end_date,\n                \"valid_from\": start_date,\n                \"valid_to\": \"+inf\" if start_date != \"unknown\" else \"unknown\",\n                \"time_source\": \"metadata\",\n            }\n        )\n        manual_triplets.append(\n            {\n                \"assertion_id\": f\"manual-step-{sid}\",\n                \"subject\": node_id,\n                \"predicate\": \"states\",\n                \"object\": str(step.get(\"claim\") or \"\"),\n                \"start_date\": start_date,\n                \"end_date\": end_date,\n                \"valid_from\": start_date,\n                \"valid_to\": \"+inf\" if start_date != \"unknown\" else \"unknown\",\n                \"time_source\": \"metadata\",\n                \"time_interval\": f\"evidence:{start_date}..{end_date}|valid:{start_date}..{'+inf' if start_date != 'unknown' else 'unknown'}\",\n                \"evidence\": {\n                    \"page\": None,\n                    \"figure_or_table\": None,\n                    \"snippet_or_summary\": str(step.get(\"inference\") or step.get(\"claim\") or \"\"),\n                },\n                \"verdict\": \"accepted\",\n                \"rationale\": \"Reference step reconstructed from Task 1 trajectory YAML.\",\n            }\n        )\n\n        for src_idx, src in enumerate(step.get(\"sources\", []) or [], start=1):\n            if not isinstance(src, dict):\n                continue\n            source_id = f\"step:{sid}:source:{src_idx:02d}\"\n            stype = str(src.get(\"type\") or \"text\")\n            source_label = _source_label(src, src_idx)\n            page_value = src.get(\"page\")\n            try:\n                page_value = int(page_value) if page_value not in (None, \"\") else None\n            except Exception:\n                page_value = None\n            manual_nodes.append(\n                {\n                    \"id\": source_id,\n                    \"type\": \"evidence_source\",\n                    \"label\": source_label,\n                    \"source_type\": stype,\n                    \"source_ref\": str(src.get(\"source\") or \"\"),\n                    \"locator\": str(src.get(\"locator\") or \"\"),\n                    \"page\": page_value,\n                    \"snippet_or_summary\": str(src.get(\"snippet_or_summary\") or \"\"),\n                    \"start_date\": start_date,\n                    \"end_date\": end_date,\n                    \"valid_from\": start_date,\n                    \"valid_to\": end_date,\n                    \"time_source\": \"metadata\",\n                }\n            )\n            manual_edges.append(\n                {\n                    \"source\": source_id,\n                    \"target\": node_id,\n                    \"predicate\": \"supports\",\n                    \"type\": \"evidence_support\",\n                    \"start_date\": start_date,\n                    \"end_date\": end_date,\n                }\n            )\n            manual_triplets.append(\n                {\n                    \"assertion_id\": f\"manual-source-{sid}-{src_idx}\",\n                    \"subject\": str(src.get(\"source\") or source_id),\n                    \"predicate\": \"supports_step\",\n                    \"object\": node_id,\n                    \"start_date\": start_date,\n                    \"end_date\": end_date,\n                    \"valid_from\": start_date,\n                    \"valid_to\": end_date,\n                    \"time_source\": \"metadata\",\n                    \"time_interval\": f\"evidence:{start_date}..{end_date}|valid:{start_date}..{end_date}\",\n                    \"evidence\": {\n                        \"page\": page_value,\n                        \"figure_or_table\": str(src.get(\"locator\") or \"\") or None,\n                        \"snippet_or_summary\": str(src.get(\"snippet_or_summary\") or \"\"),\n                    },\n                    \"verdict\": \"accepted\",\n                    \"rationale\": \"Evidence link reconstructed from Task 1 trajectory YAML.\",\n                }\n            )\n\n    raw_edges = doc.get(\"edges\") or []\n    if not raw_edges and len(step_ids) > 1:\n        raw_edges = [[step_ids[i], step_ids[i + 1]] for i in range(len(step_ids) - 1)]\n\n    step_map = {int(step.get(\"step_id\") or idx): step for idx, step in enumerate(steps, start=1)}\n    for edge_idx, edge in enumerate(raw_edges, start=1):\n        if not isinstance(edge, (list, tuple)) or len(edge) != 2:\n            continue\n        try:\n            src_step = int(edge[0])\n            dst_step = int(edge[1])\n        except Exception:\n            continue\n        src_node = f\"step:{src_step}\"\n        dst_node = f\"step:{dst_step}\"\n        src_step_obj = step_map.get(src_step, {})\n        start_date, end_date = _step_time_window(src_step_obj, papers_by_key) if src_step_obj else (\"unknown\", \"unknown\")\n        manual_edges.append(\n            {\n                \"source\": src_node,\n                \"target\": dst_node,\n                \"predicate\": \"leads_to\",\n                \"type\": \"reasoning_transition\",\n                \"start_date\": start_date,\n                \"end_date\": end_date,\n            }\n        )\n        manual_triplets.append(\n            {\n                \"assertion_id\": f\"manual-edge-{edge_idx}\",\n                \"subject\": src_node,\n                \"predicate\": \"leads_to\",\n                \"object\": dst_node,\n                \"start_date\": start_date,\n                \"end_date\": end_date,\n                \"valid_from\": start_date,\n                \"valid_to\": \"+inf\" if start_date != \"unknown\" else \"unknown\",\n                \"time_source\": \"metadata\",\n                \"time_interval\": f\"evidence:{start_date}..{end_date}|valid:{start_date}..{'+inf' if start_date != 'unknown' else 'unknown'}\",\n                \"evidence\": {\n                    \"page\": None,\n                    \"figure_or_table\": None,\n                    \"snippet_or_summary\": str(src_step_obj.get(\"next_question\") or src_step_obj.get(\"inference\") or \"\"),\n                },\n                \"verdict\": \"accepted\",\n                \"rationale\": \"Reasoning transition reconstructed from Task 1 trajectory YAML.\",\n            }\n        )\n\n    return {\n        \"meta\": {\n            \"kind\": \"reference_reasoning_graph\",\n            \"artifact_version\": int(doc.get(\"artifact_version\") or 1),\n            \"topic\": str(doc.get(\"topic\") or \"\"),\n            \"domain\": str(doc.get(\"domain\") or \"\"),\n            \"submission_id\": str(doc.get(\"submission_id\") or \"\"),\n            \"generated_at\": _utc_now(),\n            \"n_steps\": len(steps),\n            \"n_papers\": len(doc.get(\"papers\") or []),\n        },\n        \"nodes\": manual_nodes,\n        \"edges\": manual_edges,\n        \"triplets\": manual_triplets,\n    }\n\n\ndef _paperrecord_from_metadata(meta: PaperMetadata) -> PaperRecord:\n    text = (meta.abstract or \"\").strip() or meta.title\n    return PaperRecord(\n        paper_id=meta.id,\n        title=meta.title,\n        year=_paper_year(meta),\n        text=text,\n        url=meta.url or \"\",\n        source=str(meta.source.value if hasattr(meta.source, \"value\") else meta.source),\n    )\n\n\ndef _load_domain_from_trajectory(doc: Dict[str, Any]) -> DomainConfig:\n    value = str(doc.get(\"domain\") or \"science\").strip()\n    return load_domain_config(value or None)\n\n\ndef _flatten_automatic_graph(kg: TemporalKnowledgeGraph) -> List[Dict[str, Any]]:\n    rows: List[Dict[str, Any]] = []\n    kg_json = kg.to_json_dict()\n    for idx, edge in enumerate(kg_json.get(\"edges\", []) or [], start=1):\n        years = []\n        try:\n            years = sorted(int(y) for y in (edge.get(\"yearly_count\") or {}).keys())\n        except Exception:\n            years = []\n        start_date = str(years[0]) if years else \"unknown\"\n        end_date = str(years[-1]) if years else \"unknown\"\n        quotes = edge.get(\"evidence_quotes\") or []\n        first_quote = quotes[0] if quotes else {}\n        rows.append(\n            {\n                \"assertion_id\": f\"auto-{idx:05d}\",\n                \"subject\": str(edge.get(\"source\") or \"\"),\n                \"predicate\": str(edge.get(\"predicate\") or \"\"),\n                \"object\": str(edge.get(\"target\") or \"\"),\n                \"start_date\": start_date,\n                \"end_date\": end_date,\n                \"valid_from\": start_date,\n                \"valid_to\": \"+inf\" if start_date != \"unknown\" else \"unknown\",\n                \"time_source\": \"metadata\",\n                \"time_interval\": f\"evidence:{start_date}..{end_date}|valid:{start_date}..{'+inf' if start_date != 'unknown' else 'unknown'}\",\n                \"score\": edge.get(\"score\"),\n                \"mean_confidence\": edge.get(\"mean_confidence\"),\n                \"papers\": edge.get(\"papers\") or [],\n                \"evidence\": {\n                    \"page\": None,\n                    \"figure_or_table\": None,\n                    \"snippet_or_summary\": str(first_quote.get(\"quote\") or \"\"),\n                    \"paper_id\": str(first_quote.get(\"paper_id\") or \"\"),\n                },\n            }\n        )\n    return rows\n\n\ndef _prefill_graph_review(doc: Dict[str, Any], automatic_rows: Sequence[Dict[str, Any]]) -> Dict[str, Any]:\n    reviewer_id = \"\"\n    expert = doc.get(\"expert\") if isinstance(doc.get(\"expert\"), dict) else {}\n    if expert:\n        reviewer_id = str(expert.get(\"latin_slug\") or expert.get(\"full_name\") or \"\")\n\n    assertions = []\n    for row in automatic_rows:\n        item = dict(row)\n        item[\"verdict\"] = \"\"\n        item[\"rationale\"] = \"\"\n        item[\"time_source_note\"] = \"\"\n        assertions.append(item)\n\n    return {\n        \"artifact_version\": 3,\n        \"domain\": str(doc.get(\"domain\") or \"\"),\n        \"topic\": str(doc.get(\"topic\") or \"\"),\n        \"trajectory_submission_id\": str(doc.get(\"submission_id\") or \"\"),\n        \"reviewer_id\": reviewer_id,\n        \"timestamp\": _utc_now(),\n        \"assertions\": assertions,\n    }\n\n\ndef _empty_temporal_corrections(doc: Dict[str, Any]) -> Dict[str, Any]:\n    reviewer_id = \"\"\n    expert = doc.get(\"expert\") if isinstance(doc.get(\"expert\"), dict) else {}\n    if expert:\n        reviewer_id = str(expert.get(\"latin_slug\") or expert.get(\"full_name\") or \"\")\n\n    return {\n        \"artifact_version\": 2,\n        \"domain\": str(doc.get(\"domain\") or \"\"),\n        \"paper_id\": \"\",\n        \"reviewer_id\": reviewer_id,\n        \"trajectory_submission_id\": str(doc.get(\"submission_id\") or \"\"),\n        \"corrections\": [],\n    }\n\n\ndef _compare_graphs(reference_graph: Dict[str, Any], automatic_rows: Sequence[Dict[str, Any]], resolved_papers: Sequence[PaperMetadata]) -> Dict[str, Any]:\n    ref_step_labels = {str(n.get(\"label\") or \"\") for n in reference_graph.get(\"nodes\", []) if n.get(\"type\") == \"trajectory_step\"}\n    auto_terms = {str(r.get(\"subject\") or \"\") for r in automatic_rows} | {str(r.get(\"object\") or \"\") for r in automatic_rows}\n    paper_ids = {p.id for p in resolved_papers}\n    auto_papers = set()\n    for row in automatic_rows:\n        auto_papers.update(str(x) for x in (row.get(\"papers\") or []))\n\n    return {\n        \"reference_steps\": len([n for n in reference_graph.get(\"nodes\", []) if n.get(\"type\") == \"trajectory_step\"]),\n        \"reference_edges\": len([e for e in reference_graph.get(\"edges\", []) if e.get(\"predicate\") == \"leads_to\"]),\n        \"automatic_edges\": len(list(automatic_rows)),\n        \"automatic_unique_terms\": len(auto_terms),\n        \"resolved_papers\": len(list(resolved_papers)),\n        \"papers_reaching_automatic_graph\": len(auto_papers & paper_ids),\n        \"trajectory_claim_examples\": sorted(list(ref_step_labels))[:10],\n        \"automatic_term_examples\": sorted(list(auto_terms))[:20],\n    }\n\n\ndef suggest_link_candidates(\n    doc: Dict[str, Any],\n    *,\n    known_papers: Sequence[PaperMetadata],\n    max_queries: int = 4,\n    per_query: int = 8,\n) -> List[Dict[str, Any]]:\n    queries: List[str] = []\n    topic = str(doc.get(\"topic\") or \"\").strip()\n    if topic:\n        queries.append(topic)\n\n    for step in doc.get(\"steps\", []) or []:\n        if not isinstance(step, dict):\n            continue\n        nq = str(step.get(\"next_question\") or \"\").strip()\n        if nq and nq not in queries:\n            queries.append(nq)\n        inf = str(step.get(\"inference\") or \"\").strip()\n        if inf and len(queries) < max_queries and inf not in queries:\n            queries.append(inf)\n        if len(queries) >= max_queries:\n            break\n\n    queries = queries[:max_queries]\n    known_titles = {_norm_title(p.title) for p in known_papers}\n    known_ids = {p.id for p in known_papers}\n    suggestions: Dict[str, Dict[str, Any]] = {}\n\n    for q in queries:\n        try:\n            found = search_papers(q, limit=per_query, sources=DEFAULT_SEARCH_SOURCES)\n        except Exception:\n            found = []\n        for cand in found:\n            if cand.id in known_ids or _norm_title(cand.title) in known_titles:\n                continue\n            score = 2.0 * _score_title_match(q, cand.title)\n            if cand.abstract:\n                score += 0.25\n            if cand.pdf_url:\n                score += 0.25\n            if cand.citation_count:\n                score += min(float(cand.citation_count) / 500.0, 0.5)\n            existing = suggestions.get(cand.id)\n            payload = {\n                \"paper_id\": cand.id,\n                \"title\": cand.title,\n                \"year\": _paper_year(cand),\n                \"url\": cand.url,\n                \"pdf_url\": cand.pdf_url,\n                \"source\": str(cand.source.value if hasattr(cand.source, \"value\") else cand.source),\n                \"trigger_queries\": [q],\n                \"score\": round(score, 4),\n            }\n            if existing is None:\n                suggestions[cand.id] = payload\n            else:\n                existing[\"score\"] = max(float(existing.get(\"score\") or 0.0), payload[\"score\"])\n                tq = list(existing.get(\"trigger_queries\") or [])\n                if q not in tq:\n                    tq.append(q)\n                existing[\"trigger_queries\"] = tq\n\n    ranked = sorted(suggestions.values(), key=lambda x: float(x.get(\"score\") or 0.0), reverse=True)\n    return ranked[:20]\n\n\ndef prepare_task2_validation_bundle(\n    trajectory_yaml: Path,\n    *,\n    out_dir: Path,\n    include_multimodal: bool = True,\n    run_vlm: bool = True,\n    edge_mode: str = \"auto\",\n    suggest_links: bool = True,\n    max_papers: int = 0,\n    max_link_queries: int = 4,\n) -> Path:\n    doc = yaml.safe_load(trajectory_yaml.read_text(encoding=\"utf-8\")) or {}\n    if not isinstance(doc, dict):\n        raise ValueError(\"Trajectory YAML must contain a top-level object.\")\n\n    run_name = str(doc.get(\"submission_id\") or _slugify(str(doc.get(\"topic\") or trajectory_yaml.stem)))\n    out = out_dir / run_name\n    out.mkdir(parents=True, exist_ok=True)\n\n    shutil.copy2(trajectory_yaml, out / trajectory_yaml.name)\n\n    reference_graph = build_reference_graph(doc)\n    (out / \"reference_graph.json\").write_text(json.dumps(reference_graph, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    (out / \"reference_triplets.json\").write_text(json.dumps(reference_graph.get(\"triplets\") or [], ensure_ascii=False, indent=2), encoding=\"utf-8\")\n\n    domain_cfg = _load_domain_from_trajectory(doc)\n\n    resolved = resolve_papers_from_trajectory(doc)\n    if max_papers and max_papers > 0:\n        resolved = resolved[:max_papers]\n\n    (out / \"papers_resolved.json\").write_text(\n        json.dumps([p.model_dump(mode=\"json\") for p in resolved], ensure_ascii=False, indent=2),\n        encoding=\"utf-8\",\n    )\n\n    acquire_dir = out / \"automatic_graph\"\n    raw_dir = acquire_dir / \"raw_pdfs\"\n    meta_dir = acquire_dir / \"raw_meta\"\n    processed_dir = acquire_dir / \"processed_papers\"\n    processed_dir.mkdir(parents=True, exist_ok=True)\n\n    acq: List[AcquireResult] = acquire_pdfs(resolved, raw_dir=raw_dir, meta_dir=meta_dir)\n    (out / \"acquire_results.json\").write_text(\n        json.dumps(\n            [asdict(a) | {\"pdf_path\": str(a.pdf_path) if a.pdf_path else None, \"meta_path\": str(a.meta_path)} for a in acq],\n            ensure_ascii=False,\n            indent=2,\n        ),\n        encoding=\"utf-8\",\n    )\n\n    ingested_ids: List[str] = []\n    for meta, a in zip(resolved, acq):\n        if not a.pdf_path:\n            continue\n        meta_json = meta.model_dump(mode=\"json\")\n        try:\n            if include_multimodal:\n                ingest_pdf_multimodal_auto(a.pdf_path, meta_json, processed_dir, run_vlm=run_vlm)\n            else:\n                ingest_pdf_auto(a.pdf_path, meta_json, processed_dir)\n            ingested_ids.append(meta.id)\n        except Exception as e:\n            console.print(f\"[yellow]Ingest failed for {meta.id}: {e}[/yellow]\")\n\n    processed_records = load_papers_from_processed(processed_dir)\n    processed_by_id = {p.paper_id: p for p in processed_records}\n    paper_records: List[PaperRecord] = []\n    for meta in resolved:\n        if meta.id in processed_by_id:\n            paper_records.append(processed_by_id[meta.id])\n        else:\n            paper_records.append(_paperrecord_from_metadata(meta))\n\n    (out / \"paper_records.json\").write_text(\n        json.dumps([asdict(r) for r in paper_records], ensure_ascii=False, indent=2),\n        encoding=\"utf-8\",\n    )\n\n    kg = build_temporal_kg(\n        paper_records,\n        domain=domain_cfg,\n        query=str(doc.get(\"topic\") or \"\"),\n        edge_mode=edge_mode,  # type: ignore[arg-type]\n        expert_overrides_path=None,\n    )\n    kg.dump_json(out / \"automatic_graph\" / \"temporal_kg.json\")\n\n    automatic_rows = _flatten_automatic_graph(kg)\n    (out / \"automatic_triplets.json\").write_text(json.dumps(automatic_rows, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n\n    prefill_review = _prefill_graph_review(doc, automatic_rows)\n    review_dir = out / \"review_templates\"\n    review_dir.mkdir(parents=True, exist_ok=True)\n    (review_dir / \"graph_review_prefill.json\").write_text(json.dumps(prefill_review, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    (review_dir / \"temporal_corrections_template.json\").write_text(json.dumps(_empty_temporal_corrections(doc), ensure_ascii=False, indent=2), encoding=\"utf-8\")\n\n    comparison = _compare_graphs(reference_graph, automatic_rows, resolved)\n    (out / \"comparison_summary.json\").write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n\n    if suggest_links:\n        suggestions = suggest_link_candidates(doc, known_papers=resolved, max_queries=max_link_queries)\n        scout_dir = out / \"scout\"\n        scout_dir.mkdir(parents=True, exist_ok=True)\n        (scout_dir / \"suggested_links.json\").write_text(json.dumps(suggestions, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n\n    manifest = {\n        \"generated_at\": _utc_now(),\n        \"trajectory_file\": trajectory_yaml.name,\n        \"submission_id\": str(doc.get(\"submission_id\") or \"\"),\n        \"topic\": str(doc.get(\"topic\") or \"\"),\n        \"domain\": str(doc.get(\"domain\") or \"\"),\n        \"resolved_papers\": len(resolved),\n        \"ingested_pdfs\": len(ingested_ids),\n        \"automatic_edges\": len(automatic_rows),\n        \"reference_steps\": len([n for n in reference_graph.get(\"nodes\", []) if n.get(\"type\") == \"trajectory_step\"]),\n        \"artifacts\": {\n            \"reference_graph\": \"reference_graph.json\",\n            \"reference_triplets\": \"reference_triplets.json\",\n            \"automatic_graph\": \"automatic_graph/temporal_kg.json\",\n            \"automatic_triplets\": \"automatic_triplets.json\",\n            \"papers_resolved\": \"papers_resolved.json\",\n            \"comparison_summary\": \"comparison_summary.json\",\n            \"review_prefill\": \"review_templates/graph_review_prefill.json\",\n        },\n    }\n    (out / \"manifest.json\").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    return out\n", "scripts/ci/validate_expert_artifacts.py": "#!/usr/bin/env python3\n\"\"\"\nValidate expert artifacts (YAML/JSON) for required fields:\n- evidence present\n- conditions present\n- time_scope / time_interval present (where applicable)\n\nExit code:\n- 0 if all good\n- 1 if any problems found\n\nDesigned to be used in CI on pull requests.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport re\nimport sys\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Tuple\n\ntry:\n    import yaml  # type: ignore\nexcept Exception:\n    print(\"ERROR: Missing dependency PyYAML. Install with `pip install pyyaml`.\", file=sys.stderr)\n    raise\n\nREPO_ROOT = Path(__file__).resolve().parents[2]\n\nTRAJ_DIR = REPO_ROOT / \"data\" / \"experts\" / \"trajectories\"\nGRAPH_REV_DIR = REPO_ROOT / \"data\" / \"experts\" / \"graph_reviews\"\nHYP_REV_DIR = REPO_ROOT / \"data\" / \"experts\" / \"hypothesis_reviews\"\nMM_REV_DIR = REPO_ROOT / \"data\" / \"experts\" / \"mm_reviews\"\nTEMP_CORR_DIR = REPO_ROOT / \"data\" / \"experts\" / \"temporal_corrections\"\n\n\ndef _load_yaml(path: Path) -> Dict[str, Any]:\n    with path.open(\"r\", encoding=\"utf-8\") as f:\n        return yaml.safe_load(f) or {}\n\n\ndef _load_json(path: Path) -> Dict[str, Any]:\n    with path.open(\"r\", encoding=\"utf-8\") as f:\n        return json.load(f)\n\n\ndef _is_empty(v: Any) -> bool:\n    if v is None:\n        return True\n    if isinstance(v, str) and not v.strip():\n        return True\n    if isinstance(v, (list, dict)) and len(v) == 0:\n        return True\n    return False\n\n\nTIME_TOKEN_RE = re.compile(r\"^(?:\\d{4}|\\d{4}-\\d{2}|\\d{4}-\\d{2}-\\d{2}|unknown|\\+inf|-inf)$\")\n\n\ndef _is_time_token(v: Any) -> bool:\n    if v is None:\n        return False\n    return bool(TIME_TOKEN_RE.fullmatch(str(v).strip()))\n\n\ndef _has_structured_temporal_fields(a: Dict[str, Any]) -> bool:\n    return any(not _is_empty(a.get(k)) for k in (\"start_date\", \"end_date\", \"valid_from\", \"valid_to\"))\n\n\ndef _validate_temporal_fields(a: Dict[str, Any], prefix: str) -> List[str]:\n    errs: List[str] = []\n    if _has_structured_temporal_fields(a):\n        # Evidence interval is the primary temporal axis in graph reviews.\n        if _is_empty(a.get(\"start_date\")):\n            errs.append(f\"{prefix}: missing start_date (use 'unknown' or '-inf' if needed)\")\n        elif not _is_time_token(a.get(\"start_date\")):\n            errs.append(f\"{prefix}: invalid start_date '{a.get('start_date')}'\")\n\n        if _is_empty(a.get(\"end_date\")):\n            errs.append(f\"{prefix}: missing end_date (use publication date, 'unknown' or '+inf' if needed)\")\n        elif not _is_time_token(a.get(\"end_date\")):\n            errs.append(f\"{prefix}: invalid end_date '{a.get('end_date')}'\")\n\n        for k in (\"valid_from\", \"valid_to\"):\n            if not _is_empty(a.get(k)) and not _is_time_token(a.get(k)):\n                errs.append(f\"{prefix}: invalid {k} '{a.get(k)}'\")\n    return errs\n\n\ndef _resolve_domain_config(domain_value: str) -> Dict[str, Any]:\n    \"\"\"Resolve domain config from configs/domains/*.yaml.\n\n    Supports both:\n    - legacy domain id (e.g. \"science\")\n    - Wikidata QID stored in trajectory artifact (e.g. \"Q336\")\n    \"\"\"\n    dv = (domain_value or \"\").strip()\n    if not dv:\n        return {}\n\n    qid = dv.upper() if re.fullmatch(r\"Q\\d+\", dv.upper()) else \"\"\n\n    domain_dir = REPO_ROOT / \"configs\" / \"domains\"\n    for cfg_path in sorted(domain_dir.glob(\"*.yaml\")):\n        try:\n            cfg = yaml.safe_load(cfg_path.read_text(encoding=\"utf-8\")) or {}\n        except Exception:\n            continue\n\n        # QID match\n        if qid:\n            if str(cfg.get(\"wikidata_qid\") or \"\").strip().upper() == qid:\n                return cfg\n            continue\n\n        # legacy match\n        domain_id = str(cfg.get(\"domain_id\") or \"\").strip().lower()\n        stem = cfg_path.stem.strip().lower()\n        dv_norm = dv.strip().lower()\n        if dv_norm in {domain_id, stem}:\n            return cfg\n\n    return {}\n\n\ndef _required_condition_keys(domain: str) -> List[str]:\n    \"\"\"Domain-specific condition requirements for trajectories.\n\n    For artifact v2 the trajectory stores `domain` as a Wikidata QID (e.g. Q336).\n    We map it back to configs/domains/*.yaml via `wikidata_qid`.\n    \"\"\"\n    cfg = _resolve_domain_config(domain)\n    av = (cfg.get(\"artifact_validation\") or {}) if isinstance(cfg, dict) else {}\n    req = av.get(\"trajectory_required_conditions\") or []\n    if isinstance(req, list):\n        return [str(x) for x in req]\n    return []\n\n\n\ndef validate_trajectory(path: Path) -> List[str]:\n    errs: List[str] = []\n    doc = _load_yaml(path)\n\n    artifact_version = int(doc.get(\"artifact_version\") or 1)\n    domain = str(doc.get(\"domain\") or \"\").strip()\n\n    if _is_empty(domain):\n        errs.append(\"domain required\")\n\n    steps = doc.get(\"steps\", [])\n    if not isinstance(steps, list) or len(steps) == 0:\n        errs.append(\"steps[] must be a non-empty list\")\n        return errs\n\n    n_steps = len(steps)\n    required_keys = _required_condition_keys(domain) if domain else []\n\n    for i, step in enumerate(steps, start=1):\n        if not isinstance(step, dict):\n            errs.append(f\"step {i}: must be object\")\n            continue\n\n        if _is_empty(step.get(\"claim\")):\n            errs.append(f\"step {i}: missing claim\")\n\n        # conditions\n        conditions = step.get(\"conditions\", {})\n        if not isinstance(conditions, dict) or _is_empty(conditions):\n            errs.append(f\"step {i}: missing conditions\")\n        else:\n            for k in required_keys:\n                if _is_empty(conditions.get(k)):\n                    errs.append(f\"step {i}: conditions.{k} required for domain={domain}\")\n\n        # inference / next_question (recommended to be non-empty)\n        if _is_empty(step.get(\"inference\")):\n            errs.append(f\"step {i}: missing inference\")\n        if _is_empty(step.get(\"next_question\")):\n            errs.append(f\"step {i}: missing next_question\")\n\n        if artifact_version >= 2:\n            # v2: sources[]\n            sources = step.get(\"sources\", [])\n            if not isinstance(sources, list) or len(sources) == 0:\n                errs.append(f\"step {i}: missing sources[] (must be non-empty list)\")\n            else:\n                for j, src in enumerate(sources, start=1):\n                    if not isinstance(src, dict):\n                        errs.append(f\"step {i} source {j}: must be object\")\n                        continue\n                    stype = str(src.get(\"type\") or \"\").strip().lower()\n                    if stype == \"figure\":\n                        stype = \"image\"\n                    if stype not in {\"text\", \"image\", \"table\"}:\n                        errs.append(f\"step {i} source {j}: invalid type='{src.get('type')}' (use text/image/table)\")\n                    if _is_empty(src.get(\"source\")):\n                        errs.append(f\"step {i} source {j}: source required\")\n                    if _is_empty(src.get(\"snippet_or_summary\")):\n                        errs.append(f\"step {i} source {j}: snippet_or_summary required\")\n        else:\n            # v1 legacy: evidence{}\n            evidence = step.get(\"evidence\", {})\n            if not isinstance(evidence, dict) or _is_empty(evidence):\n                errs.append(f\"step {i}: missing evidence\")\n            else:\n                if _is_empty(evidence.get(\"source\")):\n                    errs.append(f\"step {i}: evidence.source required\")\n                if _is_empty(evidence.get(\"page\")):\n                    errs.append(f\"step {i}: evidence.page required\")\n                if _is_empty(evidence.get(\"snippet_or_summary\")):\n                    errs.append(f\"step {i}: evidence.snippet_or_summary required\")\n                etype = str(evidence.get(\"type\", \"\")).strip().lower()\n                if etype in {\"figure\", \"table\"} and _is_empty(evidence.get(\"figure_or_table\")):\n                    errs.append(f\"step {i}: evidence.figure_or_table required for {etype}\")\n\n    # edges (optional): directed graph as ordered pairs [from, to]\n    edges = doc.get(\"edges\", [])\n    if edges is None:\n        edges = []\n    if not _is_empty(edges):\n        if not isinstance(edges, list):\n            errs.append(\"edges must be a list of [from,to]\")\n        else:\n            seen: set[tuple[int, int]] = set()\n            for k, e in enumerate(edges, start=1):\n                if not isinstance(e, (list, tuple)) or len(e) != 2:\n                    errs.append(f\"edge {k}: must be [from, to]\")\n                    continue\n                a, b = e[0], e[1]\n                try:\n                    a_i = int(a)\n                    b_i = int(b)\n                except Exception:\n                    errs.append(f\"edge {k}: from/to must be integers\")\n                    continue\n                if a_i == b_i:\n                    errs.append(f\"edge {k}: self-loop {a_i}->{b_i} is not allowed\")\n                    continue\n                if not (1 <= a_i <= n_steps) or not (1 <= b_i <= n_steps):\n                    errs.append(f\"edge {k}: out of range (steps are 1..{n_steps})\")\n                    continue\n                t = (a_i, b_i)\n                if t in seen:\n                    errs.append(f\"edge {k}: duplicate edge {a_i}->{b_i}\")\n                else:\n                    seen.add(t)\n\n    return errs\n\n\n\ndef validate_graph_review(path: Path) -> List[str]:\n    errs: List[str] = []\n    doc = _load_json(path)\n\n    assertions = doc.get(\"assertions\", [])\n    if not isinstance(assertions, list) or len(assertions) == 0:\n        return [\"assertions[] must be a non-empty list\"]\n\n    for i, a in enumerate(assertions, start=1):\n        if not isinstance(a, dict):\n            errs.append(f\"assertion {i}: must be object\")\n            continue\n\n        for k in (\"subject\", \"predicate\", \"object\"):\n            if _is_empty(a.get(k)):\n                errs.append(f\"assertion {i}: missing {k}\")\n\n        has_legacy_interval = not _is_empty(a.get(\"time_interval\"))\n        if has_legacy_interval:\n            # Legacy free-form field remains supported for backward compatibility.\n            pass\n        elif not _has_structured_temporal_fields(a):\n            errs.append(\n                f\"assertion {i}: missing temporal information (provide time_interval or start_date/end_date; use 'unknown' if not stated)\"\n            )\n\n        errs.extend(_validate_temporal_fields(a, f\"assertion {i}\"))\n\n        ev = a.get(\"evidence\", {})\n        if not isinstance(ev, dict) or _is_empty(ev):\n            errs.append(f\"assertion {i}: missing evidence\")\n        else:\n            if _is_empty(ev.get(\"snippet_or_summary\")):\n                errs.append(f\"assertion {i}: evidence.snippet_or_summary required\")\n            has_locator = not _is_empty(ev.get(\"page\")) or not _is_empty(ev.get(\"figure_or_table\")) or not _is_empty(ev.get(\"paper_id\")) or not _is_empty(ev.get(\"source\"))\n            if not has_locator:\n                errs.append(f\"assertion {i}: evidence should include page, figure_or_table, paper_id, or source\")\n\n        verdict = str(a.get(\"verdict\", \"\")).strip()\n        if verdict not in {\"accepted\", \"rejected\", \"needs_time_fix\", \"needs_evidence_fix\", \"added\"}:\n            errs.append(f\"assertion {i}: invalid verdict '{verdict}'\")\n\n        if _is_empty(a.get(\"rationale\")):\n            errs.append(f\"assertion {i}: missing rationale\")\n\n    return errs\n\n\ndef validate_hypothesis_review(path: Path) -> List[str]:\n    errs: List[str] = []\n    doc = _load_json(path)\n\n    if _is_empty(doc.get(\"time_scope\")):\n        errs.append(\"missing time_scope (conditions/time applicability)\")\n\n    scores = doc.get(\"scores\")\n    if not isinstance(scores, dict):\n        errs.append(\"missing scores object\")\n    else:\n        for k in (\"novelty\", \"soundness\", \"testability\"):\n            if k not in scores:\n                errs.append(f\"missing scores.{k}\")\n\n    if \"accept\" not in doc:\n        errs.append(\"missing accept (true/false)\")\n    if _is_empty(doc.get(\"major_issues\")):\n        errs.append(\"missing major_issues[] (at least 1, or ['none'] if truly clean)\")\n    if _is_empty(doc.get(\"required_experiments\")):\n        errs.append(\"missing required_experiments (how to test)\")\n\n    return errs\n\n\ndef validate_mm_review(path: Path) -> List[str]:\n    errs: List[str] = []\n    doc = _load_json(path)\n\n    items = doc.get(\"items\", [])\n    if not isinstance(items, list) or len(items) == 0:\n        return [\"items[] must be a non-empty list\"]\n\n    for i, it in enumerate(items, start=1):\n        if not isinstance(it, dict):\n            errs.append(f\"item {i}: must be object\")\n            continue\n\n        if it.get(\"page\") is None:\n            errs.append(f\"item {i}: missing page\")\n        if _is_empty(it.get(\"verdict\")):\n            errs.append(f\"item {i}: missing verdict\")\n        if _is_empty(it.get(\"rationale\")):\n            errs.append(f\"item {i}: missing rationale\")\n\n        # At least one modality field should be present\n        if _is_empty(it.get(\"vlm_caption\")) and _is_empty(it.get(\"tables_md\")) and _is_empty(it.get(\"equations_md\")):\n            errs.append(f\"item {i}: provide at least one of vlm_caption/tables_md/equations_md\")\n\n        v = str(it.get(\"verdict\", \"\")).strip()\n        if v not in {\"accepted\", \"needs_fix\", \"rejected\"}:\n            errs.append(f\"item {i}: invalid verdict '{v}'\")\n\n    return errs\n\n\ndef validate_temporal_correction(path: Path) -> List[str]:\n    errs: List[str] = []\n    doc = _load_json(path)\n\n    corrections = doc.get(\"corrections\", [])\n    if not isinstance(corrections, list) or len(corrections) == 0:\n        return [\"corrections[] must be a non-empty list\"]\n\n    for i, c in enumerate(corrections, start=1):\n        if not isinstance(c, dict):\n            errs.append(f\"correction {i}: must be object\")\n            continue\n        if _is_empty(c.get(\"assertion_id\")):\n            errs.append(f\"correction {i}: missing assertion_id\")\n        if _is_empty(c.get(\"rationale\")):\n            errs.append(f\"correction {i}: missing rationale\")\n\n        ct = c.get(\"corrected_time\")\n        if not isinstance(ct, dict) or _is_empty(ct):\n            errs.append(f\"correction {i}: missing corrected_time object\")\n        else:\n            if _is_empty(ct.get(\"granularity\")):\n                errs.append(f\"correction {i}: corrected_time.granularity required\")\n            if _is_empty(ct.get(\"start\")) and _is_empty(ct.get(\"end\")):\n                errs.append(f\"correction {i}: corrected_time must include start and/or end\")\n            for key in (\"start\", \"end\"):\n                if not _is_empty(ct.get(key)) and not _is_time_token(ct.get(key)):\n                    errs.append(f\"correction {i}: invalid corrected_time.{key} '{ct.get(key)}'\")\n\n        ot = c.get(\"original_time\")\n        if isinstance(ot, dict) and not _is_empty(ot):\n            for key in (\"start\", \"end\"):\n                if not _is_empty(ot.get(key)) and not _is_time_token(ot.get(key)):\n                    errs.append(f\"correction {i}: invalid original_time.{key} '{ot.get(key)}'\")\n\n    return errs\n\n\ndef _collect_files() -> List[Tuple[str, Path]]:\n    files: List[Tuple[str, Path]] = []\n    for p in sorted(TRAJ_DIR.glob(\"**/*.y*ml\")):\n        files.append((\"trajectory\", p))\n    for p in sorted(GRAPH_REV_DIR.glob(\"**/*.json\")):\n        files.append((\"graph_review\", p))\n    for p in sorted(HYP_REV_DIR.glob(\"**/*.json\")):\n        files.append((\"hypothesis_review\", p))\n    for p in sorted(MM_REV_DIR.glob(\"**/*.json\")):\n        files.append((\"mm_review\", p))\n    for p in sorted(TEMP_CORR_DIR.glob(\"**/*.json\")):\n        files.append((\"temporal_correction\", p))\n    return files\n\n\ndef main() -> int:\n    problems: List[str] = []\n\n    for kind, path in _collect_files():\n        try:\n            if kind == \"trajectory\":\n                errs = validate_trajectory(path)\n            elif kind == \"graph_review\":\n                errs = validate_graph_review(path)\n            elif kind == \"hypothesis_review\":\n                errs = validate_hypothesis_review(path)\n            elif kind == \"mm_review\":\n                errs = validate_mm_review(path)\n            else:\n                errs = validate_temporal_correction(path)\n        except Exception as e:\n            problems.append(f\"{kind} {path}: failed to parse ({e})\")\n            continue\n\n        for e in errs:\n            problems.append(f\"{kind} {path}: {e}\")\n\n    if problems:\n        print(\"❌ Expert artifacts validation failed:\\n\")\n        for p in problems:\n            print(\" - \" + p)\n        print(f\"\\nTotal problems: {len(problems)}\")\n        return 1\n\n    print(\"✅ Expert artifacts validation passed.\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "data/experts/graph_reviews/_template.json": "{\n  \"artifact_version\": 3,\n  \"domain\": \"example_domain_or_wikidata_qid\",\n  \"topic\": \"example topic\",\n  \"paper_id\": \"doi:...\",\n  \"reviewer_id\": \"expert_001\",\n  \"timestamp\": \"2026-02-01T12:00:00Z\",\n  \"assertions\": [\n    {\n      \"assertion_id\": \"A-000123\",\n      \"subject\": \"Method X\",\n      \"predicate\": \"improves\",\n      \"object\": \"Outcome Y\",\n      \"start_date\": \"2021\",\n      \"end_date\": \"2024\",\n      \"valid_from\": \"2021\",\n      \"valid_to\": \"+inf\",\n      \"time_source\": \"explicit_text\",\n      \"time_source_note\": \"В статье явно указан период исследования 2021-2024; claim считается актуальным до появления опровержения.\",\n      \"time_interval\": \"evidence:2021..2024|valid:2021..+inf\",\n      \"evidence\": {\n        \"paper_id\": \"doi:10.xxxx/xxxxx\",\n        \"page\": 6,\n        \"figure_or_table\": \"Figure 5\",\n        \"source\": \"doi:10.xxxx/xxxxx\",\n        \"snippet_or_summary\": \"Коротко: что именно утверждается источником\"\n      },\n      \"verdict\": \"accepted\",\n      \"rationale\": \"Почему так (границы применимости, альтернативы, риски)\"\n    }\n  ]\n}\n", "data/experts/temporal_corrections/_template.json": "{\n  \"artifact_version\": 2,\n  \"domain\": \"example_domain\",\n  \"paper_id\": \"arxiv:2401.01234\",\n  \"reviewer_id\": \"anon\",\n  \"trajectory_submission_id\": \"example_submission\",\n  \"corrections\": [\n    {\n      \"assertion_id\": \"arxiv:2401.01234|Li plating|is triggered by|high overpotential|supports|year:2018:2018\",\n      \"original_time\": {\"start\": \"2018\", \"end\": \"2018\", \"granularity\": \"year\"},\n      \"corrected_time\": {\"start\": \"2016\", \"end\": \"+inf\", \"granularity\": \"year\"},\n      \"evidence_quote\": \"...\",\n      \"rationale\": \"Откуда следует правильный период; допускаются -inf/+inf для открытых интервалов.\"\n    }\n  ]\n}\n", "docs/course/experts/docs_experts/task2_graph_verification_temporal_mm.md": "# Task 2 — Валидация темпорального графа знаний\n\n## Что делает эксперт\nЭксперт загружает YAML из **Task 1** и получает **два графа**:\n\n1. **Reference graph** — точное восстановление вручную размеченной reasoning-схемы из YAML.\n2. **Automatic temporal KG** — автоматически построенный темпоральный граф по статьям из YAML через CLI `top-papers-graph prepare-task2-validation`.\n\nДальше эксперт сравнивает оба графа, просматривает триплеты, оценивает временные поля и при необходимости вносит правки в `graph_review_prefill.json` и `temporal_corrections_template.json`.\n\n## Temporal-схема\nВ Task 2 используется **двойная темпоральность**:\n\n- `start_date` — начало интервала, к которому относится утверждение / наблюдение.\n- `end_date` — конец интервала, обычно дата публикации или последний год, подтверждённый источником.\n- `valid_from` — момент, с которого утверждение считаем валидным в графе.\n- `valid_to` — момент, до которого утверждение считается валидным; если опровержения нет, допустимо `+inf`.\n- `time_source` — откуда взято время: `explicit_text`, `figure_or_table`, `metadata`, `expert_inference`, `unknown`.\n- `time_source_note` — свободный комментарий эксперта.\n\nПоддерживаются специальные значения: `unknown`, `-inf`, `+inf`.\nПоле `time_interval` сохраняется только ради обратной совместимости с существующим пайплайном и формируется автоматически.\n\n## Как интерпретировать время\nРекомендуемый приоритет:\n\n1. **explicit_text** — время явно указано в тексте статьи.\n2. **figure_or_table** — время извлечено из подписи, оси, таблицы.\n3. **metadata** — время взято из года публикации / внешних метаданных.\n4. **expert_inference** — время реконструировано экспертом, если другого источника нет.\n5. **unknown** — если время определить нельзя.\n\n## Вердикты эксперта\n- `accepted`\n- `rejected`\n- `needs_time_fix`\n- `needs_evidence_fix`\n- `added`\n\n## Дополнительный scouting\nCLI также создаёт `scout/suggested_links.json` — список дополнительных статей-кандидатов, найденных по `topic` и `next_question` из trajectory. Это отдельный слой поддержки эксперта: ссылки предлагает система, а решение об их использовании принимает эксперт.\n\n## Где смотреть результаты\nПосле запуска `prepare-task2-validation` в bundle появляются:\n\n- `reference_graph.json`\n- `reference_triplets.json`\n- `automatic_graph/temporal_kg.json`\n- `automatic_triplets.json`\n- `comparison_summary.json`\n- `review_templates/graph_review_prefill.json`\n- `review_templates/temporal_corrections_template.json`\n- `scout/suggested_links.json`\n", "docs/course/experts/templates/graph_review_template.json": "{\n  \"artifact_version\": 3,\n  \"domain\": \"example_domain_or_wikidata_qid\",\n  \"topic\": \"example topic\",\n  \"paper_id\": \"doi:...\",\n  \"reviewer_id\": \"expert_001\",\n  \"timestamp\": \"2026-02-01T12:00:00Z\",\n  \"assertions\": [\n    {\n      \"assertion_id\": \"A-000123\",\n      \"subject\": \"Method X\",\n      \"predicate\": \"improves\",\n      \"object\": \"Outcome Y\",\n      \"start_date\": \"2021\",\n      \"end_date\": \"2024\",\n      \"valid_from\": \"2021\",\n      \"valid_to\": \"+inf\",\n      \"time_source\": \"explicit_text\",\n      \"time_source_note\": \"Период 2021-2024 прямо указан в статье; claim считаем валидным до опровержения.\",\n      \"time_interval\": \"evidence:2021..2024|valid:2021..+inf\",\n      \"evidence\": {\n        \"paper_id\": \"doi:10.xxxx/xxxxx\",\n        \"page\": 6,\n        \"figure_or_table\": \"Figure 5\",\n        \"source\": \"doi:10.xxxx/xxxxx\",\n        \"snippet_or_summary\": \"Коротко: что именно утверждается источником\"\n      },\n      \"verdict\": \"accepted\",\n      \"rationale\": \"Почему так (границы применимости, альтернативы, риски)\"\n    }\n  ]\n}\n", "README_TASK2_VALIDATION.md": "# Task 2 validation bundle\n\nThis repository snapshot includes:\n\n- patched CLI command `top-papers-graph prepare-task2-validation`\n- temporal review schema v3 (`start_date`, `end_date`, `valid_from`, `valid_to`)\n- backward compatibility with legacy `time_interval`\n- Colab notebook `notebooks/task2_temporal_graph_validation_colab.ipynb`\n- sample Task 1 YAML inputs under `examples/task2_validation_inputs/`\n\nRecommended entrypoint for experts: open the notebook and run it top-to-bottom.\n"}''')


def run_cmd

In [ ]:
# @title
# ФОРМА: клонирование репозитория, загрузка YAML, выбор VLM и запуск Task 2
repo_url = W.Text(value='https://github.com/top-papers/top-papers-graph.git', description='Repo URL:', layout=W.Layout(width='900px'))
repo_branch = W.Text(value='main', description='Branch:', layout=W.Layout(width='300px'))
repo_dir = W.Text(value=str(WORK_ROOT / 'top-papers-graph'), description='Repo dir:', layout=W.Layout(width='900px'))

clone_btn = W.Button(description='1. Клонировать и развернуть репозиторий', button_style='primary')
probe_btn = W.Button(description='2. Проверить g4f и выбрать VLM', button_style='warning')
build_btn = W.Button(description='3. Собрать Task 2 bundle', button_style='success')

upload = W.FileUpload(accept='.yaml,.yml', multiple=False, description='Загрузить YAML')
existing_yaml_path = W.Text(value='', description='или path:', placeholder='опционально: путь к YAML, если файл уже лежит на диске', layout=W.Layout(width='900px'))

multimodal_cb = W.Checkbox(value=True, description='Multimodal ingest')
vlm_cb = W.Checkbox(value=True, description='VLM on PDF pages')
suggest_links_cb = W.Checkbox(value=True, description='Scout suggested links')
edge_mode_dd = W.Dropdown(options=['auto', 'llm_triplets', 'cooccurrence'], value='auto', description='edge_mode:')
max_papers_int = W.BoundedIntText(value=0, min=0, max=500, description='max_papers:')

status_out = W.Output(layout={'border': '1px solid #ddd'})


def on_clone_clicked(_):
    with status_out:
        clear_output()
        try:
            rdir = clone_and_setup_repo(repo_url.value.strip(), repo_branch.value.strip(), Path(repo_dir.value.strip()))
            print('Репозиторий готов:', rdir)
            print('Теперь загрузите YAML и нажмите кнопку выбора VLM.')
        except Exception as e:
            print('Ошибка на этапе setup:', e)


def on_probe_clicked(_):
    with status_out:
        clear_output()
        if not SESSION.get('repo_dir'):
            print('Сначала клонируйте репозиторий.')
            return
        try:
            result = choose_vlm_backend(Path(SESSION['repo_dir']))
            print('Выбранный режим:', result['mode'])
            print('Модель:', result['model'])
            print('Файл env:', result['env_path'])
            ok_count = sum(1 for x in result['report'] if x['ok'])
            print('Успешных g4f probe:', ok_count)
            if result['mode'] == 'qwen2_vl':
                print('
ВНИМАНИЕ: включён fallback на Qwen2.5-VL-3B-Instruct.')
                print('Для него рекомендуется GPU в Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU.')
        except Exception as e:
            print('Ошибка при выборе VLM:', e)


def on_build_clicked(_):
    with status_out:
        clear_output()
        if not SESSION.get('repo_dir'):
            print('Сначала клонируйте репозиторий.')
            return
        try:
            if existing_yaml_path.value.strip():
                trajectory_path = Path(existing_yaml_path.value.strip())
            else:
                trajectory_path = _save_upload_value(upload.value, Path(SESSION['repo_dir']) / 'task2_inputs')
            SESSION['trajectory_path'] = str(trajectory_path)
            print('Использую YAML:', trajectory_path)
            bundle_dir = build_task2_bundle(
                repo_dir=Path(SESSION['repo_dir']),
                trajectory_path=trajectory_path,
                out_dir=Path(SESSION['repo_dir']) / 'runs' / 'task2_validation',
                multimodal=multimodal_cb.value,
                vlm=vlm_cb.value,
                suggest_links=suggest_links_cb.value,
                edge_mode=edge_mode_dd.value,
                max_papers=max_papers_int.value,
            )
            print('
Bundle собран:', bundle_dir)
            display_bundle(bundle_dir)
        except Exception as e:
            print('Ошибка при сборке Task 2 bundle:', e)

clone_btn.on_click(on_clone_clicked)
probe_btn.on_click(on_probe_clicked)
build_btn.on_click(on_build_clicked)

ui = W.VBox([
    W.HTML('<h3>Task 2 workflow</h3>'),
    repo_url,
    W.HBox([repo_branch]),
    repo_dir,
    clone_btn,
    W.HTML('<hr><b>YAML из Task 1</b>'),
    upload,
    existing_yaml_path,
    W.HBox([multimodal_cb, vlm_cb, suggest_links_cb]),
    W.HBox([edge_mode_dd, max_papers_int]),
    probe_btn,
    build_btn,
    status_out,
])

display(ui)



# FAQ

## 1) Нужно ли редактировать код?
Нет. Обычный сценарий — просто запускать ячейки и пользоваться кнопками.

## 2) Почему в Task 2 два времени?
Потому что у темпорального графа здесь две разные задачи:
- время, **когда утверждение относится к наблюдаемому миру** (`start_date`, `end_date`)
- время, **когда утверждение считаем валидным в KG** (`valid_from`, `valid_to`)

## 3) Зачем сохраняется `time_interval`, если есть новые ключи?
Для полной совместимости с текущим пайплайном и обратной совместимости с уже существующими артефактами.

## 4) Что делать, если g4f нестабилен?
Это ожидаемо. Блокнот автоматически включает fallback на локальный `Qwen/Qwen2.5-VL-3B-Instruct`.

## 5) Что делать, если локальный Qwen не запускается?
В Colab переключите среду на GPU и затем снова нажмите кнопку выбора VLM.

## 6) Что эксперт валидирует руками?
- корректность триплетов
- временные поля
- корректность evidence
- недостающие или лишние связи
- дополнительные статьи из `scout/suggested_links.json`
